# Entrenamiento y validación de modelos de predicción de abandono

Este notebook desarrolla la fase de modelado a partir del dataset preparado y analizado previamente. La unidad de observación es el cliente y el objetivo consiste en estimar la probabilidad de abandono representada por la variable objetivo binaria `churn`.

El proceso respeta estrictamente las particiones originales. Los 52.004 clientes del conjunto de entrenamiento se utilizan para comparar modelos, ajustar hiperparámetros y tomar decisiones metodológicas. Los 13.001 clientes del conjunto de prueba permanecen aislados hasta la evaluación final.

La comparación de modelos se realiza mediante validación cruzada estratificada sobre el conjunto de entrenamiento. Este procedimiento permite estimar el rendimiento en diferentes subconjuntos de clientes, manteniendo aproximadamente la proporción de cada clase en todas las divisiones.

Debido al desequilibrio moderado de la variable objetivo, la evaluación no se basa únicamente en la exactitud. También se consideran las siguientes métricas y herramientas de diagnóstico:

- El área bajo la curva ROC.
- El área bajo la curva de precisión-recall.
- La precisión.
- La sensibilidad o `recall`.
- La puntuación F1.
- La matriz de confusión.
- La capacidad discriminativa de las probabilidades estimadas.

La comparación y selección inicial de los modelos se lleva a cabo exclusivamente con el conjunto de entrenamiento. El conjunto de prueba se reserva para evaluar el rendimiento del modelo y del umbral de clasificación seleccionados.

Durante esta fase se almacenan las configuraciones, los resultados de validación, las métricas y los modelos entrenados, garantizando la trazabilidad y reproducibilidad del proceso.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ---------------------------------------------------------
# Definición de las rutas del proyecto
# ---------------------------------------------------------

RUTA_CUADERNO = Path.cwd()

if RUTA_CUADERNO.name == "notebooks":
    RUTA_PROYECTO = RUTA_CUADERNO.parent
else:
    RUTA_PROYECTO = RUTA_CUADERNO

RUTA_RESULTADOS = (
    RUTA_PROYECTO
    / "resultados"
)

RUTA_MODELOS = (
    RUTA_PROYECTO
    / "modelos"
)

RUTA_EVIDENCIAS = (
    RUTA_PROYECTO
    / "evidencias"
)

RUTA_DATASET_MODELADO = (
    RUTA_RESULTADOS
    / "dataset_modelado_90d.pkl"
)

RUTA_COLUMNAS_PREDICTORAS = (
    RUTA_RESULTADOS
    / "columnas_predictoras_176_90d.csv"
)


# ---------------------------------------------------------
# Comprobación de los elementos necesarios
# ---------------------------------------------------------

rutas_modelado = {
    "proyecto": RUTA_PROYECTO,
    "resultados": RUTA_RESULTADOS,
    "modelos": RUTA_MODELOS,
    "evidencias": RUTA_EVIDENCIAS,
    "dataset_modelado": RUTA_DATASET_MODELADO,
    "columnas_predictoras": RUTA_COLUMNAS_PREDICTORAS
}

comprobacion_rutas_modelado = pd.DataFrame({
    "elemento": rutas_modelado.keys(),
    "ruta": [
        str(ruta)
        for ruta in rutas_modelado.values()
    ],
    "existe": [
        ruta.exists()
        for ruta in rutas_modelado.values()
    ]
})

display(
    comprobacion_rutas_modelado
)

print(
    "¿Están disponibles todos los elementos "
    "necesarios para el modelado?",
    comprobacion_rutas_modelado["existe"].all()
)

## Carga de los datos y creación del holdout interno bloqueado

El conjunto original de entrenamiento contiene 52.004 clientes. Utilizar todos estos registros tanto para desarrollar los modelos como para evaluar el resultado seleccionado podría producir una estimación influida por las decisiones tomadas durante el proceso.

Para evitarlo, el conjunto de entrenamiento se divide en dos subconjuntos:

- **Desarrollo:** se utiliza para completar la ingeniería temporal, comparar modelos mediante validación cruzada, ajustar hiperparámetros y seleccionar el umbral de clasificación.
- **Holdout interno bloqueado:** permanece sin consultar hasta que se hayan fijado la ventana temporal, las variables predictoras, el algoritmo, los hiperparámetros y el umbral.

La división se realiza mediante muestreo estratificado para conservar aproximadamente la proporción de `churn` en ambos subconjuntos. Se reserva el 15 % de los datos como holdout interno y se utiliza la semilla `314159` para que la asignación sea reproducible.

El conjunto externo de prueba, formado por 13.001 clientes, no participa en esta división y permanece reservado para la evaluación final.

Como resultado, se obtienen:

- 44.203 clientes en el conjunto de desarrollo.
- 7.801 clientes en el holdout interno.
- 13.001 clientes en el conjunto externo de prueba.
- Ningún cliente compartido entre los tres conjuntos.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit


# ---------------------------------------------------------
# Carga del dataset preparado y del inventario de variables
# ---------------------------------------------------------

dataset_modelado = pd.read_pickle(
    RUTA_DATASET_MODELADO
)

inventario_variables = pd.read_csv(
    RUTA_COLUMNAS_PREDICTORAS
)

columnas_predictoras = (
    inventario_variables["variable"]
    .tolist()
)


# ---------------------------------------------------------
# Separación del train original y del test externo
# ---------------------------------------------------------

datos_train_original = (
    dataset_modelado.loc[
        dataset_modelado["particion"] == "train"
    ]
    .reset_index(drop=True)
    .copy()
)

datos_test_externo = (
    dataset_modelado.loc[
        dataset_modelado["particion"] == "test"
    ]
    .reset_index(drop=True)
    .copy()
)


# ---------------------------------------------------------
# Creación reproducible del holdout interno bloqueado
# ---------------------------------------------------------

SEMILLA_HOLDOUT = 314159
FRACCION_HOLDOUT = 0.15

divisor_holdout = StratifiedShuffleSplit(
    n_splits=1,
    test_size=FRACCION_HOLDOUT,
    random_state=SEMILLA_HOLDOUT
)

indices_desarrollo, indices_holdout = next(
    divisor_holdout.split(
        datos_train_original[
            ["customer_id"]
        ],
        datos_train_original["churn"]
    )
)

datos_desarrollo = (
    datos_train_original
    .iloc[indices_desarrollo]
    .reset_index(drop=True)
    .copy()
)

datos_holdout = (
    datos_train_original
    .iloc[indices_holdout]
    .reset_index(drop=True)
    .copy()
)


# ---------------------------------------------------------
# Construcción del manifiesto de partición
# ---------------------------------------------------------

manifiesto_split = (
    datos_train_original[
        ["customer_id", "churn"]
    ]
    .copy()
)

manifiesto_split["split"] = "development"

manifiesto_split.loc[
    indices_holdout,
    "split"
] = "locked_internal_holdout"

manifiesto_split[
    "split_random_state"
] = SEMILLA_HOLDOUT

manifiesto_split[
    "holdout_fraction"
] = FRACCION_HOLDOUT


# ---------------------------------------------------------
# Comprobaciones de separación
# ---------------------------------------------------------

ids_desarrollo = set(
    datos_desarrollo["customer_id"]
)

ids_holdout = set(
    datos_holdout["customer_id"]
)

ids_test_externo = set(
    datos_test_externo["customer_id"]
)

comprobaciones_split = {
    "train_original_correcto": (
        len(datos_train_original) == 52_004
    ),
    "desarrollo_correcto": (
        len(datos_desarrollo) == 44_203
    ),
    "holdout_correcto": (
        len(datos_holdout) == 7_801
    ),
    "test_externo_correcto": (
        len(datos_test_externo) == 13_001
    ),
    "desarrollo_holdout_sin_solapamiento": (
        len(
            ids_desarrollo.intersection(
                ids_holdout
            )
        ) == 0
    ),
    "desarrollo_test_sin_solapamiento": (
        len(
            ids_desarrollo.intersection(
                ids_test_externo
            )
        ) == 0
    ),
    "holdout_test_sin_solapamiento": (
        len(
            ids_holdout.intersection(
                ids_test_externo
            )
        ) == 0
    ),
    "union_reconstruye_train": (
        ids_desarrollo.union(ids_holdout)
        == set(
            datos_train_original[
                "customer_id"
            ]
        )
    ),
    "variables_predictoras_correctas": (
        len(columnas_predictoras) == 176
    )
}

display(
    pd.DataFrame({
        "comprobacion": comprobaciones_split.keys(),
        "resultado": comprobaciones_split.values()
    })
)


# ---------------------------------------------------------
# Distribución de churn por subconjunto
# ---------------------------------------------------------

resumen_split = pd.concat([
    datos_desarrollo.assign(
        subconjunto="development"
    ),
    datos_holdout.assign(
        subconjunto="locked_internal_holdout"
    ),
    datos_test_externo.assign(
        subconjunto="external_test"
    )
])[
    ["subconjunto", "churn"]
]

resumen_split = (
    resumen_split
    .groupby(
        ["subconjunto", "churn"],
        observed=True
    )
    .size()
    .rename("clientes")
    .reset_index()
)

resumen_split[
    "porcentaje_subconjunto"
] = (
    resumen_split["clientes"]
    / resumen_split
        .groupby("subconjunto")["clientes"]
        .transform("sum")
    * 100
)

display(resumen_split)


# ---------------------------------------------------------
# Almacenamiento del manifiesto
# ---------------------------------------------------------

RUTA_MANIFIESTO_SPLIT = (
    RUTA_RESULTADOS
    / "split_manifest_v1.csv"
)

manifiesto_split.to_csv(
    RUTA_MANIFIESTO_SPLIT,
    index=False,
    encoding="utf-8-sig"
)


# ---------------------------------------------------------
# Resultado final
# ---------------------------------------------------------

print(
    "Clientes en desarrollo:",
    len(datos_desarrollo)
)

print(
    "Clientes en holdout interno:",
    len(datos_holdout)
)

print(
    "Clientes en test externo:",
    len(datos_test_externo)
)

print(
    "¿El manifiesto se ha guardado?",
    RUTA_MANIFIESTO_SPLIT.exists()
)

print(
    "¿La separación interna es válida?",
    all(comprobaciones_split.values())
)

## Preparación de las matrices y del protocolo de validación cruzada

Una vez separados los conjuntos de desarrollo, holdout interno y prueba externa, se preparan las matrices utilizadas durante la comparación de modelos.

Todas las decisiones de selección se toman exclusivamente con los 44.203 clientes del conjunto de desarrollo. El holdout interno y la prueba externa permanecen apartados y no intervienen en la elección de algoritmos, el ajuste de hiperparámetros ni la selección del umbral de clasificación.

Sobre el conjunto de desarrollo se aplica una validación cruzada estratificada de cinco pliegues. En cada iteración, cuatro pliegues se utilizan para entrenar el modelo y el restante para validarlo. El proceso se repite hasta que cada pliegue ha actuado una vez como conjunto de validación.

La estratificación conserva aproximadamente la proporción de las clases de `churn` en cada pliegue. Además, los registros se mezclan utilizando una semilla fija para que las divisiones sean reproducibles.

La comparación de modelos considera las siguientes métricas:

- `accuracy`: proporción total de clasificaciones correctas.
- `balanced_accuracy`: media del `recall` obtenido para cada clase.
- `precision`: proporción de predicciones positivas que corresponden realmente a clientes con `churn`.
- `recall`: proporción de clientes con `churn` que el modelo identifica correctamente.
- `F1`: media armónica entre `precision` y `recall`.
- `F2`: medida que combina `precision` y `recall`, otorgando mayor importancia al segundo.
- `AUC-ROC`: capacidad del modelo para ordenar y distinguir ambas clases a través de diferentes umbrales.
- `average_precision`: resumen de la relación entre `precision` y `recall` a través de diferentes umbrales.
- `Brier score`: error cuadrático medio de las probabilidades estimadas.
- `log loss`: penalización de las probabilidades incorrectas, especialmente cuando se asignan con un nivel elevado de confianza.

Las variables predictoras se mantienen en el orden definido en el inventario de 176 variables. Las columnas `customer_id`, `churn` y `particion` quedan excluidas de las matrices de entrada.

In [ ]:
from sklearn.metrics import (
    make_scorer,
    fbeta_score
)


# ---------------------------------------------------------
# Construcción de matrices para los tres subconjuntos
# ---------------------------------------------------------

X_desarrollo = datos_desarrollo[
    columnas_predictoras
].copy()

y_desarrollo = datos_desarrollo[
    "churn"
].copy()

X_holdout = datos_holdout[
    columnas_predictoras
].copy()

y_holdout = datos_holdout[
    "churn"
].copy()

X_test_externo = datos_test_externo[
    columnas_predictoras
].copy()

y_test_externo = datos_test_externo[
    "churn"
].copy()


# ---------------------------------------------------------
# Configuración reproducible de la validación cruzada
# ---------------------------------------------------------

SEMILLA_CV = 314159
NUMERO_PLIEGUES = 5

validacion_cruzada = StratifiedKFold(
    n_splits=NUMERO_PLIEGUES,
    shuffle=True,
    random_state=SEMILLA_CV
)


# ---------------------------------------------------------
# Métricas comunes para todos los modelos
# ---------------------------------------------------------

metricas_validacion = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "f2": make_scorer(
        fbeta_score,
        beta=2
    ),
    "auc_roc": "roc_auc",
    "average_precision": "average_precision",
    "brier": "neg_brier_score",
    "log_loss": "neg_log_loss"
}


# ---------------------------------------------------------
# Comprobaciones estructurales
# ---------------------------------------------------------

comprobaciones_protocolo = {
    "X_desarrollo_correcto": (
        X_desarrollo.shape == (44_203, 176)
    ),
    "y_desarrollo_correcto": (
        y_desarrollo.shape == (44_203,)
    ),
    "X_holdout_correcto": (
        X_holdout.shape == (7_801, 176)
    ),
    "y_holdout_correcto": (
        y_holdout.shape == (7_801,)
    ),
    "X_test_externo_correcto": (
        X_test_externo.shape == (13_001, 176)
    ),
    "y_test_externo_correcto": (
        y_test_externo.shape == (13_001,)
    ),
    "mismo_orden_de_columnas": (
        X_desarrollo.columns.tolist()
        == X_holdout.columns.tolist()
        == X_test_externo.columns.tolist()
        == columnas_predictoras
    ),
    "sin_nulos_desarrollo": (
        X_desarrollo.isna().sum().sum() == 0
    ),
    "sin_nulos_holdout": (
        X_holdout.isna().sum().sum() == 0
    ),
    "sin_nulos_test_externo": (
        X_test_externo.isna().sum().sum() == 0
    ),
    "objetivo_desarrollo_binario": (
        set(y_desarrollo.unique()) == {0, 1}
    ),
    "numero_pliegues_correcto": (
        validacion_cruzada.n_splits == 5
    ),
    "metricas_definidas": (
        len(metricas_validacion) == 10
    )
}

display(
    pd.DataFrame({
        "comprobacion": comprobaciones_protocolo.keys(),
        "resultado": comprobaciones_protocolo.values()
    })
)


# ---------------------------------------------------------
# Comprobación de la estratificación de los cinco pliegues
# ---------------------------------------------------------

resumen_pliegues = []

for numero_pliegue, (
    indices_entrenamiento,
    indices_validacion
) in enumerate(
    validacion_cruzada.split(
        X_desarrollo,
        y_desarrollo
    ),
    start=1
):
    y_entrenamiento_pliegue = (
        y_desarrollo.iloc[
            indices_entrenamiento
        ]
    )

    y_validacion_pliegue = (
        y_desarrollo.iloc[
            indices_validacion
        ]
    )

    resumen_pliegues.append({
        "pliegue": numero_pliegue,
        "clientes_entrenamiento": len(
            indices_entrenamiento
        ),
        "clientes_validacion": len(
            indices_validacion
        ),
        "tasa_churn_entrenamiento": (
            y_entrenamiento_pliegue.mean()
        ),
        "tasa_churn_validacion": (
            y_validacion_pliegue.mean()
        )
    })

resumen_pliegues = pd.DataFrame(
    resumen_pliegues
)

display(
    resumen_pliegues.style.format({
        "tasa_churn_entrenamiento": "{:.4%}",
        "tasa_churn_validacion": "{:.4%}"
    })
)


# ---------------------------------------------------------
# Resultado final
# ---------------------------------------------------------

diferencia_maxima_tasa_churn = (
    resumen_pliegues[
        "tasa_churn_validacion"
    ].max()
    - resumen_pliegues[
        "tasa_churn_validacion"
    ].min()
)

print(
    "Dimensión de X_desarrollo:",
    X_desarrollo.shape
)

print(
    "Dimensión de X_holdout:",
    X_holdout.shape
)

print(
    "Dimensión de X_test_externo:",
    X_test_externo.shape
)

print(
    "Número de pliegues:",
    validacion_cruzada.n_splits
)

print(
    "Número de métricas:",
    len(metricas_validacion)
)

print(
    "Diferencia máxima de tasa de churn "
    "entre pliegues de validación:",
    f"{diferencia_maxima_tasa_churn:.4%}"
)

print(
    "¿El protocolo de validación está correctamente preparado?",
    all(comprobaciones_protocolo.values())
    and diferencia_maxima_tasa_churn < 0.001
)

## Definición de los modelos candidatos

La comparación inicial incluye algoritmos con diferentes niveles de complejidad y capacidad para representar relaciones entre las variables predictoras.

Se evalúan seis candidatos:

1. **Baseline de prevalencia.** Genera predicciones a partir de la distribución general de las clases y proporciona una referencia básica frente a la que comparar el resto de los modelos.
2. **Regresión logística con regularización L2.** Proporciona un modelo lineal regularizado e interpretable. El escalado de las variables se integra en el pipeline para evitar fugas de información durante la validación cruzada.
3. **Regresión logística balanceada.** Mantiene la misma estructura del modelo anterior, pero ajusta el peso de las clases para compensar su desequilibrio.
4. **Árbol de decisión.** Permite representar relaciones y reglas no lineales, y ofrece una referencia de fácil interpretación.
5. **Random Forest.** Combina múltiples árboles para reducir la variabilidad de las predicciones y capturar interacciones no lineales.
6. **XGBoost.** Construye árboles de manera secuencial mediante boosting y actúa como un candidato de mayor capacidad predictiva.

En esta primera comparación, todos los modelos utilizan una configuración fija. El objetivo no es realizar todavía una búsqueda exhaustiva de hiperparámetros, sino evaluar diferentes enfoques mediante el mismo protocolo de validación cruzada estratificada de cinco pliegues.

Las transformaciones necesarias, como `StandardScaler`, se integran dentro del pipeline correspondiente. De este modo, el escalador se ajusta únicamente con los datos de entrenamiento de cada pliegue y no utiliza información del pliegue de validación.

Antes de ejecutar la comparación completa, se comprueba que las dependencias necesarias, incluida la correspondiente a `XGBClassifier`, están disponibles y que los seis estimadores pueden inicializarse correctamente.

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


# ---------------------------------------------------------
# Comprobación de la disponibilidad de XGBoost
# ---------------------------------------------------------

try:
    import xgboost
    from xgboost import XGBClassifier

    XGBOOST_DISPONIBLE = True
    VERSION_XGBOOST = xgboost.__version__

except ImportError:
    XGBOOST_DISPONIBLE = False
    VERSION_XGBOOST = None


print(
    "¿XGBoost está disponible?",
    XGBOOST_DISPONIBLE
)

print(
    "Versión de XGBoost:",
    VERSION_XGBOOST
)


if not XGBOOST_DISPONIBLE:
    raise ImportError(
        "XGBoost no está instalado en el entorno churn_tfm. "
        "Detén aquí la ejecución y no continúes con los modelos."
    )


# ---------------------------------------------------------
# Semilla común de los modelos
# ---------------------------------------------------------

SEMILLA_MODELOS = 42


# ---------------------------------------------------------
# Definición de los seis modelos candidatos
# ---------------------------------------------------------

modelos_candidatos = {
    "dummy_prior": DummyClassifier(
        strategy="prior"
    ),

    "logistic_l2": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                solver="liblinear",
                max_iter=200,
                random_state=SEMILLA_MODELOS
            )
        )
    ]),

    "logistic_balanced": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                solver="liblinear",
                class_weight="balanced",
                max_iter=200,
                random_state=SEMILLA_MODELOS
            )
        )
    ]),

    "decision_tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=20,
        random_state=SEMILLA_MODELOS
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=SEMILLA_MODELOS
    ),

    "xgboost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=SEMILLA_MODELOS
    )
}


# ---------------------------------------------------------
# Comprobación de que todos pueden clonarse
# ---------------------------------------------------------

comprobacion_modelos = []

for nombre_modelo, modelo in modelos_candidatos.items():

    try:
        modelo_clonado = clone(modelo)

        comprobacion_modelos.append({
            "modelo": nombre_modelo,
            "clase": type(modelo).__name__,
            "clonable": True,
            "error": ""
        })

    except Exception as error:

        comprobacion_modelos.append({
            "modelo": nombre_modelo,
            "clase": type(modelo).__name__,
            "clonable": False,
            "error": str(error)
        })


comprobacion_modelos_df = pd.DataFrame(
    comprobacion_modelos
)

display(comprobacion_modelos_df)


# ---------------------------------------------------------
# Resumen y validación
# ---------------------------------------------------------

print(
    "Número de modelos definidos:",
    len(modelos_candidatos)
)

print(
    "Modelos definidos:"
)

for nombre_modelo in modelos_candidatos:
    print("-", nombre_modelo)


modelos_correctamente_definidos = (
    XGBOOST_DISPONIBLE
    and len(modelos_candidatos) == 6
    and comprobacion_modelos_df[
        "clonable"
    ].all()
)

print(
    "¿Los modelos candidatos están correctamente definidos?",
    modelos_correctamente_definidos
)

> **Dependencia XGBoost.** La instalación de `xgboost-cpu` se realiza al crear el entorno a partir de `environment.yml`. Se ha desactivado aquí la instalación automática para no modificar el entorno durante la ejecución del notebook.

## Comparación inicial de modelos mediante validación cruzada

Los seis modelos candidatos se comparan utilizando las mismas divisiones estratificadas del conjunto de desarrollo. De este modo, todos se evalúan sobre los mismos clientes en cada pliegue y la comparación se realiza en condiciones equivalentes.

La validación cruzada consta de cinco iteraciones. En cada una, el modelo se ajusta con cuatro pliegues y se evalúa sobre el restante. Al finalizar, se calculan la media y la desviación estándar de cada métrica.

Los modelos se ejecutan de forma secuencial para evitar combinar la paralelización externa de la validación cruzada con la paralelización interna de Random Forest y XGBoost. Esta combinación podría provocar un consumo excesivo de memoria y capacidad de procesamiento.

Las métricas basadas en clases predichas —`precision`, `recall`, F1 y F2— utilizan inicialmente el umbral convencional de `0,5`. Este valor no se considera definitivo, ya que el umbral final se seleccionará posteriormente a partir de probabilidades generadas exclusivamente dentro del conjunto de desarrollo.

El `Brier score` y la `log loss` son métricas de error, por lo que valores menores indican un mejor resultado. Scikit-learn las devuelve con signo negativo para mantener la convención de que una puntuación mayor representa un mejor rendimiento. En el resumen se recupera su signo positivo para facilitar la interpretación.

El baseline de prevalencia proporciona una referencia básica para valorar la mejora conseguida por los modelos predictivos, especialmente en `AUC-ROC`, `average_precision` y las métricas de detección de la clase positiva.

Los resultados de cada pliegue y su resumen agregado se conservan para su análisis y almacenamiento posterior. El holdout interno y el conjunto externo de prueba no intervienen en esta comparación.


In [ ]:
from time import perf_counter

from sklearn.metrics import make_scorer


# ---------------------------------------------------------
# Redefinición robusta de las métricas de clasificación
# ---------------------------------------------------------

metricas_validacion = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",

    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),

    "recall": make_scorer(
        recall_score,
        zero_division=0
    ),

    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),

    "f2": make_scorer(
        fbeta_score,
        beta=2,
        zero_division=0
    ),

    "auc_roc": "roc_auc",
    "average_precision": "average_precision",
    "brier": "neg_brier_score",
    "log_loss": "neg_log_loss"
}


# ---------------------------------------------------------
# Ejecución secuencial de la validación cruzada
# ---------------------------------------------------------

resultados_pliegues = []
resumen_modelos = []

for numero_modelo, (
    nombre_modelo,
    modelo
) in enumerate(
    modelos_candidatos.items(),
    start=1
):

    print("\n" + "=" * 72)
    print(
        f"MODELO {numero_modelo} DE "
        f"{len(modelos_candidatos)}"
    )
    print(nombre_modelo)
    print("=" * 72)

    inicio_modelo = perf_counter()

    resultado_cv = cross_validate(
        estimator=modelo,
        X=X_desarrollo,
        y=y_desarrollo,
        cv=validacion_cruzada,
        scoring=metricas_validacion,
        n_jobs=1,
        return_train_score=False,
        error_score="raise"
    )

    tiempo_total_modelo = (
        perf_counter() - inicio_modelo
    )

    tabla_pliegues_modelo = pd.DataFrame({
        "modelo": nombre_modelo,
        "pliegue": np.arange(
            1,
            NUMERO_PLIEGUES + 1
        ),
        "fit_time": resultado_cv["fit_time"],
        "score_time": resultado_cv["score_time"],
        "accuracy": resultado_cv[
            "test_accuracy"
        ],
        "balanced_accuracy": resultado_cv[
            "test_balanced_accuracy"
        ],
        "precision": resultado_cv[
            "test_precision"
        ],
        "recall": resultado_cv[
            "test_recall"
        ],
        "f1": resultado_cv[
            "test_f1"
        ],
        "f2": resultado_cv[
            "test_f2"
        ],
        "auc_roc": resultado_cv[
            "test_auc_roc"
        ],
        "average_precision": resultado_cv[
            "test_average_precision"
        ],
        "brier": -resultado_cv[
            "test_brier"
        ],
        "log_loss": -resultado_cv[
            "test_log_loss"
        ]
    })

    resultados_pliegues.append(
        tabla_pliegues_modelo
    )

    resumen_modelo = {
        "modelo": nombre_modelo,
        "tiempo_total_segundos": (
            tiempo_total_modelo
        )
    }

    metricas_resumen = [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "f2",
        "auc_roc",
        "average_precision",
        "brier",
        "log_loss",
        "fit_time",
        "score_time"
    ]

    for metrica in metricas_resumen:
        resumen_modelo[
            f"{metrica}_media"
        ] = tabla_pliegues_modelo[
            metrica
        ].mean()

        resumen_modelo[
            f"{metrica}_desviacion"
        ] = tabla_pliegues_modelo[
            metrica
        ].std(ddof=1)

    resumen_modelos.append(
        resumen_modelo
    )

    print(
        "AUC-ROC media:",
        f"{resumen_modelo['auc_roc_media']:.4f}"
    )

    print(
        "Average precision media:",
        f"{resumen_modelo['average_precision_media']:.4f}"
    )

    print(
        "F1 media:",
        f"{resumen_modelo['f1_media']:.4f}"
    )

    print(
        "Recall medio:",
        f"{resumen_modelo['recall_media']:.4f}"
    )

    print(
        "Tiempo total:",
        f"{tiempo_total_modelo:,.2f} segundos"
    )


# ---------------------------------------------------------
# Integración de resultados
# ---------------------------------------------------------

resultados_cv_pliegues = pd.concat(
    resultados_pliegues,
    ignore_index=True
)

resumen_cv_modelos = pd.DataFrame(
    resumen_modelos
)

resumen_cv_modelos = (
    resumen_cv_modelos
    .sort_values(
        [
            "average_precision_media",
            "auc_roc_media"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

resumen_cv_modelos.insert(
    0,
    "posicion_average_precision",
    np.arange(
        1,
        len(resumen_cv_modelos) + 1
    )
)


# ---------------------------------------------------------
# Presentación del resumen principal
# ---------------------------------------------------------

columnas_resumen_principal = [
    "posicion_average_precision",
    "modelo",
    "average_precision_media",
    "average_precision_desviacion",
    "auc_roc_media",
    "auc_roc_desviacion",
    "f1_media",
    "recall_media",
    "precision_media",
    "f2_media",
    "balanced_accuracy_media",
    "brier_media",
    "log_loss_media",
    "tiempo_total_segundos"
]

display(
    resumen_cv_modelos[
        columnas_resumen_principal
    ].style.format({
        "average_precision_media": "{:.4f}",
        "average_precision_desviacion": "{:.4f}",
        "auc_roc_media": "{:.4f}",
        "auc_roc_desviacion": "{:.4f}",
        "f1_media": "{:.4f}",
        "recall_media": "{:.4f}",
        "precision_media": "{:.4f}",
        "f2_media": "{:.4f}",
        "balanced_accuracy_media": "{:.4f}",
        "brier_media": "{:.4f}",
        "log_loss_media": "{:.4f}",
        "tiempo_total_segundos": "{:,.2f}"
    })
)


# ---------------------------------------------------------
# Validación de los resultados obtenidos
# ---------------------------------------------------------

columnas_metricas_finitas = [
    columna
    for columna in resultados_cv_pliegues.columns
    if columna not in [
        "modelo",
        "pliegue"
    ]
]

comparacion_inicial_valida = (
    resultados_cv_pliegues.shape[0]
        == (
            len(modelos_candidatos)
            * NUMERO_PLIEGUES
        )
    and resultados_cv_pliegues[
        "modelo"
    ].nunique() == 6
    and resultados_cv_pliegues[
        "pliegue"
    ].between(
        1,
        NUMERO_PLIEGUES
    ).all()
    and np.isfinite(
        resultados_cv_pliegues[
            columnas_metricas_finitas
        ].to_numpy()
    ).all()
    and resumen_cv_modelos.shape[0] == 6
)

print(
    "\nFilas de resultados por pliegue:",
    len(resultados_cv_pliegues)
)

print(
    "Modelos evaluados:",
    resultados_cv_pliegues[
        "modelo"
    ].nunique()
)

print(
    "Pliegues por modelo:"
)

display(
    resultados_cv_pliegues
    .groupby("modelo")
    .size()
    .rename("numero_pliegues")
    .reset_index()
)

print(
    "¿La comparación inicial es válida?",
    comparacion_inicial_valida
)

## Análisis y almacenamiento de la comparación inicial

Una vez completada la validación cruzada, se analizan conjuntamente los resultados de los seis modelos candidatos.

La comparación no se basa en una única métrica. Un modelo puede mostrar una buena capacidad general de discriminación y, al mismo tiempo, generar probabilidades poco precisas o detectar pocos clientes con `churn` al aplicar el umbral inicial de `0,5`.

Para obtener una evaluación equilibrada, se consideran los siguientes criterios:

- `average_precision`, especialmente relevante cuando la clase positiva es minoritaria.
- `AUC-ROC`, que mide la capacidad del modelo para ordenar y distinguir ambas clases.
- F1 y F2, que combinan `precision` y `recall` con diferente importancia relativa.
- `recall`, que representa la proporción de clientes con `churn` identificados correctamente.
- `Brier score` y `log loss`, que evalúan la calidad de las probabilidades estimadas.
- La estabilidad de los resultados entre pliegues.
- El tiempo de ejecución.

Se calculan posiciones independientes para las métricas principales y se construye un ranking de consenso. Este ranking se utiliza únicamente para identificar los modelos finalistas y no determina todavía la selección definitiva.

Los resultados de cada pliegue y el resumen agregado se almacenan como evidencias reproducibles de la comparación. El holdout interno y el conjunto externo de prueba permanecen sin utilizar.

In [ ]:
# ---------------------------------------------------------
# Construcción de rankings por métrica
# ---------------------------------------------------------

tabla_comparacion_modelos = (
    resumen_cv_modelos.copy()
)

# Métricas en las que un valor mayor es mejor.
metricas_mayor_mejor = [
    "average_precision_media",
    "auc_roc_media",
    "f1_media",
    "f2_media",
    "recall_media",
    "precision_media",
    "balanced_accuracy_media"
]

# Métricas en las que un valor menor es mejor.
metricas_menor_mejor = [
    "brier_media",
    "log_loss_media",
    "average_precision_desviacion",
    "auc_roc_desviacion",
    "tiempo_total_segundos"
]

for metrica in metricas_mayor_mejor:
    tabla_comparacion_modelos[
        f"ranking__{metrica}"
    ] = (
        tabla_comparacion_modelos[metrica]
        .rank(
            method="min",
            ascending=False
        )
        .astype(int)
    )

for metrica in metricas_menor_mejor:
    tabla_comparacion_modelos[
        f"ranking__{metrica}"
    ] = (
        tabla_comparacion_modelos[metrica]
        .rank(
            method="min",
            ascending=True
        )
        .astype(int)
    )


# ---------------------------------------------------------
# Ranking de consenso
# ---------------------------------------------------------

columnas_ranking_consenso = [
    "ranking__average_precision_media",
    "ranking__auc_roc_media",
    "ranking__f1_media",
    "ranking__f2_media",
    "ranking__brier_media",
    "ranking__log_loss_media"
]

tabla_comparacion_modelos[
    "ranking_consenso_medio"
] = (
    tabla_comparacion_modelos[
        columnas_ranking_consenso
    ]
    .mean(axis=1)
)

tabla_comparacion_modelos = (
    tabla_comparacion_modelos
    .sort_values(
        [
            "ranking_consenso_medio",
            "average_precision_media",
            "auc_roc_media"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

tabla_comparacion_modelos.insert(
    0,
    "posicion_consenso",
    np.arange(
        1,
        len(tabla_comparacion_modelos) + 1
    )
)


# ---------------------------------------------------------
# Identificación provisional de finalistas
# ---------------------------------------------------------

NUMERO_FINALISTAS = 3

modelos_finalistas_provisionales = (
    tabla_comparacion_modelos
    .head(NUMERO_FINALISTAS)["modelo"]
    .tolist()
)


# ---------------------------------------------------------
# Presentación de resultados
# ---------------------------------------------------------

columnas_presentacion = [
    "posicion_consenso",
    "modelo",
    "ranking_consenso_medio",
    "average_precision_media",
    "average_precision_desviacion",
    "auc_roc_media",
    "auc_roc_desviacion",
    "precision_media",
    "recall_media",
    "f1_media",
    "f2_media",
    "balanced_accuracy_media",
    "brier_media",
    "log_loss_media",
    "tiempo_total_segundos"
]

display(
    tabla_comparacion_modelos[
        columnas_presentacion
    ].style.format({
        "ranking_consenso_medio": "{:.2f}",
        "average_precision_media": "{:.4f}",
        "average_precision_desviacion": "{:.4f}",
        "auc_roc_media": "{:.4f}",
        "auc_roc_desviacion": "{:.4f}",
        "precision_media": "{:.4f}",
        "recall_media": "{:.4f}",
        "f1_media": "{:.4f}",
        "f2_media": "{:.4f}",
        "balanced_accuracy_media": "{:.4f}",
        "brier_media": "{:.4f}",
        "log_loss_media": "{:.4f}",
        "tiempo_total_segundos": "{:,.2f}"
    })
)

print(
    "Modelos finalistas provisionales:"
)

for posicion, modelo in enumerate(
    modelos_finalistas_provisionales,
    start=1
):
    print(
        f"{posicion}. {modelo}"
    )


# ---------------------------------------------------------
# Comparación con el baseline
# ---------------------------------------------------------

fila_baseline = (
    tabla_comparacion_modelos.loc[
        tabla_comparacion_modelos["modelo"]
        == "dummy_prior"
    ]
    .iloc[0]
)

comparacion_baseline = (
    tabla_comparacion_modelos.loc[
        tabla_comparacion_modelos["modelo"]
        != "dummy_prior",
        [
            "modelo",
            "average_precision_media",
            "auc_roc_media",
            "f1_media",
            "recall_media",
            "brier_media",
            "log_loss_media"
        ]
    ]
    .copy()
)

comparacion_baseline[
    "mejora_average_precision"
] = (
    comparacion_baseline[
        "average_precision_media"
    ]
    - fila_baseline[
        "average_precision_media"
    ]
)

comparacion_baseline[
    "mejora_auc_roc"
] = (
    comparacion_baseline[
        "auc_roc_media"
    ]
    - fila_baseline[
        "auc_roc_media"
    ]
)

display(
    comparacion_baseline
    .sort_values(
        "mejora_average_precision",
        ascending=False
    )
)


# ---------------------------------------------------------
# Almacenamiento de evidencias
# ---------------------------------------------------------

RUTA_RESULTADOS_CV_PLIEGUES = (
    RUTA_RESULTADOS
    / "modelado_resultados_cv_pliegues.csv"
)

RUTA_RESUMEN_CV_MODELOS = (
    RUTA_RESULTADOS
    / "modelado_resumen_cv_modelos.csv"
)

RUTA_RANKING_MODELOS = (
    RUTA_RESULTADOS
    / "modelado_ranking_consenso.csv"
)

resultados_cv_pliegues.to_csv(
    RUTA_RESULTADOS_CV_PLIEGUES,
    index=False
)

resumen_cv_modelos.to_csv(
    RUTA_RESUMEN_CV_MODELOS,
    index=False
)

tabla_comparacion_modelos.to_csv(
    RUTA_RANKING_MODELOS,
    index=False
)


# ---------------------------------------------------------
# Validación del bloque
# ---------------------------------------------------------

archivos_comparacion = [
    RUTA_RESULTADOS_CV_PLIEGUES,
    RUTA_RESUMEN_CV_MODELOS,
    RUTA_RANKING_MODELOS
]

analisis_comparacion_valido = (
    len(tabla_comparacion_modelos) == 6
    and tabla_comparacion_modelos[
        "modelo"
    ].nunique() == 6
    and len(
        modelos_finalistas_provisionales
    ) == 3
    and all(
        ruta.exists()
        for ruta in archivos_comparacion
    )
)

print(
    "\nResultados por pliegue guardados:",
    RUTA_RESULTADOS_CV_PLIEGUES.exists()
)

print(
    "Resumen de modelos guardado:",
    RUTA_RESUMEN_CV_MODELOS.exists()
)

print(
    "Ranking de consenso guardado:",
    RUTA_RANKING_MODELOS.exists()
)

print(
    "¿El análisis de la comparación inicial es válido?",
    analisis_comparacion_valido
)

In [ ]:
from sklearn.metrics import log_loss

## Generación de probabilidades fuera de muestra para los modelos finalistas

La comparación inicial identifica como finalistas provisionales a `xgboost`, `logistic_l2` y `logistic_balanced`. Antes de seleccionar el umbral de clasificación, se generan probabilidades para cada cliente mediante modelos que no han sido entrenados con sus propios registros.

Para ello, se utilizan predicciones *out-of-fold*. En cada uno de los cinco pliegues, el modelo se ajusta con cuatro partes del conjunto de desarrollo y estima las probabilidades de la parte restante. Al finalizar el proceso, cada uno de los 44.203 clientes dispone de una probabilidad generada por un modelo que no utilizó sus datos durante el entrenamiento.

Estas probabilidades permiten:

- Comparar en condiciones equivalentes la capacidad discriminativa de los modelos finalistas.
- Estudiar la distribución de los riesgos estimados.
- Evaluar la calidad de las probabilidades mediante el `Brier score` y la `log loss`.
- Seleccionar posteriormente un umbral sin utilizar el holdout interno.
- Evitar el optimismo asociado a la evaluación de un modelo sobre sus propios datos de entrenamiento.

El umbral convencional de `0,5` se evalúa únicamente como referencia y no constituye todavía el umbral operativo definitivo.

Como resultado, se obtiene una tabla con los 44.203 clientes del conjunto de desarrollo, la variable objetivo real y una columna con la probabilidad estimada por cada uno de los tres modelos finalistas.

In [ ]:
from sklearn.model_selection import cross_val_predict


# ---------------------------------------------------------
# Comprobación de los finalistas
# ---------------------------------------------------------

FINALISTAS_ESPERADOS = [
    "xgboost",
    "logistic_l2",
    "logistic_balanced"
]

if modelos_finalistas_provisionales != FINALISTAS_ESPERADOS:
    raise ValueError(
        "Los modelos finalistas no coinciden con el orden "
        "esperado del ranking."
    )


# ---------------------------------------------------------
# Tabla base de predicciones fuera de muestra
# ---------------------------------------------------------

predicciones_oof_finalistas = pd.DataFrame({
    "customer_id": (
        datos_desarrollo["customer_id"]
        .to_numpy()
    ),
    "churn_real": (
        y_desarrollo.to_numpy()
    )
})

resumen_oof_finalistas = []


# ---------------------------------------------------------
# Generación secuencial de probabilidades OOF
# ---------------------------------------------------------

for numero_modelo, nombre_modelo in enumerate(
    modelos_finalistas_provisionales,
    start=1
):

    print("\n" + "=" * 72)
    print(
        f"FINALISTA {numero_modelo} DE "
        f"{len(modelos_finalistas_provisionales)}"
    )
    print(nombre_modelo)
    print("=" * 72)

    modelo = clone(
        modelos_candidatos[nombre_modelo]
    )

    inicio_modelo = perf_counter()

    probabilidades_oof = cross_val_predict(
        estimator=modelo,
        X=X_desarrollo,
        y=y_desarrollo,
        cv=validacion_cruzada,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    tiempo_total = (
        perf_counter() - inicio_modelo
    )

    if not np.isfinite(
        probabilidades_oof
    ).all():
        raise ValueError(
            f"{nombre_modelo}: existen probabilidades "
            "no finitas."
        )

    if not (
        (probabilidades_oof >= 0)
        & (probabilidades_oof <= 1)
    ).all():
        raise ValueError(
            f"{nombre_modelo}: existen probabilidades "
            "fuera del intervalo [0, 1]."
        )

    predicciones_oof_finalistas[
        f"probabilidad__{nombre_modelo}"
    ] = probabilidades_oof

    predicciones_umbral_05 = (
        probabilidades_oof >= 0.5
    ).astype(np.int8)

    matriz_confusion_05 = confusion_matrix(
        y_desarrollo,
        predicciones_umbral_05,
        labels=[0, 1]
    )

    tn, fp, fn, tp = (
        matriz_confusion_05.ravel()
    )

    resumen_oof_finalistas.append({
        "modelo": nombre_modelo,
        "auc_roc": roc_auc_score(
            y_desarrollo,
            probabilidades_oof
        ),
        "average_precision": (
            average_precision_score(
                y_desarrollo,
                probabilidades_oof
            )
        ),
        "brier": np.mean(
            (
                probabilidades_oof
                - y_desarrollo.to_numpy()
            ) ** 2
        ),
        "log_loss": log_loss(
            y_desarrollo,
            probabilidades_oof,
            labels=[0, 1]
        ),
        "accuracy_umbral_05": accuracy_score(
            y_desarrollo,
            predicciones_umbral_05
        ),
        "balanced_accuracy_umbral_05": (
            (
                tp / (tp + fn)
                + tn / (tn + fp)
            )
            / 2
        ),
        "precision_umbral_05": precision_score(
            y_desarrollo,
            predicciones_umbral_05,
            zero_division=0
        ),
        "recall_umbral_05": recall_score(
            y_desarrollo,
            predicciones_umbral_05,
            zero_division=0
        ),
        "f1_umbral_05": f1_score(
            y_desarrollo,
            predicciones_umbral_05,
            zero_division=0
        ),
        "f2_umbral_05": fbeta_score(
            y_desarrollo,
            predicciones_umbral_05,
            beta=2,
            zero_division=0
        ),
        "verdaderos_negativos": int(tn),
        "falsos_positivos": int(fp),
        "falsos_negativos": int(fn),
        "verdaderos_positivos": int(tp),
        "probabilidad_media": float(
            probabilidades_oof.mean()
        ),
        "probabilidad_minima": float(
            probabilidades_oof.min()
        ),
        "probabilidad_maxima": float(
            probabilidades_oof.max()
        ),
        "tiempo_total_segundos": float(
            tiempo_total
        )
    })

    print(
        "AUC-ROC:",
        f"{resumen_oof_finalistas[-1]['auc_roc']:.4f}"
    )

    print(
        "Average precision:",
        f"{resumen_oof_finalistas[-1]['average_precision']:.4f}"
    )

    print(
        "Recall con umbral 0,5:",
        f"{resumen_oof_finalistas[-1]['recall_umbral_05']:.4f}"
    )

    print(
        "Tiempo total:",
        f"{tiempo_total:,.2f} segundos"
    )


# ---------------------------------------------------------
# Resumen comparativo
# ---------------------------------------------------------

resumen_oof_finalistas = pd.DataFrame(
    resumen_oof_finalistas
)

resumen_oof_finalistas = (
    resumen_oof_finalistas
    .sort_values(
        [
            "average_precision",
            "auc_roc"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

resumen_oof_finalistas.insert(
    0,
    "posicion",
    np.arange(
        1,
        len(resumen_oof_finalistas) + 1
    )
)

columnas_presentacion_oof = [
    "posicion",
    "modelo",
    "average_precision",
    "auc_roc",
    "brier",
    "log_loss",
    "precision_umbral_05",
    "recall_umbral_05",
    "f1_umbral_05",
    "f2_umbral_05",
    "balanced_accuracy_umbral_05",
    "probabilidad_media",
    "tiempo_total_segundos"
]

display(
    resumen_oof_finalistas[
        columnas_presentacion_oof
    ].style.format({
        "average_precision": "{:.4f}",
        "auc_roc": "{:.4f}",
        "brier": "{:.4f}",
        "log_loss": "{:.4f}",
        "precision_umbral_05": "{:.4f}",
        "recall_umbral_05": "{:.4f}",
        "f1_umbral_05": "{:.4f}",
        "f2_umbral_05": "{:.4f}",
        "balanced_accuracy_umbral_05": "{:.4f}",
        "probabilidad_media": "{:.4f}",
        "tiempo_total_segundos": "{:,.2f}"
    })
)


# ---------------------------------------------------------
# Validación del bloque
# ---------------------------------------------------------

columnas_probabilidad = [
    f"probabilidad__{modelo}"
    for modelo in modelos_finalistas_provisionales
]

predicciones_oof_validas = (
    predicciones_oof_finalistas.shape
        == (44_203, 5)
    and predicciones_oof_finalistas[
        "customer_id"
    ].nunique() == 44_203
    and predicciones_oof_finalistas[
        "churn_real"
    ].isin([0, 1]).all()
    and predicciones_oof_finalistas[
        columnas_probabilidad
    ].notna().all().all()
    and np.isfinite(
        predicciones_oof_finalistas[
            columnas_probabilidad
        ].to_numpy()
    ).all()
    and (
        predicciones_oof_finalistas[
            columnas_probabilidad
        ].to_numpy() >= 0
    ).all()
    and (
        predicciones_oof_finalistas[
            columnas_probabilidad
        ].to_numpy() <= 1
    ).all()
    and resumen_oof_finalistas.shape[0] == 3
)

print(
    "\nDimensión de la tabla OOF:",
    predicciones_oof_finalistas.shape
)

print(
    "Clientes únicos:",
    predicciones_oof_finalistas[
        "customer_id"
    ].nunique()
)

print(
    "Modelos finalistas evaluados:",
    len(resumen_oof_finalistas)
)

print(
    "¿Las probabilidades fuera de muestra son válidas?",
    predicciones_oof_validas
)

## Análisis de umbrales para los modelos finalistas

Los modelos generan probabilidades de `churn`, pero su conversión en decisiones binarias requiere establecer un umbral de clasificación. Un cliente se clasifica en la clase positiva cuando su probabilidad estimada es igual o superior al umbral seleccionado.

El valor convencional de `0,5` no tiene por qué ser el más adecuado para el objetivo operativo. Un umbral elevado suele reducir las falsas alertas, pero también puede dejar sin identificar a clientes que realmente abandonan. Por el contrario, un umbral más bajo suele aumentar el `recall`, aunque normalmente produce más falsos positivos.

En este bloque se evalúan umbrales comprendidos entre `0,05` y `0,95` para los tres modelos finalistas. El análisis utiliza exclusivamente las probabilidades *out-of-fold* generadas dentro del conjunto de desarrollo. El holdout interno y el conjunto externo de prueba permanecen aislados.

Para cada combinación de modelo y umbral se calculan:

- `precision`.
- `recall` o sensibilidad.
- Especificidad.
- F1.
- F2, que concede mayor importancia al `recall`.
- Exactitud.
- Exactitud balanceada.
- Proporción de clientes clasificados en la clase positiva.
- Verdaderos positivos y falsos positivos.
- Verdaderos negativos y falsos negativos.

Además, se identifica para cada modelo el umbral que maximiza F1, F2 y la exactitud balanceada. Estos resultados no determinan todavía la elección definitiva, sino que permiten analizar el equilibrio entre la detección de clientes con `churn` y la generación de falsas alertas antes de fijar la regla operativa.

In [ ]:
from sklearn.metrics import balanced_accuracy_score


# ---------------------------------------------------------
# Definición de los umbrales que se evaluarán
# ---------------------------------------------------------

umbrales_evaluados = np.round(
    np.arange(
        0.05,
        0.951,
        0.01
    ),
    2
)

resultados_umbrales = []


# ---------------------------------------------------------
# Evaluación de cada modelo y cada umbral
# ---------------------------------------------------------

for nombre_modelo in modelos_finalistas_provisionales:

    columna_probabilidad = (
        f"probabilidad__{nombre_modelo}"
    )

    probabilidades = (
        predicciones_oof_finalistas[
            columna_probabilidad
        ]
        .to_numpy()
    )

    y_real = (
        predicciones_oof_finalistas[
            "churn_real"
        ]
        .to_numpy()
    )

    for umbral in umbrales_evaluados:

        predicciones = (
            probabilidades >= umbral
        ).astype(np.int8)

        tn, fp, fn, tp = confusion_matrix(
            y_real,
            predicciones,
            labels=[0, 1]
        ).ravel()

        sensibilidad = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        especificidad = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0.0
        )

        resultados_umbrales.append({
            "modelo": nombre_modelo,
            "umbral": float(umbral),
            "accuracy": accuracy_score(
                y_real,
                predicciones
            ),
            "balanced_accuracy": (
                balanced_accuracy_score(
                    y_real,
                    predicciones
                )
            ),
            "precision": precision_score(
                y_real,
                predicciones,
                zero_division=0
            ),
            "recall": recall_score(
                y_real,
                predicciones,
                zero_division=0
            ),
            "especificidad": especificidad,
            "f1": f1_score(
                y_real,
                predicciones,
                zero_division=0
            ),
            "f2": fbeta_score(
                y_real,
                predicciones,
                beta=2,
                zero_division=0
            ),
            "tasa_alertas": (
                predicciones.mean()
            ),
            "verdaderos_negativos": int(tn),
            "falsos_positivos": int(fp),
            "falsos_negativos": int(fn),
            "verdaderos_positivos": int(tp)
        })


tabla_umbrales = pd.DataFrame(
    resultados_umbrales
)


# ---------------------------------------------------------
# Identificación de los mejores umbrales por criterio
# ---------------------------------------------------------

def seleccionar_mejor_umbral(
    tabla,
    metrica
):
    """
    Selecciona el umbral con mayor valor de la métrica.

    En caso de empate se escoge el umbral más alto,
    reduciendo el número de alertas.
    """
    return (
        tabla
        .sort_values(
            [metrica, "umbral"],
            ascending=[False, False]
        )
        .iloc[0]
    )


resumen_mejores_umbrales = []

for nombre_modelo in modelos_finalistas_provisionales:

    tabla_modelo = (
        tabla_umbrales.loc[
            tabla_umbrales["modelo"]
            == nombre_modelo
        ]
        .copy()
    )

    for criterio in [
        "f1",
        "f2",
        "balanced_accuracy"
    ]:

        mejor_fila = seleccionar_mejor_umbral(
            tabla_modelo,
            criterio
        )

        resumen_mejores_umbrales.append({
            "modelo": nombre_modelo,
            "criterio_optimizado": criterio,
            "umbral": mejor_fila["umbral"],
            "precision": mejor_fila["precision"],
            "recall": mejor_fila["recall"],
            "especificidad": (
                mejor_fila["especificidad"]
            ),
            "f1": mejor_fila["f1"],
            "f2": mejor_fila["f2"],
            "balanced_accuracy": (
                mejor_fila[
                    "balanced_accuracy"
                ]
            ),
            "tasa_alertas": (
                mejor_fila["tasa_alertas"]
            ),
            "falsos_positivos": int(
                mejor_fila[
                    "falsos_positivos"
                ]
            ),
            "falsos_negativos": int(
                mejor_fila[
                    "falsos_negativos"
                ]
            )
        })


resumen_mejores_umbrales = pd.DataFrame(
    resumen_mejores_umbrales
)


# ---------------------------------------------------------
# Presentación de resultados
# ---------------------------------------------------------

print(
    "Umbrales evaluados por modelo:",
    len(umbrales_evaluados)
)

print(
    "Combinaciones modelo-umbral:",
    len(tabla_umbrales)
)

display(
    resumen_mejores_umbrales.style.format({
        "umbral": "{:.2f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "especificidad": "{:.4f}",
        "f1": "{:.4f}",
        "f2": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "tasa_alertas": "{:.2%}"
    })
)


print(
    "\nMejores umbrales según F2:"
)

display(
    resumen_mejores_umbrales.loc[
        resumen_mejores_umbrales[
            "criterio_optimizado"
        ] == "f2"
    ]
    .sort_values(
        "f2",
        ascending=False
    )
    .style.format({
        "umbral": "{:.2f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "especificidad": "{:.4f}",
        "f1": "{:.4f}",
        "f2": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "tasa_alertas": "{:.2%}"
    })
)


# ---------------------------------------------------------
# Almacenamiento de las evidencias
# ---------------------------------------------------------

RUTA_ANALISIS_UMBRALES = (
    RUTA_RESULTADOS
    / "modelado_analisis_umbrales_oof.csv"
)

RUTA_MEJORES_UMBRALES = (
    RUTA_RESULTADOS
    / "modelado_mejores_umbrales_oof.csv"
)

tabla_umbrales.to_csv(
    RUTA_ANALISIS_UMBRALES,
    index=False
)

resumen_mejores_umbrales.to_csv(
    RUTA_MEJORES_UMBRALES,
    index=False
)


# ---------------------------------------------------------
# Validación del bloque
# ---------------------------------------------------------

analisis_umbrales_valido = (
    len(umbrales_evaluados) == 91
    and tabla_umbrales.shape[0] == 273
    and tabla_umbrales[
        "modelo"
    ].nunique() == 3
    and tabla_umbrales[
        "umbral"
    ].between(0.05, 0.95).all()
    and resumen_mejores_umbrales.shape[0] == 9
    and np.isfinite(
        tabla_umbrales[
            [
                "accuracy",
                "balanced_accuracy",
                "precision",
                "recall",
                "especificidad",
                "f1",
                "f2",
                "tasa_alertas"
            ]
        ].to_numpy()
    ).all()
    and RUTA_ANALISIS_UMBRALES.exists()
    and RUTA_MEJORES_UMBRALES.exists()
)

print(
    "\n¿El análisis de umbrales es válido?",
    analisis_umbrales_valido
)

## Selección y congelación de la configuración final

Una vez generadas las probabilidades *out-of-fold* y analizados los diferentes umbrales, se selecciona una única configuración compuesta por:

- El algoritmo.
- Sus hiperparámetros.
- El conjunto de 176 variables predictoras.
- La ventana temporal utilizada.
- El umbral de clasificación.

La selección se realiza exclusivamente con los 44.203 clientes del conjunto de desarrollo. El holdout interno y el conjunto externo de prueba permanecen sin utilizar.

La métrica principal de selección es F2, que concede mayor importancia al `recall` que a la `precision`. Este criterio es coherente con un sistema de detección temprana en el que no identificar a un cliente que abandona puede resultar más perjudicial que generar una alerta adicional.

Para cada modelo finalista se selecciona el umbral que maximiza F2. A continuación, las configuraciones resultantes se comparan aplicando, en orden, los siguientes criterios:

1. Mayor F2.
2. Mayor `recall`, en caso de empate en F2.
3. Mayor `precision`, si persiste el empate.
4. Menor proporción de clientes clasificados en la clase positiva, cuando las métricas anteriores sean equivalentes.

La configuración seleccionada queda congelada antes de consultar el holdout interno. A partir de este momento, sus componentes no se modifican en función de los resultados obtenidos sobre dicho conjunto.

Finalmente, la configuración y los criterios utilizados para seleccionarla se almacenan en un archivo JSON, garantizando la trazabilidad de la decisión metodológica.

In [ ]:
# ---------------------------------------------------------
# Obtención de la mejor configuración F2 de cada finalista
# ---------------------------------------------------------

candidatos_f2 = (
    resumen_mejores_umbrales.loc[
        resumen_mejores_umbrales[
            "criterio_optimizado"
        ] == "f2"
    ]
    .copy()
)

candidatos_f2 = (
    candidatos_f2
    .sort_values(
        [
            "f2",
            "recall",
            "precision",
            "umbral"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

candidatos_f2.insert(
    0,
    "posicion_seleccion",
    np.arange(
        1,
        len(candidatos_f2) + 1
    )
)


# ---------------------------------------------------------
# Selección de la configuración ganadora
# ---------------------------------------------------------

configuracion_seleccionada = (
    candidatos_f2.iloc[0]
)

MODELO_SELECCIONADO = str(
    configuracion_seleccionada["modelo"]
)

UMBRAL_SELECCIONADO = float(
    configuracion_seleccionada["umbral"]
)

CRITERIO_SELECCION = "max_f2_oof"

VENTANA_SELECCIONADA_DIAS = 90

VARIABLES_SELECCIONADAS = (
    columnas_predictoras.copy()
)


# ---------------------------------------------------------
# Presentación de los candidatos
# ---------------------------------------------------------

print(
    "Configuraciones candidatas ordenadas por F2:"
)

display(
    candidatos_f2.style.format({
        "umbral": "{:.2f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "especificidad": "{:.4f}",
        "f1": "{:.4f}",
        "f2": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "tasa_alertas": "{:.2%}"
    })
)


print(
    "\nModelo seleccionado:",
    MODELO_SELECCIONADO
)

print(
    "Umbral seleccionado:",
    f"{UMBRAL_SELECCIONADO:.2f}"
)

print(
    "Criterio de selección:",
    CRITERIO_SELECCION
)

print(
    "F2 estimado mediante OOF:",
    f"{configuracion_seleccionada['f2']:.4f}"
)

print(
    "Recall estimado mediante OOF:",
    f"{configuracion_seleccionada['recall']:.4f}"
)

print(
    "Precisión estimada mediante OOF:",
    f"{configuracion_seleccionada['precision']:.4f}"
)

print(
    "Tasa estimada de alertas:",
    f"{configuracion_seleccionada['tasa_alertas']:.2%}"
)


# ---------------------------------------------------------
# Recuperación de los hiperparámetros del modelo
# ---------------------------------------------------------

modelo_base_seleccionado = clone(
    modelos_candidatos[
        MODELO_SELECCIONADO
    ]
)

parametros_modelo_seleccionado = (
    modelo_base_seleccionado.get_params()
)

parametros_serializables = {
    clave: (
        valor
        if isinstance(
            valor,
            (
                str,
                int,
                float,
                bool,
                type(None)
            )
        )
        else str(valor)
    )
    for clave, valor
    in parametros_modelo_seleccionado.items()
}


# ---------------------------------------------------------
# Construcción del manifiesto congelado
# ---------------------------------------------------------

manifiesto_configuracion_final = {
    "selection_status": "frozen_before_holdout",
    "selection_criterion": CRITERIO_SELECCION,
    "selected_model": MODELO_SELECCIONADO,
    "selected_threshold": UMBRAL_SELECCIONADO,
    "window_days": VENTANA_SELECCIONADA_DIAS,
    "selected_features": len(
        VARIABLES_SELECCIONADAS
    ),
    "development_clients": int(
        len(X_desarrollo)
    ),
    "internal_holdout_clients": int(
        len(X_holdout)
    ),
    "external_test_clients": int(
        len(X_test_externo)
    ),
    "cv_folds": int(
        NUMERO_PLIEGUES
    ),
    "cv_random_state": int(
        SEMILLA_CV
    ),
    "model_random_state": int(
        SEMILLA_MODELOS
    ),
    "oof_f2": float(
        configuracion_seleccionada["f2"]
    ),
    "oof_recall": float(
        configuracion_seleccionada["recall"]
    ),
    "oof_precision": float(
        configuracion_seleccionada["precision"]
    ),
    "oof_specificity": float(
        configuracion_seleccionada[
            "especificidad"
        ]
    ),
    "oof_balanced_accuracy": float(
        configuracion_seleccionada[
            "balanced_accuracy"
        ]
    ),
    "oof_alert_rate": float(
        configuracion_seleccionada[
            "tasa_alertas"
        ]
    ),
    "model_parameters": (
        parametros_serializables
    )
}


# ---------------------------------------------------------
# Almacenamiento de la configuración congelada
# ---------------------------------------------------------

RUTA_CONFIGURACION_FINAL = (
    RUTA_RESULTADOS
    / "modelado_configuracion_congelada.json"
)

RUTA_CANDIDATOS_F2 = (
    RUTA_RESULTADOS
    / "modelado_candidatos_f2_oof.csv"
)

with open(
    RUTA_CONFIGURACION_FINAL,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        manifiesto_configuracion_final,
        archivo,
        ensure_ascii=False,
        indent=4
    )

candidatos_f2.to_csv(
    RUTA_CANDIDATOS_F2,
    index=False
)


# ---------------------------------------------------------
# Validación del bloque
# ---------------------------------------------------------

configuracion_congelada_valida = (
    len(candidatos_f2) == 3
    and MODELO_SELECCIONADO
        in modelos_finalistas_provisionales
    and 0.05
        <= UMBRAL_SELECCIONADO
        <= 0.95
    and len(
        VARIABLES_SELECCIONADAS
    ) == 176
    and CRITERIO_SELECCION
        == "max_f2_oof"
    and RUTA_CONFIGURACION_FINAL.exists()
    and RUTA_CANDIDATOS_F2.exists()
)

print(
    "\n¿La configuración final se ha congelado correctamente?",
    configuracion_congelada_valida
)

## Evaluación de la configuración congelada en el holdout interno

La selección realizada exclusivamente a partir de las predicciones *out-of-fold* del conjunto de desarrollo determina la siguiente configuración:

- Modelo: `xgboost`.
- Umbral de clasificación: `0,15`.
- Criterio principal de selección: maximización de F2.
- Variables predictoras: 176.
- Ventana temporal: 90 días.

Una vez congelada la configuración, se realiza una única evaluación sobre los 7.801 clientes del holdout interno. Este conjunto no ha intervenido en la comparación de algoritmos, el ajuste de hiperparámetros, el análisis de umbrales ni la selección del modelo.

El modelo se entrena desde cero utilizando exclusivamente los 44.203 clientes del conjunto de desarrollo. A continuación, genera probabilidades de `churn` para los clientes del holdout interno, que se convierten en predicciones binarias mediante el umbral previamente fijado de `0,15`.

La evaluación incluye métricas independientes del umbral, como `AUC-ROC`, `average_precision`, `Brier score` y `log loss`, y métricas dependientes del umbral:

- `precision`.
- `recall`.
- Especificidad.
- F1.
- F2.
- Exactitud.
- Exactitud balanceada.

También se obtiene la matriz de confusión y la proporción de clientes clasificados en la clase positiva. Los resultados se comparan de forma descriptiva con las estimaciones *out-of-fold*, pero no se utilizan para modificar la configuración congelada.

El conjunto externo de prueba permanece completamente aislado y no interviene en esta evaluación.

In [ ]:
from sklearn.metrics import balanced_accuracy_score


# ---------------------------------------------------------
# Comprobación de la configuración congelada
# ---------------------------------------------------------

if MODELO_SELECCIONADO != "xgboost":
    raise ValueError(
        "El modelo congelado no coincide con xgboost."
    )

if not np.isclose(
    UMBRAL_SELECCIONADO,
    0.15
):
    raise ValueError(
        "El umbral congelado no coincide con 0.15."
    )

if not configuracion_congelada_valida:
    raise ValueError(
        "La configuración no figura como validada."
    )


# ---------------------------------------------------------
# Entrenamiento exclusivo con el conjunto de desarrollo
# ---------------------------------------------------------

modelo_holdout = clone(
    modelos_candidatos[
        MODELO_SELECCIONADO
    ]
)

inicio_entrenamiento_holdout = perf_counter()

modelo_holdout.fit(
    X_desarrollo,
    y_desarrollo
)

tiempo_entrenamiento_holdout = (
    perf_counter()
    - inicio_entrenamiento_holdout
)


# ---------------------------------------------------------
# Predicción sobre el holdout interno
# ---------------------------------------------------------

probabilidades_holdout = (
    modelo_holdout.predict_proba(
        X_holdout
    )[:, 1]
)

predicciones_holdout = (
    probabilidades_holdout
    >= UMBRAL_SELECCIONADO
).astype(np.int8)


# ---------------------------------------------------------
# Comprobación de las probabilidades
# ---------------------------------------------------------

if not np.isfinite(
    probabilidades_holdout
).all():
    raise ValueError(
        "Existen probabilidades no finitas "
        "en el holdout."
    )

if not (
    (probabilidades_holdout >= 0)
    & (probabilidades_holdout <= 1)
).all():
    raise ValueError(
        "Existen probabilidades fuera del "
        "intervalo [0, 1]."
    )


# ---------------------------------------------------------
# Matriz de confusión
# ---------------------------------------------------------

matriz_confusion_holdout = confusion_matrix(
    y_holdout,
    predicciones_holdout,
    labels=[0, 1]
)

tn_holdout, fp_holdout, fn_holdout, tp_holdout = (
    matriz_confusion_holdout.ravel()
)

sensibilidad_holdout = (
    tp_holdout
    / (tp_holdout + fn_holdout)
)

especificidad_holdout = (
    tn_holdout
    / (tn_holdout + fp_holdout)
)


# ---------------------------------------------------------
# Métricas del holdout
# ---------------------------------------------------------

metricas_holdout = {
    "modelo": MODELO_SELECCIONADO,
    "umbral": UMBRAL_SELECCIONADO,
    "clientes": int(
        len(X_holdout)
    ),
    "auc_roc": roc_auc_score(
        y_holdout,
        probabilidades_holdout
    ),
    "average_precision": (
        average_precision_score(
            y_holdout,
            probabilidades_holdout
        )
    ),
    "brier": float(
        np.mean(
            (
                probabilidades_holdout
                - y_holdout.to_numpy()
            ) ** 2
        )
    ),
    "log_loss": log_loss(
        y_holdout,
        probabilidades_holdout,
        labels=[0, 1]
    ),
    "accuracy": accuracy_score(
        y_holdout,
        predicciones_holdout
    ),
    "balanced_accuracy": (
        balanced_accuracy_score(
            y_holdout,
            predicciones_holdout
        )
    ),
    "precision": precision_score(
        y_holdout,
        predicciones_holdout,
        zero_division=0
    ),
    "recall": recall_score(
        y_holdout,
        predicciones_holdout,
        zero_division=0
    ),
    "especificidad": especificidad_holdout,
    "f1": f1_score(
        y_holdout,
        predicciones_holdout,
        zero_division=0
    ),
    "f2": fbeta_score(
        y_holdout,
        predicciones_holdout,
        beta=2,
        zero_division=0
    ),
    "tasa_alertas": float(
        predicciones_holdout.mean()
    ),
    "verdaderos_negativos": int(
        tn_holdout
    ),
    "falsos_positivos": int(
        fp_holdout
    ),
    "falsos_negativos": int(
        fn_holdout
    ),
    "verdaderos_positivos": int(
        tp_holdout
    ),
    "tiempo_entrenamiento_segundos": float(
        tiempo_entrenamiento_holdout
    )
}

metricas_holdout_df = pd.DataFrame(
    [metricas_holdout]
)


# ---------------------------------------------------------
# Presentación de las métricas
# ---------------------------------------------------------

columnas_metricas_holdout = [
    "modelo",
    "umbral",
    "clientes",
    "average_precision",
    "auc_roc",
    "brier",
    "log_loss",
    "precision",
    "recall",
    "especificidad",
    "f1",
    "f2",
    "balanced_accuracy",
    "accuracy",
    "tasa_alertas",
    "tiempo_entrenamiento_segundos"
]

display(
    metricas_holdout_df[
        columnas_metricas_holdout
    ].style.format({
        "umbral": "{:.2f}",
        "average_precision": "{:.4f}",
        "auc_roc": "{:.4f}",
        "brier": "{:.4f}",
        "log_loss": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "especificidad": "{:.4f}",
        "f1": "{:.4f}",
        "f2": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "accuracy": "{:.4f}",
        "tasa_alertas": "{:.2%}",
        "tiempo_entrenamiento_segundos": "{:,.2f}"
    })
)


# ---------------------------------------------------------
# Presentación de la matriz de confusión
# ---------------------------------------------------------

matriz_confusion_holdout_df = pd.DataFrame(
    matriz_confusion_holdout,
    index=[
        "Real: no churn",
        "Real: churn"
    ],
    columns=[
        "Predicho: no churn",
        "Predicho: churn"
    ]
)

print(
    "Matriz de confusión del holdout:"
)

display(
    matriz_confusion_holdout_df
)


# ---------------------------------------------------------
# Comparación descriptiva entre OOF y holdout
# ---------------------------------------------------------

comparacion_oof_holdout = pd.DataFrame({
    "metrica": [
        "precision",
        "recall",
        "f1",
        "f2",
        "balanced_accuracy",
        "tasa_alertas"
    ],
    "estimacion_oof": [
        float(
            configuracion_seleccionada[
                "precision"
            ]
        ),
        float(
            configuracion_seleccionada[
                "recall"
            ]
        ),
        float(
            configuracion_seleccionada[
                "f1"
            ]
        ),
        float(
            configuracion_seleccionada[
                "f2"
            ]
        ),
        float(
            configuracion_seleccionada[
                "balanced_accuracy"
            ]
        ),
        float(
            configuracion_seleccionada[
                "tasa_alertas"
            ]
        )
    ],
    "resultado_holdout": [
        metricas_holdout["precision"],
        metricas_holdout["recall"],
        metricas_holdout["f1"],
        metricas_holdout["f2"],
        metricas_holdout[
            "balanced_accuracy"
        ],
        metricas_holdout["tasa_alertas"]
    ]
})

comparacion_oof_holdout[
    "diferencia_holdout_menos_oof"
] = (
    comparacion_oof_holdout[
        "resultado_holdout"
    ]
    - comparacion_oof_holdout[
        "estimacion_oof"
    ]
)

print(
    "Comparación descriptiva entre OOF y holdout:"
)

display(
    comparacion_oof_holdout.style.format({
        "estimacion_oof": "{:.4f}",
        "resultado_holdout": "{:.4f}",
        "diferencia_holdout_menos_oof": "{:+.4f}"
    })
)


# ---------------------------------------------------------
# Tabla de predicciones del holdout
# ---------------------------------------------------------

predicciones_holdout_df = pd.DataFrame({
    "customer_id": (
        datos_holdout[
            "customer_id"
        ].to_numpy()
    ),
    "churn_real": (
        y_holdout.to_numpy()
    ),
    "probabilidad_churn": (
        probabilidades_holdout
    ),
    "umbral": UMBRAL_SELECCIONADO,
    "prediccion_churn": (
        predicciones_holdout
    )
})


# ---------------------------------------------------------
# Almacenamiento de evidencias
# ---------------------------------------------------------

RUTA_METRICAS_HOLDOUT = (
    RUTA_RESULTADOS
    / "modelado_metricas_holdout_interno.csv"
)

RUTA_PREDICCIONES_HOLDOUT = (
    RUTA_RESULTADOS
    / "modelado_predicciones_holdout_interno.csv"
)

RUTA_COMPARACION_OOF_HOLDOUT = (
    RUTA_RESULTADOS
    / "modelado_comparacion_oof_holdout.csv"
)

metricas_holdout_df.to_csv(
    RUTA_METRICAS_HOLDOUT,
    index=False
)

predicciones_holdout_df.to_csv(
    RUTA_PREDICCIONES_HOLDOUT,
    index=False
)

comparacion_oof_holdout.to_csv(
    RUTA_COMPARACION_OOF_HOLDOUT,
    index=False
)


# ---------------------------------------------------------
# Validación final del bloque
# ---------------------------------------------------------

evaluacion_holdout_valida = (
    len(predicciones_holdout_df) == 7_801
    and predicciones_holdout_df[
        "customer_id"
    ].nunique() == 7_801
    and predicciones_holdout_df[
        "probabilidad_churn"
    ].between(0, 1).all()
    and predicciones_holdout_df[
        "prediccion_churn"
    ].isin([0, 1]).all()
    and (
        tn_holdout
        + fp_holdout
        + fn_holdout
        + tp_holdout
    ) == 7_801
    and RUTA_METRICAS_HOLDOUT.exists()
    and RUTA_PREDICCIONES_HOLDOUT.exists()
    and RUTA_COMPARACION_OOF_HOLDOUT.exists()
)

print(
    "\nModelo evaluado:",
    MODELO_SELECCIONADO
)

print(
    "Umbral aplicado:",
    f"{UMBRAL_SELECCIONADO:.2f}"
)

print(
    "Clientes evaluados:",
    len(predicciones_holdout_df)
)

print(
    "F2 en holdout:",
    f"{metricas_holdout['f2']:.4f}"
)

print(
    "Recall en holdout:",
    f"{metricas_holdout['recall']:.4f}"
)

print(
    "Precisión en holdout:",
    f"{metricas_holdout['precision']:.4f}"
)

print(
    "¿La evaluación del holdout interno es válida?",
    evaluacion_holdout_valida
)

## Auditoría de separabilidad extrema y posibles fugas de información

La configuración congelada obtiene resultados muy elevados tanto con las predicciones *out-of-fold* como en el holdout interno. Aunque este rendimiento puede reflejar una señal conductual intensa, también podría indicar que alguna variable predictora reproduce de forma directa o indirecta la definición utilizada para construir la variable objetivo `churn`.

Antes de realizar la evaluación final sobre el conjunto externo de prueba, se lleva a cabo una auditoría específica de posibles fugas de información. Esta revisión no utiliza el conjunto externo ni modifica por sí misma el modelo, el umbral o las variables seleccionadas.

La auditoría examina los siguientes aspectos:

1. **Nombres de las variables.** Se comprueba que ninguna columna predictora contenga referencias explícitas a `churn`, a la variable objetivo o a la partición de los datos.
2. **Capacidad predictiva individual.** Se calcula el `AUC-ROC` de cada variable por separado. Una discriminación individual casi perfecta, tanto en sentido directo como inverso, puede indicar que la variable representa una regla estrechamente relacionada con la definición del objetivo.
3. **Importancia en el modelo.** Se identifican las variables a las que XGBoost asigna una mayor importancia para determinar si el rendimiento se concentra en un grupo reducido de predictores.
4. **Reglas simples de inactividad.** Se evalúa si variables como la ausencia de actividad reciente o una recencia elevada permiten reproducir casi por sí solas la etiqueta.

Una capacidad predictiva elevada no implica necesariamente una fuga de información. Por ejemplo, una ausencia prolongada de actividad puede constituir una señal legítima si se observa antes del periodo utilizado para definir el abandono. Sin embargo, existiría una fuga temporal si la etiqueta `churn` se hubiera construido utilizando total o parcialmente los mismos días empleados para generar las variables predictoras.

Por este motivo, los resultados de la auditoría se interpretan junto con la definición temporal de la variable objetivo. El conjunto externo de prueba permanece aislado hasta confirmar que las variables solo contienen información disponible antes del momento de predicción. Si se detecta una posible fuga, se revisará la preparación de los datos y se repetirá el proceso de selección antes de realizar la evaluación final.

In [ ]:
# ---------------------------------------------------------
# 1. Revisión de los nombres de las variables
# ---------------------------------------------------------

terminos_sospechosos = [
    "churn",
    "target",
    "label",
    "particion",
    "partition",
    "split"
]

variables_nombre_sospechoso = [
    variable
    for variable in columnas_predictoras
    if any(
        termino in variable.lower()
        for termino in terminos_sospechosos
    )
]

print(
    "Variables con nombres potencialmente sospechosos:",
    len(variables_nombre_sospechoso)
)

print(variables_nombre_sospechoso)


# ---------------------------------------------------------
# 2. AUC individual de cada variable
# ---------------------------------------------------------

def calcular_auc_individual(
    X,
    y
):
    resultados = []

    y_array = np.asarray(y)

    for variable in X.columns:

        valores = (
            X[variable]
            .to_numpy()
        )

        if np.unique(valores).size < 2:
            auc_directo = 0.5
        else:
            auc_directo = roc_auc_score(
                y_array,
                valores
            )

        auc_orientado = max(
            auc_directo,
            1 - auc_directo
        )

        direccion = (
            "mayor_valor_mayor_riesgo"
            if auc_directo >= 0.5
            else "menor_valor_mayor_riesgo"
        )

        resultados.append({
            "variable": variable,
            "auc_directo": auc_directo,
            "auc_orientado": auc_orientado,
            "direccion": direccion
        })

    return (
        pd.DataFrame(resultados)
        .sort_values(
            "auc_orientado",
            ascending=False
        )
        .reset_index(drop=True)
    )


auc_individual_desarrollo = (
    calcular_auc_individual(
        X_desarrollo,
        y_desarrollo
    )
)

auc_individual_holdout = (
    calcular_auc_individual(
        X_holdout,
        y_holdout
    )
    .rename(columns={
        "auc_directo": "auc_directo_holdout",
        "auc_orientado": "auc_orientado_holdout",
        "direccion": "direccion_holdout"
    })
)

comparacion_auc_individual = (
    auc_individual_desarrollo
    .merge(
        auc_individual_holdout,
        on="variable",
        how="inner",
        validate="one_to_one"
    )
)

print(
    "\nVariables con AUC individual >= 0.95 "
    "en desarrollo:",
    (
        comparacion_auc_individual[
            "auc_orientado"
        ] >= 0.95
    ).sum()
)

print(
    "Variables con AUC individual >= 0.98 "
    "en desarrollo:",
    (
        comparacion_auc_individual[
            "auc_orientado"
        ] >= 0.98
    ).sum()
)

print(
    "\nVariables con mayor capacidad "
    "predictiva individual:"
)

display(
    comparacion_auc_individual[
        [
            "variable",
            "auc_orientado",
            "auc_orientado_holdout",
            "direccion",
            "direccion_holdout"
        ]
    ]
    .sort_values(
        "auc_orientado",
        ascending=False
    )
    .head(20)
    .style.format({
        "auc_orientado": "{:.4f}",
        "auc_orientado_holdout": "{:.4f}"
    })
)


# ---------------------------------------------------------
# 3. Importancia de variables del modelo XGBoost
# ---------------------------------------------------------

importancias_modelo = pd.DataFrame({
    "variable": columnas_predictoras,
    "importancia": (
        modelo_holdout.feature_importances_
    )
})

importancias_modelo = (
    importancias_modelo
    .sort_values(
        "importancia",
        ascending=False
    )
    .reset_index(drop=True)
)

importancias_modelo[
    "importancia_acumulada"
] = (
    importancias_modelo[
        "importancia"
    ].cumsum()
)

print(
    "\nVariables más importantes para XGBoost:"
)

display(
    importancias_modelo
    .head(20)
    .style.format({
        "importancia": "{:.6f}",
        "importancia_acumulada": "{:.4f}"
    })
)


# ---------------------------------------------------------
# 4. Evaluación de reglas simples de inactividad
# ---------------------------------------------------------

reglas_simples = {}

if "agg_global__last7" in columnas_predictoras:
    reglas_simples[
        "actividad_global_ultimos_7_igual_0"
    ] = (
        lambda X:
        X["agg_global__last7"]
        .eq(0)
        .astype(np.int8)
    )

if "agg_global__last14" in columnas_predictoras:
    reglas_simples[
        "actividad_global_ultimos_14_igual_0"
    ] = (
        lambda X:
        X["agg_global__last14"]
        .eq(0)
        .astype(np.int8)
    )

if "agg_global__recency" in columnas_predictoras:
    reglas_simples[
        "recencia_global_mayor_igual_7"
    ] = (
        lambda X:
        X["agg_global__recency"]
        .ge(7)
        .astype(np.int8)
    )

    reglas_simples[
        "recencia_global_mayor_igual_14"
    ] = (
        lambda X:
        X["agg_global__recency"]
        .ge(14)
        .astype(np.int8)
    )

if "agg_global__total" in columnas_predictoras:
    reglas_simples[
        "actividad_global_total_igual_0"
    ] = (
        lambda X:
        X["agg_global__total"]
        .eq(0)
        .astype(np.int8)
    )


def evaluar_regla_simple(
    nombre_regla,
    funcion_regla,
    X,
    y,
    subconjunto
):
    predicciones = (
        funcion_regla(X)
        .to_numpy()
    )

    tn, fp, fn, tp = confusion_matrix(
        y,
        predicciones,
        labels=[0, 1]
    ).ravel()

    return {
        "regla": nombre_regla,
        "subconjunto": subconjunto,
        "precision": precision_score(
            y,
            predicciones,
            zero_division=0
        ),
        "recall": recall_score(
            y,
            predicciones,
            zero_division=0
        ),
        "f1": f1_score(
            y,
            predicciones,
            zero_division=0
        ),
        "f2": fbeta_score(
            y,
            predicciones,
            beta=2,
            zero_division=0
        ),
        "tasa_alertas": (
            predicciones.mean()
        ),
        "verdaderos_negativos": int(tn),
        "falsos_positivos": int(fp),
        "falsos_negativos": int(fn),
        "verdaderos_positivos": int(tp)
    }


resultados_reglas_simples = []

for nombre_regla, funcion_regla in (
    reglas_simples.items()
):

    resultados_reglas_simples.append(
        evaluar_regla_simple(
            nombre_regla,
            funcion_regla,
            X_desarrollo,
            y_desarrollo,
            "development"
        )
    )

    resultados_reglas_simples.append(
        evaluar_regla_simple(
            nombre_regla,
            funcion_regla,
            X_holdout,
            y_holdout,
            "locked_internal_holdout"
        )
    )


resultados_reglas_simples = pd.DataFrame(
    resultados_reglas_simples
)

print(
    "\nRendimiento de reglas simples:"
)

display(
    resultados_reglas_simples
    .sort_values(
        [
            "subconjunto",
            "f2"
        ],
        ascending=[
            True,
            False
        ]
    )
    .style.format({
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "f2": "{:.4f}",
        "tasa_alertas": "{:.2%}"
    })
)


# ---------------------------------------------------------
# 5. Coincidencia entre importancias y AUC individual
# ---------------------------------------------------------

auditoria_variables = (
    comparacion_auc_individual
    .merge(
        importancias_modelo,
        on="variable",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        "importancia",
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Almacenamiento de evidencias
# ---------------------------------------------------------

RUTA_AUDITORIA_AUC_VARIABLES = (
    RUTA_RESULTADOS
    / "modelado_auditoria_auc_individual.csv"
)

RUTA_AUDITORIA_IMPORTANCIAS = (
    RUTA_RESULTADOS
    / "modelado_auditoria_importancias_xgboost.csv"
)

RUTA_AUDITORIA_REGLAS = (
    RUTA_RESULTADOS
    / "modelado_auditoria_reglas_simples.csv"
)

auditoria_variables.to_csv(
    RUTA_AUDITORIA_AUC_VARIABLES,
    index=False
)

importancias_modelo.to_csv(
    RUTA_AUDITORIA_IMPORTANCIAS,
    index=False
)

resultados_reglas_simples.to_csv(
    RUTA_AUDITORIA_REGLAS,
    index=False
)


# ---------------------------------------------------------
# 7. Validación del bloque
# ---------------------------------------------------------

auditoria_tecnica_valida = (
    len(variables_nombre_sospechoso) == 0
    and len(comparacion_auc_individual) == 176
    and comparacion_auc_individual[
        "variable"
    ].nunique() == 176
    and np.isfinite(
        comparacion_auc_individual[
            [
                "auc_orientado",
                "auc_orientado_holdout"
            ]
        ].to_numpy()
    ).all()
    and len(importancias_modelo) == 176
    and np.isfinite(
        importancias_modelo[
            "importancia"
        ].to_numpy()
    ).all()
    and RUTA_AUDITORIA_AUC_VARIABLES.exists()
    and RUTA_AUDITORIA_IMPORTANCIAS.exists()
    and RUTA_AUDITORIA_REGLAS.exists()
)

print(
    "\n¿La auditoría técnica se ha ejecutado correctamente?",
    auditoria_tecnica_valida
)

## Auditoría de separabilidad extrema y posibles fugas de información

La configuración congelada ha alcanzado un rendimiento extraordinariamente elevado tanto en las predicciones *out-of-fold* del conjunto de desarrollo como en el holdout interno. En este último se han obtenido un F2 de `0,9960`, un `recall` de `0,9983` y una `precision` de `0,9868`.

Estos resultados pueden deberse a una señal conductual legítimamente intensa. Sin embargo, antes de avanzar, se comprueba que ninguna variable predictora reproduzca directa o indirectamente la definición utilizada para construir la variable objetivo `churn`.

La auditoría examina los siguientes aspectos:

1. **Nombres de las variables.** Se comprueba que ninguna variable predictora contenga referencias explícitas a `churn`, a la variable objetivo o a las particiones de los datos.
2. **Capacidad predictiva individual.** Se calcula el `AUC-ROC` de cada variable por separado tanto en el conjunto de desarrollo como en el holdout interno. Una capacidad discriminativa casi perfecta, ya sea en sentido directo o inverso, requiere una revisión específica.
3. **Importancia asignada por XGBoost.** Se identifican las variables con mayor influencia en las predicciones del modelo entrenado.
4. **Reglas simples de inactividad.** Se evalúa si determinadas reglas basadas en la actividad reciente o en la recencia permiten reproducir por sí solas la variable objetivo.

Una asociación elevada no demuestra automáticamente la existencia de una fuga de información. La inactividad reciente puede constituir una señal válida si se observa antes del periodo utilizado para definir el abandono. En cambio, existiría una posible fuga temporal si la etiqueta se hubiera construido utilizando total o parcialmente los mismos días empleados para calcular las variables predictoras.

Este bloque proporciona una auditoría estadística, pero la ausencia de fugas temporales debe confirmarse revisando la definición exacta de la variable objetivo y la separación entre la ventana de observación y la ventana de predicción.

La auditoría no modifica la configuración congelada. El conjunto externo de prueba permanece completamente aislado. Si se confirma una fuga de información, será necesario corregir la preparación de los datos y repetir el proceso de modelado antes de realizar la evaluación final.

In [ ]:
# ---------------------------------------------------------
# 1. Revisión de nombres potencialmente sospechosos
# ---------------------------------------------------------

terminos_sospechosos = [
    "churn",
    "target",
    "label",
    "particion",
    "partition",
    "split"
]

variables_nombre_sospechoso = [
    variable
    for variable in columnas_predictoras
    if any(
        termino in variable.lower()
        for termino in terminos_sospechosos
    )
]

print(
    "Variables con nombres potencialmente sospechosos:",
    len(variables_nombre_sospechoso)
)

print(variables_nombre_sospechoso)


# ---------------------------------------------------------
# 2. AUC individual de cada variable
# ---------------------------------------------------------

def calcular_auc_individual(X, y):
    """
    Calcula la capacidad discriminativa individual
    de cada variable.

    El AUC orientado conserva el valor mayor entre
    AUC y 1 - AUC, de forma que mide la intensidad
    de la asociación con independencia del sentido.
    """
    resultados = []
    y_array = np.asarray(y)

    for variable in X.columns:

        valores = X[variable].to_numpy()

        if np.unique(valores).size < 2:
            auc_directo = 0.5
        else:
            auc_directo = roc_auc_score(
                y_array,
                valores
            )

        auc_orientado = max(
            auc_directo,
            1 - auc_directo
        )

        direccion = (
            "mayor_valor_mayor_riesgo"
            if auc_directo >= 0.5
            else "menor_valor_mayor_riesgo"
        )

        resultados.append({
            "variable": variable,
            "auc_directo": auc_directo,
            "auc_orientado": auc_orientado,
            "direccion": direccion
        })

    return (
        pd.DataFrame(resultados)
        .sort_values(
            "auc_orientado",
            ascending=False
        )
        .reset_index(drop=True)
    )


auc_individual_desarrollo = calcular_auc_individual(
    X_desarrollo,
    y_desarrollo
)

auc_individual_holdout = (
    calcular_auc_individual(
        X_holdout,
        y_holdout
    )
    .rename(columns={
        "auc_directo": "auc_directo_holdout",
        "auc_orientado": "auc_orientado_holdout",
        "direccion": "direccion_holdout"
    })
)

comparacion_auc_individual = (
    auc_individual_desarrollo
    .merge(
        auc_individual_holdout,
        on="variable",
        how="inner",
        validate="one_to_one"
    )
)

print(
    "\nVariables con AUC individual >= 0.95 "
    "en desarrollo:",
    (
        comparacion_auc_individual[
            "auc_orientado"
        ] >= 0.95
    ).sum()
)

print(
    "Variables con AUC individual >= 0.98 "
    "en desarrollo:",
    (
        comparacion_auc_individual[
            "auc_orientado"
        ] >= 0.98
    ).sum()
)

print(
    "\nVariables con mayor capacidad "
    "predictiva individual:"
)

display(
    comparacion_auc_individual[
        [
            "variable",
            "auc_orientado",
            "auc_orientado_holdout",
            "direccion",
            "direccion_holdout"
        ]
    ]
    .sort_values(
        "auc_orientado",
        ascending=False
    )
    .head(20)
    .style.format({
        "auc_orientado": "{:.4f}",
        "auc_orientado_holdout": "{:.4f}"
    })
)


# ---------------------------------------------------------
# 3. Importancia de las variables en XGBoost
# ---------------------------------------------------------

if not hasattr(
    modelo_holdout,
    "feature_importances_"
):
    raise ValueError(
        "El modelo entrenado no expone "
        "feature_importances_."
    )

importancias_modelo = pd.DataFrame({
    "variable": columnas_predictoras,
    "importancia": (
        modelo_holdout.feature_importances_
    )
})

importancias_modelo = (
    importancias_modelo
    .sort_values(
        "importancia",
        ascending=False
    )
    .reset_index(drop=True)
)

importancias_modelo[
    "importancia_acumulada"
] = (
    importancias_modelo[
        "importancia"
    ].cumsum()
)

print(
    "\nVariables más importantes para XGBoost:"
)

display(
    importancias_modelo
    .head(20)
    .style.format({
        "importancia": "{:.6f}",
        "importancia_acumulada": "{:.4f}"
    })
)


# ---------------------------------------------------------
# 4. Evaluación de reglas simples de inactividad
# ---------------------------------------------------------

reglas_simples = {}

if "agg_global__last7" in columnas_predictoras:
    reglas_simples[
        "actividad_global_ultimos_7_igual_0"
    ] = (
        lambda X:
        X["agg_global__last7"]
        .eq(0)
        .astype(np.int8)
    )

if "agg_global__last14" in columnas_predictoras:
    reglas_simples[
        "actividad_global_ultimos_14_igual_0"
    ] = (
        lambda X:
        X["agg_global__last14"]
        .eq(0)
        .astype(np.int8)
    )

if "agg_global__recency" in columnas_predictoras:
    reglas_simples[
        "recencia_global_mayor_igual_7"
    ] = (
        lambda X:
        X["agg_global__recency"]
        .ge(7)
        .astype(np.int8)
    )

    reglas_simples[
        "recencia_global_mayor_igual_14"
    ] = (
        lambda X:
        X["agg_global__recency"]
        .ge(14)
        .astype(np.int8)
    )

if "agg_global__total" in columnas_predictoras:
    reglas_simples[
        "actividad_global_total_igual_0"
    ] = (
        lambda X:
        X["agg_global__total"]
        .eq(0)
        .astype(np.int8)
    )


def evaluar_regla_simple(
    nombre_regla,
    funcion_regla,
    X,
    y,
    subconjunto
):
    predicciones = (
        funcion_regla(X)
        .to_numpy()
    )

    tn, fp, fn, tp = confusion_matrix(
        y,
        predicciones,
        labels=[0, 1]
    ).ravel()

    return {
        "regla": nombre_regla,
        "subconjunto": subconjunto,
        "precision": precision_score(
            y,
            predicciones,
            zero_division=0
        ),
        "recall": recall_score(
            y,
            predicciones,
            zero_division=0
        ),
        "f1": f1_score(
            y,
            predicciones,
            zero_division=0
        ),
        "f2": fbeta_score(
            y,
            predicciones,
            beta=2,
            zero_division=0
        ),
        "tasa_alertas": (
            predicciones.mean()
        ),
        "verdaderos_negativos": int(tn),
        "falsos_positivos": int(fp),
        "falsos_negativos": int(fn),
        "verdaderos_positivos": int(tp)
    }


resultados_reglas_simples = []

for nombre_regla, funcion_regla in (
    reglas_simples.items()
):

    resultados_reglas_simples.append(
        evaluar_regla_simple(
            nombre_regla,
            funcion_regla,
            X_desarrollo,
            y_desarrollo,
            "development"
        )
    )

    resultados_reglas_simples.append(
        evaluar_regla_simple(
            nombre_regla,
            funcion_regla,
            X_holdout,
            y_holdout,
            "locked_internal_holdout"
        )
    )

resultados_reglas_simples = pd.DataFrame(
    resultados_reglas_simples
)

print(
    "\nReglas simples evaluadas:",
    len(reglas_simples)
)

print(
    "\nRendimiento de las reglas simples:"
)

display(
    resultados_reglas_simples
    .sort_values(
        [
            "subconjunto",
            "f2"
        ],
        ascending=[
            True,
            False
        ]
    )
    .style.format({
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "f2": "{:.4f}",
        "tasa_alertas": "{:.2%}"
    })
)


# ---------------------------------------------------------
# 5. Integración del diagnóstico
# ---------------------------------------------------------

auditoria_variables = (
    comparacion_auc_individual
    .merge(
        importancias_modelo,
        on="variable",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        "importancia",
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Almacenamiento de evidencias
# ---------------------------------------------------------

RUTA_AUDITORIA_AUC_VARIABLES = (
    RUTA_RESULTADOS
    / "modelado_auditoria_auc_individual.csv"
)

RUTA_AUDITORIA_IMPORTANCIAS = (
    RUTA_RESULTADOS
    / "modelado_auditoria_importancias_xgboost.csv"
)

RUTA_AUDITORIA_REGLAS = (
    RUTA_RESULTADOS
    / "modelado_auditoria_reglas_simples.csv"
)

auditoria_variables.to_csv(
    RUTA_AUDITORIA_AUC_VARIABLES,
    index=False
)

importancias_modelo.to_csv(
    RUTA_AUDITORIA_IMPORTANCIAS,
    index=False
)

resultados_reglas_simples.to_csv(
    RUTA_AUDITORIA_REGLAS,
    index=False
)


# ---------------------------------------------------------
# 7. Validación del bloque
# ---------------------------------------------------------

auditoria_tecnica_valida = (
    len(variables_nombre_sospechoso) == 0
    and len(comparacion_auc_individual) == 176
    and comparacion_auc_individual[
        "variable"
    ].nunique() == 176
    and np.isfinite(
        comparacion_auc_individual[
            [
                "auc_orientado",
                "auc_orientado_holdout"
            ]
        ].to_numpy()
    ).all()
    and len(importancias_modelo) == 176
    and np.isfinite(
        importancias_modelo[
            "importancia"
        ].to_numpy()
    ).all()
    and len(resultados_reglas_simples)
        == len(reglas_simples) * 2
    and RUTA_AUDITORIA_AUC_VARIABLES.exists()
    and RUTA_AUDITORIA_IMPORTANCIAS.exists()
    and RUTA_AUDITORIA_REGLAS.exists()
)

print(
    "\n¿La auditoría técnica se ha ejecutado correctamente?",
    auditoria_tecnica_valida
)

## Conclusión de la auditoría de posibles fugas de información

La auditoría no identifica variables cuyos nombres revelen explícitamente la variable objetivo o la partición de los datos. Tampoco se observa que una regla trivial basada en la ausencia absoluta de actividad permita reproducir `churn`.

Las reglas que consideran la ausencia total de actividad durante los últimos 7 o 14 días presentan una `precision` elevada, pero un `recall` prácticamente nulo. Por tanto, solo identifican un número muy reducido de casos y no explican el rendimiento global del modelo.

La elevada capacidad predictiva procede principalmente de variables relacionadas con la evolución temporal. Las que presentan un mayor `AUC-ROC` individual representan la pendiente de la actividad global, la diferencia entre el último y el primer tercio, la actividad de los últimos 14 días, la variación logarítmica entre periodos y la actividad de los últimos 7 días.

La dirección de estas asociaciones es coherente en desarrollo y holdout: una menor actividad reciente y una evolución más negativa se relacionan con una mayor probabilidad de `churn`. Además, los valores de `AUC-ROC` son similares en ambos conjuntos, lo que reduce la probabilidad de que la separabilidad observada se deba únicamente al sobreajuste sobre los datos de desarrollo.

No obstante, seis variables presentan un `AUC-ROC` individual superior a `0,98` y el modelo completo alcanza resultados próximos a una clasificación perfecta. Este grado de separabilidad requiere una interpretación prudente.

La variable objetivo `churn` procede directamente del dataset original y no se ha reconstruido durante el proyecto. Sin embargo, la documentación disponible no permite confirmar de manera concluyente que el periodo utilizado para definirla sea posterior a los días empleados para construir las variables predictoras.

En consecuencia, la auditoría no muestra evidencias de una fuga explícita ni de una regla trivial que reproduzca la variable objetivo, pero tampoco permite descartar por completo una posible proximidad o superposición temporal entre la ventana de actividad y el periodo utilizado para definir `churn`.

El conjunto externo de prueba permanece bloqueado. Como siguiente paso, se comparan ventanas de 30, 60 y 90 días utilizando exclusivamente el conjunto de desarrollo. Este análisis permite comprobar si la señal predictiva aparece con antelación o si se concentra principalmente en los últimos días de la serie.

In [ ]:
# ---------------------------------------------------------
# Resumen estructurado de la auditoría
# ---------------------------------------------------------

variables_auc_095 = int(
    (
        comparacion_auc_individual[
            "auc_orientado"
        ] >= 0.95
    ).sum()
)

variables_auc_098 = int(
    (
        comparacion_auc_individual[
            "auc_orientado"
        ] >= 0.98
    ).sum()
)

cinco_variables_principales = (
    comparacion_auc_individual
    .sort_values(
        "auc_orientado",
        ascending=False
    )
    .head(5)[
        [
            "variable",
            "auc_orientado",
            "auc_orientado_holdout",
            "direccion",
            "direccion_holdout"
        ]
    ]
    .to_dict(
        orient="records"
    )
)

mejor_regla_holdout = (
    resultados_reglas_simples.loc[
        resultados_reglas_simples[
            "subconjunto"
        ] == "locked_internal_holdout"
    ]
    .sort_values(
        "f2",
        ascending=False
    )
    .iloc[0]
)

conclusion_auditoria = {
    "variables_con_nombre_sospechoso": int(
        len(variables_nombre_sospechoso)
    ),
    "variables_auc_individual_ge_095": (
        variables_auc_095
    ),
    "variables_auc_individual_ge_098": (
        variables_auc_098
    ),
    "cinco_variables_mayor_auc": (
        cinco_variables_principales
    ),
    "mejor_regla_simple_holdout": {
        "regla": str(
            mejor_regla_holdout["regla"]
        ),
        "precision": float(
            mejor_regla_holdout["precision"]
        ),
        "recall": float(
            mejor_regla_holdout["recall"]
        ),
        "f2": float(
            mejor_regla_holdout["f2"]
        ),
        "tasa_alertas": float(
            mejor_regla_holdout[
                "tasa_alertas"
            ]
        )
    },
    "fuga_explicita_detectada": False,
    "regla_trivial_explica_resultado": False,
    "separabilidad_extrema_detectada": True,
    "definicion_temporal_objetivo_documentada": False,
    "test_externo_autorizado": False,
    "siguiente_comprobacion": (
        "comparacion_ventanas_30_60_90"
    )
}


# ---------------------------------------------------------
# Almacenamiento de la conclusión
# ---------------------------------------------------------

RUTA_CONCLUSION_AUDITORIA = (
    RUTA_RESULTADOS
    / "modelado_conclusion_auditoria_fuga.json"
)

with open(
    RUTA_CONCLUSION_AUDITORIA,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        conclusion_auditoria,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Presentación y validación
# ---------------------------------------------------------

print(
    "Variables con AUC individual >= 0.95:",
    variables_auc_095
)

print(
    "Variables con AUC individual >= 0.98:",
    variables_auc_098
)

print(
    "Mejor regla simple en holdout:",
    mejor_regla_holdout["regla"]
)

print(
    "F2 de la mejor regla simple:",
    f"{mejor_regla_holdout['f2']:.4f}"
)

print(
    "¿Se ha detectado fuga explícita?",
    conclusion_auditoria[
        "fuga_explicita_detectada"
    ]
)

print(
    "¿Está autorizado utilizar el test externo?",
    conclusion_auditoria[
        "test_externo_autorizado"
    ]
)

print(
    "Siguiente comprobación:",
    conclusion_auditoria[
        "siguiente_comprobacion"
    ]
)

conclusion_auditoria_valida = (
    variables_auc_095 == 18
    and variables_auc_098 == 6
    and len(
        cinco_variables_principales
    ) == 5
    and conclusion_auditoria[
        "fuga_explicita_detectada"
    ] is False
    and conclusion_auditoria[
        "test_externo_autorizado"
    ] is False
    and RUTA_CONCLUSION_AUDITORIA.exists()
)

print(
    "¿La conclusión de la auditoría se ha guardado "
    "correctamente?",
    conclusion_auditoria_valida
)

## Protocolo de comparación temporal y regla de decisión

**Objetivo.** Se comparan las ventanas de 30, 60 y 90 días para determinar cuánta información histórica necesita el modelo y comprobar si la señal predictiva aparece con suficiente antelación.

**Definición previa del criterio.** Las tres ventanas forman parte del diseño temporal del proyecto. Para evitar elegir la ventana únicamente después de observar los resultados, se establecen previamente el procedimiento de comparación y la regla de decisión. Las tolerancias utilizadas para considerar equivalentes dos configuraciones responden a criterios operativos definidos para esta reconstrucción y quedan registradas explícitamente.

**Datos utilizados.** La comparación se realiza exclusivamente con los 44.203 clientes del conjunto de desarrollo y las mismas cinco divisiones estratificadas. Se evalúan los tres modelos finalistas identificados previamente: regresión logística con regularización L2, regresión logística balanceada y XGBoost. En todas las ventanas se aplican las mismas 176 definiciones de variables, calculadas sobre el periodo correspondiente.

El holdout interno ya se ha utilizado para evaluar la configuración provisional de 90 días. Por tanto, no interviene en la selección de la ventana temporal y sus resultados se consideran evidencia complementaria, no una validación independiente de esta comparación. El conjunto externo de prueba permanece bloqueado hasta que se seleccione y congele la configuración definitiva.

**Métricas.** La métrica principal es `average_precision`, especialmente adecuada cuando la clase positiva es minoritaria. Como métricas complementarias se utilizan `AUC-ROC`, `Brier score` y `log loss`. Tras seleccionar el umbral mediante predicciones *out-of-fold*, también se calculan `precision`, `recall`, F1, F2 y la proporción de clientes clasificados en la clase positiva.

**Regla de decisión.**

1. Para cada combinación de ventana y modelo, se generan probabilidades *out-of-fold* utilizando las mismas cinco divisiones.
2. El umbral de cada configuración se selecciona maximizando F2 exclusivamente dentro del conjunto de desarrollo.
3. Se identifica como referencia la configuración que obtiene el mayor `average_precision`.
4. Se consideran equivalentes a la configuración de referencia aquellas cuya pérdida no supera `0,005` en `average_precision`, `0,005` en `AUC-ROC`, `0,01` en F2 y `0,01` en `recall`.
5. Entre las configuraciones equivalentes, se selecciona la que utiliza la ventana más corta, ya que requiere menos días de observación.
6. Si varios modelos permanecen equivalentes dentro de la ventana seleccionada, se prioriza el más sencillo e interpretable en el siguiente orden: regresión logística con regularización L2, regresión logística balanceada y XGBoost.

Esta regla no garantiza la selección de la ventana más corta. Establece que una ventana más larga debe aportar una mejora suficientemente clara para justificar el uso de un periodo de observación mayor.

In [ ]:
# ---------------------------------------------------------
# Configuración congelada de la comparación temporal
# ---------------------------------------------------------

VENTANAS_COMPARACION = [
    30,
    60,
    90
]

MODELOS_COMPARACION_TEMPORAL = [
    "logistic_l2",
    "logistic_balanced",
    "xgboost"
]

METRICA_PRINCIPAL_TEMPORAL = (
    "average_precision"
)

CRITERIO_UMBRAL_TEMPORAL = (
    "max_f2_oof"
)

ORDEN_PREFERENCIA_MODELOS = [
    "logistic_l2",
    "logistic_balanced",
    "xgboost"
]

TOLERANCIAS_EQUIVALENCIA = {
    "average_precision": 0.005,
    "auc_roc": 0.005,
    "f2": 0.01,
    "recall": 0.01
}


# ---------------------------------------------------------
# Manifiesto metodológico
# ---------------------------------------------------------

protocolo_comparacion_temporal = {
    "status": (
        "frozen_before_temporal_comparison"
    ),
    "windows_days": (
        VENTANAS_COMPARACION
    ),
    "candidate_models": (
        MODELOS_COMPARACION_TEMPORAL
    ),
    "model_preference_if_equivalent": (
        ORDEN_PREFERENCIA_MODELOS
    ),
    "development_clients": int(
        len(X_desarrollo)
    ),
    "internal_holdout_clients": int(
        len(X_holdout)
    ),
    "external_test_clients": int(
        len(X_test_externo)
    ),
    "selected_features": int(
        len(columnas_predictoras)
    ),
    "cv_method": (
        "StratifiedKFold"
    ),
    "cv_folds": int(
        NUMERO_PLIEGUES
    ),
    "cv_random_state": int(
        SEMILLA_CV
    ),
    "primary_metric": (
        METRICA_PRINCIPAL_TEMPORAL
    ),
    "threshold_selection": (
        CRITERIO_UMBRAL_TEMPORAL
    ),
    "equivalence_tolerances": (
        TOLERANCIAS_EQUIVALENCIA
    ),
    "selection_data": (
        "development_only"
    ),
    "internal_holdout_status": (
        "already_consulted_for_provisional_90d_"
        "configuration_not_allowed_for_"
        "temporal_selection"
    ),
    "external_test_status": (
        "locked"
    ),
    "selection_rule": [
        (
            "Generate out-of-fold probabilities "
            "for every window-model combination."
        ),
        (
            "Select each threshold by maximizing "
            "F2 on development OOF predictions."
        ),
        (
            "Use average precision as the primary "
            "comparison metric."
        ),
        (
            "Retain configurations within the "
            "predefined equivalence tolerances."
        ),
        (
            "Prefer the shortest equivalent window."
        ),
        (
            "Within the same window, prefer the "
            "simpler model according to the "
            "predefined order."
        )
    ]
}


# ---------------------------------------------------------
# Almacenamiento del protocolo antes de la comparación
# ---------------------------------------------------------

RUTA_PROTOCOLO_TEMPORAL = (
    RUTA_RESULTADOS
    / "modelado_protocolo_comparacion_temporal.json"
)

with open(
    RUTA_PROTOCOLO_TEMPORAL,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        protocolo_comparacion_temporal,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Validación del protocolo
# ---------------------------------------------------------

protocolo_temporal_valido = (
    VENTANAS_COMPARACION
        == [30, 60, 90]
    and len(
        MODELOS_COMPARACION_TEMPORAL
    ) == 3
    and set(
        MODELOS_COMPARACION_TEMPORAL
    ).issubset(
        modelos_candidatos.keys()
    )
    and len(X_desarrollo) == 44_203
    and len(X_holdout) == 7_801
    and len(X_test_externo) == 13_001
    and len(columnas_predictoras) == 176
    and NUMERO_PLIEGUES == 5
    and METRICA_PRINCIPAL_TEMPORAL
        == "average_precision"
    and CRITERIO_UMBRAL_TEMPORAL
        == "max_f2_oof"
    and RUTA_PROTOCOLO_TEMPORAL.exists()
)


# ---------------------------------------------------------
# Presentación de la configuración congelada
# ---------------------------------------------------------

print(
    "Ventanas que se compararán:",
    VENTANAS_COMPARACION
)

print(
    "Modelos que se compararán:",
    MODELOS_COMPARACION_TEMPORAL
)

print(
    "Métrica principal:",
    METRICA_PRINCIPAL_TEMPORAL
)

print(
    "Criterio de selección del umbral:",
    CRITERIO_UMBRAL_TEMPORAL
)

print(
    "Datos autorizados para seleccionar:",
    protocolo_comparacion_temporal[
        "selection_data"
    ]
)

print(
    "Estado del holdout interno:",
    protocolo_comparacion_temporal[
        "internal_holdout_status"
    ]
)

print(
    "Estado del test externo:",
    protocolo_comparacion_temporal[
        "external_test_status"
    ]
)

print(
    "Tolerancias de equivalencia:",
    TOLERANCIAS_EQUIVALENCIA
)

print(
    "¿El protocolo temporal se ha congelado "
    "correctamente?",
    protocolo_temporal_valido
)

## Construcción de los datasets temporales de 30 y 60 días

**Objetivo.** Se generan dos versiones adicionales del dataset utilizando únicamente los primeros 30 y 60 días de actividad, que posteriormente se comparan con la versión construida sobre la ventana completa de 90 días.

**Criterio metodológico.** Las tres ventanas utilizan las mismas 176 definiciones de variables y el mismo universo de clientes. De este modo, la principal diferencia entre los escenarios es la cantidad de información histórica disponible. La matriz de 90 días ya ha sido validada y no se vuelve a calcular.

**Datos utilizados.** Las variables predictoras se construyen a partir de los doce archivos diarios originales. La variable objetivo `churn` no interviene en estos cálculos y se incorpora únicamente después de completar la ingeniería de variables.

**Resultado esperado.** Se obtienen los archivos `dataset_modelado_30d.pkl` y `dataset_modelado_60d.pkl`. Cada uno contiene 65.005 clientes y 179 columnas: 176 variables predictoras y las columnas auxiliares `customer_id`, `churn` y `particion`.

In [ ]:
import gc


# ---------------------------------------------------------
# Configuración de las ventanas y archivos de actividad
# ---------------------------------------------------------

VENTANAS_NUEVAS = [30, 60]
MAXIMO_DIAS_LECTURA = max(VENTANAS_NUEVAS)

RUTA_USAGE_DATA = (
    RUTA_PROYECTO
    / "datos"
    / "Dataset"
    / "usage_data"
)

archivos_actividad_temporal = sorted(
    RUTA_USAGE_DATA.glob("*.csv")
)

if len(archivos_actividad_temporal) != 12:
    raise ValueError(
        "Se esperaban 12 archivos de actividad y se "
        f"encontraron {len(archivos_actividad_temporal)}."
    )

columnas_dias_60 = [
    f"Day_{dia}"
    for dia in range(
        1,
        MAXIMO_DIAS_LECTURA + 1
    )
]

NOMBRES_RESUMEN_TEMPORAL = [
    "total",
    "std",
    "max",
    "active_days",
    "first_third",
    "delta_last_first",
    "log_ratio_last_first",
    "slope",
    "recency",
    "last7",
    "last14"
]


# ---------------------------------------------------------
# Funciones de ingeniería temporal
# ---------------------------------------------------------

def calcular_pendiente_temporal(valores):
    numero_dias = valores.shape[1]

    eje_temporal = np.arange(
        1,
        numero_dias + 1,
        dtype=np.float64
    )

    eje_centrado = (
        eje_temporal
        - eje_temporal.mean()
    )

    denominador = np.sum(
        eje_centrado ** 2
    )

    valores_float = valores.astype(
        np.float64,
        copy=False
    )

    valores_centrados = (
        valores_float
        - valores_float.mean(
            axis=1,
            keepdims=True
        )
    )

    return (
        valores_centrados
        @ eje_centrado
    ) / denominador


def calcular_recencia_temporal(valores):
    actividad = valores > 0
    numero_dias = valores.shape[1]

    posiciones = np.where(
        actividad,
        np.arange(
            1,
            numero_dias + 1
        ),
        0
    )

    ultima_posicion = posiciones.max(
        axis=1
    )

    return np.where(
        ultima_posicion == 0,
        numero_dias,
        numero_dias - ultima_posicion
    )


def resumir_matriz_temporal(valores):
    valores = valores.astype(
        np.float64,
        copy=False
    )

    numero_dias = valores.shape[1]

    fin_primer_tercio = max(
        1,
        numero_dias // 3
    )

    inicio_ultimo_tercio = max(
        2,
        2 * numero_dias // 3
    )

    total_primer_tercio = (
        valores[
            :,
            :fin_primer_tercio
        ]
        .sum(axis=1)
    )

    total_ultimo_tercio = (
        valores[
            :,
            inicio_ultimo_tercio:
        ]
        .sum(axis=1)
    )

    resumen = np.column_stack([
        valores.sum(axis=1),
        valores.std(axis=1),
        valores.max(axis=1),
        (valores > 0).sum(axis=1),
        total_primer_tercio,
        (
            total_ultimo_tercio
            - total_primer_tercio
        ),
        (
            np.log1p(total_ultimo_tercio)
            - np.log1p(total_primer_tercio)
        ),
        calcular_pendiente_temporal(
            valores
        ),
        calcular_recencia_temporal(
            valores
        ),
        valores[
            :,
            -min(7, numero_dias):
        ].sum(axis=1),
        valores[
            :,
            -min(14, numero_dias):
        ].sum(axis=1)
    ])

    return resumen.astype(
        np.float32
    )


def obtener_categoria_temporal(ruta_archivo):
    nombre = ruta_archivo.stem

    if nombre.startswith("usage_app_"):
        return "apps"

    if nombre.startswith("usage_voice_"):
        return "voice"

    if nombre.startswith("usage_pack_"):
        return "packages"

    raise ValueError(
        f"No se reconoce la categoría de {nombre}."
    )


# ---------------------------------------------------------
# Comprobación de la clasificación de los perfiles
# ---------------------------------------------------------

conteo_categorias = {
    "apps": 0,
    "voice": 0,
    "packages": 0
}

for ruta_archivo in archivos_actividad_temporal:
    categoria = obtener_categoria_temporal(
        ruta_archivo
    )

    conteo_categorias[categoria] += 1

print(
    "Distribución de perfiles:",
    conteo_categorias
)

if conteo_categorias != {
    "apps": 6,
    "voice": 4,
    "packages": 2
}:
    raise ValueError(
        "La distribución de perfiles no coincide "
        "con la estructura esperada."
    )


# ---------------------------------------------------------
# Universo maestro de clientes
# ---------------------------------------------------------

orden_clientes_temporal = pd.Index(
    dataset_modelado["customer_id"],
    name="customer_id"
)

numero_clientes_temporal = len(
    orden_clientes_temporal
)

posicion_cliente = pd.Series(
    np.arange(
        numero_clientes_temporal,
        dtype=np.int64
    ),
    index=orden_clientes_temporal
)


# ---------------------------------------------------------
# Estructuras de acumulación
# ---------------------------------------------------------

matrices_perfiles = {
    ventana: []
    for ventana in VENTANAS_NUEVAS
}

acumulados_categorias = {
    ventana: {
        categoria: np.zeros(
            (
                numero_clientes_temporal,
                ventana
            ),
            dtype=np.float32
        )
        for categoria in [
            "apps",
            "voice",
            "packages"
        ]
    }
    for ventana in VENTANAS_NUEVAS
}

acumulados_globales = {
    ventana: np.zeros(
        (
            numero_clientes_temporal,
            ventana
        ),
        dtype=np.float32
    )
    for ventana in VENTANAS_NUEVAS
}


# ---------------------------------------------------------
# Lectura única de los doce archivos
# ---------------------------------------------------------

for numero_archivo, ruta_archivo in enumerate(
    archivos_actividad_temporal,
    start=1
):
    nombre_perfil = ruta_archivo.stem

    categoria = obtener_categoria_temporal(
        ruta_archivo
    )

    resumen_perfil = {
        ventana: np.empty(
            (
                numero_clientes_temporal,
                len(NOMBRES_RESUMEN_TEMPORAL)
            ),
            dtype=np.float32
        )
        for ventana in VENTANAS_NUEVAS
    }

    clientes_vistos = np.zeros(
        numero_clientes_temporal,
        dtype=bool
    )

    filas_procesadas = 0

    lector_bloques = pd.read_csv(
        ruta_archivo,
        usecols=[
            "customer_id",
            *columnas_dias_60
        ],
        chunksize=10_000
    )

    for bloque in lector_bloques:

        posiciones = (
            posicion_cliente
            .reindex(
                bloque["customer_id"]
            )
        )

        if posiciones.isna().any():
            raise ValueError(
                f"{nombre_perfil}: existen clientes "
                "ajenos al universo maestro."
            )

        posiciones = posiciones.to_numpy(
            dtype=np.int64
        )

        if clientes_vistos[
            posiciones
        ].any():
            raise ValueError(
                f"{nombre_perfil}: existen clientes "
                "duplicados."
            )

        valores_60 = (
            bloque[columnas_dias_60]
            .to_numpy(dtype=np.float32)
        )

        if not np.isfinite(
            valores_60
        ).all():
            raise ValueError(
                f"{nombre_perfil}: existen valores "
                "no finitos."
            )

        for ventana in VENTANAS_NUEVAS:

            valores_ventana = valores_60[
                :,
                :ventana
            ]

            resumen_perfil[
                ventana
            ][posiciones] = (
                resumir_matriz_temporal(
                    valores_ventana
                )
            )

            acumulados_categorias[
                ventana
            ][categoria][posiciones] += (
                valores_ventana
            )

            acumulados_globales[
                ventana
            ][posiciones] += (
                valores_ventana
            )

        clientes_vistos[posiciones] = True
        filas_procesadas += len(bloque)

    if not clientes_vistos.all():
        raise ValueError(
            f"{nombre_perfil}: faltan "
            f"{(~clientes_vistos).sum():,} clientes."
        )

    for ventana in VENTANAS_NUEVAS:

        columnas_perfil = [
            f"{nombre_perfil}__{indicador}"
            for indicador
            in NOMBRES_RESUMEN_TEMPORAL
        ]

        matrices_perfiles[
            ventana
        ].append(
            pd.DataFrame(
                resumen_perfil[ventana],
                index=orden_clientes_temporal,
                columns=columnas_perfil
            )
        )

    print(
        f"Archivo {numero_archivo:02d}/12 procesado | "
        f"{nombre_perfil} | "
        f"{filas_procesadas:,} clientes"
    )

    del resumen_perfil
    gc.collect()


# ---------------------------------------------------------
# Integración de los perfiles y agregados
# ---------------------------------------------------------

datasets_modelado_temporales = {}
matrices_variables_temporales = {}

for ventana in VENTANAS_NUEVAS:

    matriz_perfiles_ventana = pd.concat(
        matrices_perfiles[ventana],
        axis=1
    )

    bloques_agregados = []

    for categoria in [
        "global",
        "apps",
        "voice",
        "packages"
    ]:

        if categoria == "global":
            valores_agregados = (
                acumulados_globales[
                    ventana
                ]
            )
        else:
            valores_agregados = (
                acumulados_categorias[
                    ventana
                ][categoria]
            )

        columnas_agregado = [
            f"agg_{categoria}__{indicador}"
            for indicador
            in NOMBRES_RESUMEN_TEMPORAL
        ]

        bloques_agregados.append(
            pd.DataFrame(
                resumir_matriz_temporal(
                    valores_agregados
                ),
                index=orden_clientes_temporal,
                columns=columnas_agregado
            )
        )

    matriz_variables = pd.concat(
        [
            matriz_perfiles_ventana,
            *bloques_agregados
        ],
        axis=1
    )

    columnas_faltantes = sorted(
        set(columnas_predictoras)
        - set(matriz_variables.columns)
    )

    columnas_sobrantes = sorted(
        set(matriz_variables.columns)
        - set(columnas_predictoras)
    )

    if columnas_faltantes:
        raise ValueError(
            f"Ventana {ventana}: faltan variables: "
            f"{columnas_faltantes[:10]}"
        )

    if columnas_sobrantes:
        raise ValueError(
            f"Ventana {ventana}: sobran variables: "
            f"{columnas_sobrantes[:10]}"
        )

    matriz_variables = (
        matriz_variables[
            columnas_predictoras
        ]
        .reset_index()
    )

    dataset_ventana = (
        dataset_modelado[
            [
                "customer_id",
                "churn",
                "particion"
            ]
        ]
        .merge(
            matriz_variables,
            on="customer_id",
            how="inner",
            validate="one_to_one"
        )
    )

    matrices_variables_temporales[
        ventana
    ] = matriz_variables

    datasets_modelado_temporales[
        ventana
    ] = dataset_ventana

    print(
        f"\nVentana de {ventana} días:"
    )

    print(
        "Matriz de variables:",
        matriz_variables.shape
    )

    print(
        "Dataset de modelado:",
        dataset_ventana.shape
    )

    print(
        "Valores nulos:",
        dataset_ventana
        .isna()
        .sum()
        .sum()
    )


# ---------------------------------------------------------
# Almacenamiento de los datasets
# ---------------------------------------------------------

RUTA_DATASET_MODELADO_30D = (
    RUTA_RESULTADOS
    / "dataset_modelado_30d.pkl"
)

RUTA_DATASET_MODELADO_60D = (
    RUTA_RESULTADOS
    / "dataset_modelado_60d.pkl"
)

datasets_modelado_temporales[
    30
].to_pickle(
    RUTA_DATASET_MODELADO_30D
)

datasets_modelado_temporales[
    60
].to_pickle(
    RUTA_DATASET_MODELADO_60D
)


# ---------------------------------------------------------
# Validación final
# ---------------------------------------------------------

validacion_matrices_temporales = {
    "dataset_30_dimensiones": (
        datasets_modelado_temporales[
            30
        ].shape == (65_005, 179)
    ),
    "dataset_60_dimensiones": (
        datasets_modelado_temporales[
            60
        ].shape == (65_005, 179)
    ),
    "esquema_30_correcto": (
        matrices_variables_temporales[
            30
        ].columns.tolist()
        == [
            "customer_id",
            *columnas_predictoras
        ]
    ),
    "esquema_60_correcto": (
        matrices_variables_temporales[
            60
        ].columns.tolist()
        == [
            "customer_id",
            *columnas_predictoras
        ]
    ),
    "sin_nulos_30": (
        datasets_modelado_temporales[
            30
        ].isna().sum().sum() == 0
    ),
    "sin_nulos_60": (
        datasets_modelado_temporales[
            60
        ].isna().sum().sum() == 0
    ),
    "archivo_30_guardado": (
        RUTA_DATASET_MODELADO_30D.exists()
    ),
    "archivo_60_guardado": (
        RUTA_DATASET_MODELADO_60D.exists()
    )
}

display(
    pd.DataFrame({
        "comprobacion": (
            validacion_matrices_temporales.keys()
        ),
        "resultado": (
            validacion_matrices_temporales.values()
        )
    })
)

print(
    "Dataset de 30 días:",
    datasets_modelado_temporales[
        30
    ].shape
)

print(
    "Dataset de 60 días:",
    datasets_modelado_temporales[
        60
    ].shape
)

print(
    "¿Las matrices temporales se han construido "
    "correctamente?",
    all(
        validacion_matrices_temporales.values()
    )
)

## Comparación temporal mediante predicciones fuera de muestra

**Objetivo.** Se comparan las ventanas de 30, 60 y 90 días manteniendo constantes las definiciones de las 176 variables predictoras, los tres modelos finalistas y las cinco divisiones de validación cruzada.

**Criterio de evaluación.** Para cada combinación de ventana y modelo se generan probabilidades *out-of-fold*. La métrica principal es `average_precision` y el umbral de clasificación se selecciona maximizando F2. La equivalencia entre configuraciones se determina mediante las tolerancias definidas previamente.

**Datos utilizados.** La comparación utiliza exclusivamente los 44.203 clientes del conjunto de desarrollo. El holdout interno y el conjunto externo de prueba no intervienen en esta decisión.

**Eficiencia del procesamiento.** Se reutilizan las probabilidades de la ventana de 90 días calculadas anteriormente. Los resultados correspondientes a las combinaciones de 30 y 60 días se almacenan individualmente como puntos de control, evitando repetir los entrenamientos si la ejecución se interrumpe o el bloque debe ejecutarse de nuevo.

**Interpretación.** Entre las configuraciones consideradas equivalentes a la que obtiene el mayor `average_precision`, se prioriza la ventana más corta. Si varios modelos resultan equivalentes dentro de la ventana seleccionada, se elige el más sencillo de acuerdo con el orden establecido previamente.

In [ ]:
import gc
from time import perf_counter

from sklearn.model_selection import cross_val_predict


# ---------------------------------------------------------
# Carga y alineación de los tres datasets temporales
# ---------------------------------------------------------

datasets_por_ventana = {
    30: pd.read_pickle(
        RUTA_DATASET_MODELADO_30D
    ),
    60: pd.read_pickle(
        RUTA_DATASET_MODELADO_60D
    ),
    90: dataset_modelado
}


def preparar_desarrollo_temporal(
    dataset_ventana,
    ventana
):
    """
    Alinea una ventana con los clientes y el orden
    exactos del conjunto de desarrollo.
    """
    base_desarrollo = datos_desarrollo[
        [
            "customer_id",
            "churn"
        ]
    ].copy()

    datos_ventana = dataset_ventana[
        [
            "customer_id",
            "churn",
            *columnas_predictoras
        ]
    ].copy()

    datos_alineados = base_desarrollo.merge(
        datos_ventana,
        on="customer_id",
        how="left",
        validate="one_to_one",
        suffixes=(
            "_referencia",
            "_ventana"
        )
    )

    if datos_alineados[
        columnas_predictoras
    ].isna().any().any():
        raise ValueError(
            f"Ventana {ventana}: faltan variables "
            "después de alinear los clientes."
        )

    if not (
        datos_alineados["churn_referencia"]
        == datos_alineados["churn_ventana"]
    ).all():
        raise ValueError(
            f"Ventana {ventana}: la etiqueta churn "
            "no coincide con la referencia."
        )

    X_ventana = datos_alineados[
        columnas_predictoras
    ].copy()

    y_ventana = datos_alineados[
        "churn_referencia"
    ].copy()

    return X_ventana, y_ventana


matrices_desarrollo_temporales = {}

for ventana in VENTANAS_COMPARACION:

    X_ventana, y_ventana = (
        preparar_desarrollo_temporal(
            datasets_por_ventana[ventana],
            ventana
        )
    )

    matrices_desarrollo_temporales[
        ventana
    ] = {
        "X": X_ventana,
        "y": y_ventana
    }

    print(
        f"Ventana {ventana:02d} días | "
        f"X: {X_ventana.shape} | "
        f"y: {y_ventana.shape}"
    )


# ---------------------------------------------------------
# Validación del orden de los clientes y etiquetas
# ---------------------------------------------------------

for ventana in VENTANAS_COMPARACION:

    if (
        matrices_desarrollo_temporales[
            ventana
        ]["X"].shape
        != (44_203, 176)
    ):
        raise ValueError(
            f"Ventana {ventana}: dimensión incorrecta."
        )

    if not (
        matrices_desarrollo_temporales[
            ventana
        ]["y"].to_numpy()
        == y_desarrollo.to_numpy()
    ).all():
        raise ValueError(
            f"Ventana {ventana}: el orden de churn "
            "no coincide."
        )


# ---------------------------------------------------------
# Carpeta de puntos de control
# ---------------------------------------------------------

RUTA_CHECKPOINTS_TEMPORALES = (
    RUTA_RESULTADOS
    / "checkpoints_oof_temporal"
)

RUTA_CHECKPOINTS_TEMPORALES.mkdir(
    parents=True,
    exist_ok=True
)


def ruta_checkpoint_temporal(
    ventana,
    modelo
):
    return (
        RUTA_CHECKPOINTS_TEMPORALES
        / f"oof_{ventana}d_{modelo}.pkl"
    )


# ---------------------------------------------------------
# Tabla base de probabilidades temporales
# ---------------------------------------------------------

predicciones_oof_temporales = pd.DataFrame({
    "customer_id": (
        datos_desarrollo[
            "customer_id"
        ].to_numpy()
    ),
    "churn_real": (
        y_desarrollo.to_numpy()
    )
})


# ---------------------------------------------------------
# Función de evaluación y selección de umbral
# ---------------------------------------------------------

def evaluar_configuracion_temporal(
    ventana,
    modelo,
    probabilidades,
    tiempo_segundos,
    origen
):
    """
    Calcula métricas probabilísticas y selecciona el
    umbral que maximiza F2. En caso de empate, conserva
    el umbral más alto.
    """
    y_real = y_desarrollo.to_numpy()

    auc_roc = roc_auc_score(
        y_real,
        probabilidades
    )

    average_precision = (
        average_precision_score(
            y_real,
            probabilidades
        )
    )

    brier = float(
        np.mean(
            (
                probabilidades
                - y_real
            ) ** 2
        )
    )

    perdida_logaritmica = log_loss(
        y_real,
        probabilidades,
        labels=[0, 1]
    )

    resultados_umbral = []

    for umbral in np.round(
        np.arange(
            0.05,
            0.951,
            0.01
        ),
        2
    ):
        predicciones = (
            probabilidades >= umbral
        ).astype(np.int8)

        tn, fp, fn, tp = confusion_matrix(
            y_real,
            predicciones,
            labels=[0, 1]
        ).ravel()

        especificidad = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0.0
        )

        resultados_umbral.append({
            "umbral": float(umbral),
            "precision": precision_score(
                y_real,
                predicciones,
                zero_division=0
            ),
            "recall": recall_score(
                y_real,
                predicciones,
                zero_division=0
            ),
            "especificidad": especificidad,
            "f1": f1_score(
                y_real,
                predicciones,
                zero_division=0
            ),
            "f2": fbeta_score(
                y_real,
                predicciones,
                beta=2,
                zero_division=0
            ),
            "balanced_accuracy": (
                balanced_accuracy_score(
                    y_real,
                    predicciones
                )
            ),
            "tasa_alertas": float(
                predicciones.mean()
            ),
            "verdaderos_negativos": int(tn),
            "falsos_positivos": int(fp),
            "falsos_negativos": int(fn),
            "verdaderos_positivos": int(tp)
        })

    tabla_umbral = pd.DataFrame(
        resultados_umbral
    )

    mejor_umbral = (
        tabla_umbral
        .sort_values(
            [
                "f2",
                "umbral"
            ],
            ascending=[
                False,
                False
            ]
        )
        .iloc[0]
    )

    return {
        "ventana_dias": int(ventana),
        "modelo": modelo,
        "average_precision": float(
            average_precision
        ),
        "auc_roc": float(
            auc_roc
        ),
        "brier": brier,
        "log_loss": float(
            perdida_logaritmica
        ),
        "umbral_f2": float(
            mejor_umbral["umbral"]
        ),
        "precision": float(
            mejor_umbral["precision"]
        ),
        "recall": float(
            mejor_umbral["recall"]
        ),
        "especificidad": float(
            mejor_umbral[
                "especificidad"
            ]
        ),
        "f1": float(
            mejor_umbral["f1"]
        ),
        "f2": float(
            mejor_umbral["f2"]
        ),
        "balanced_accuracy": float(
            mejor_umbral[
                "balanced_accuracy"
            ]
        ),
        "tasa_alertas": float(
            mejor_umbral[
                "tasa_alertas"
            ]
        ),
        "verdaderos_negativos": int(
            mejor_umbral[
                "verdaderos_negativos"
            ]
        ),
        "falsos_positivos": int(
            mejor_umbral[
                "falsos_positivos"
            ]
        ),
        "falsos_negativos": int(
            mejor_umbral[
                "falsos_negativos"
            ]
        ),
        "verdaderos_positivos": int(
            mejor_umbral[
                "verdaderos_positivos"
            ]
        ),
        "tiempo_segundos": float(
            tiempo_segundos
        ),
        "origen_probabilidades": origen
    }


# ---------------------------------------------------------
# Generación o recuperación de probabilidades OOF
# ---------------------------------------------------------

resultados_comparacion_temporal = []

for ventana in VENTANAS_COMPARACION:

    X_ventana = (
        matrices_desarrollo_temporales[
            ventana
        ]["X"]
    )

    y_ventana = (
        matrices_desarrollo_temporales[
            ventana
        ]["y"]
    )

    for modelo_nombre in (
        MODELOS_COMPARACION_TEMPORAL
    ):

        print("\n" + "=" * 72)
        print(
            f"VENTANA: {ventana} DÍAS | "
            f"MODELO: {modelo_nombre}"
        )
        print("=" * 72)

        ruta_checkpoint = (
            ruta_checkpoint_temporal(
                ventana,
                modelo_nombre
            )
        )

        columna_probabilidad = (
            f"probabilidad__"
            f"{ventana}d__"
            f"{modelo_nombre}"
        )

        # Reutilización de las probabilidades de 90 días.
        if ventana == 90:

            columna_previa = (
                f"probabilidad__"
                f"{modelo_nombre}"
            )

            if not (
                predicciones_oof_finalistas[
                    "customer_id"
                ].to_numpy()
                == datos_desarrollo[
                    "customer_id"
                ].to_numpy()
            ).all():
                raise ValueError(
                    "El orden de las probabilidades "
                    "OOF de 90 días no coincide."
                )

            probabilidades = (
                predicciones_oof_finalistas[
                    columna_previa
                ]
                .to_numpy()
            )

            tiempo_segundos = float(
                resumen_oof_finalistas.loc[
                    resumen_oof_finalistas[
                        "modelo"
                    ] == modelo_nombre,
                    "tiempo_total_segundos"
                ].iloc[0]
            )

            origen = (
                "reutilizado_oof_90d"
            )

            if not ruta_checkpoint.exists():

                pd.DataFrame({
                    "customer_id": (
                        datos_desarrollo[
                            "customer_id"
                        ].to_numpy()
                    ),
                    "churn_real": (
                        y_desarrollo.to_numpy()
                    ),
                    "probabilidad": (
                        probabilidades
                    )
                }).to_pickle(
                    ruta_checkpoint
                )

        # Recuperación de un punto de control anterior.
        elif ruta_checkpoint.exists():

            checkpoint = pd.read_pickle(
                ruta_checkpoint
            )

            if checkpoint.shape != (
                44_203,
                3
            ):
                raise ValueError(
                    f"Checkpoint inválido: "
                    f"{ruta_checkpoint.name}"
                )

            if not (
                checkpoint[
                    "customer_id"
                ].to_numpy()
                == datos_desarrollo[
                    "customer_id"
                ].to_numpy()
            ).all():
                raise ValueError(
                    "Los clientes del checkpoint "
                    "no coinciden."
                )

            probabilidades = checkpoint[
                "probabilidad"
            ].to_numpy()

            tiempo_segundos = 0.0
            origen = "checkpoint"

        # Entrenamiento nuevo para 30 o 60 días.
        else:

            modelo = clone(
                modelos_candidatos[
                    modelo_nombre
                ]
            )

            inicio = perf_counter()

            probabilidades = cross_val_predict(
                estimator=modelo,
                X=X_ventana,
                y=y_ventana,
                cv=validacion_cruzada,
                method="predict_proba",
                n_jobs=1
            )[:, 1]

            tiempo_segundos = (
                perf_counter()
                - inicio
            )

            origen = "calculado"

            pd.DataFrame({
                "customer_id": (
                    datos_desarrollo[
                        "customer_id"
                    ].to_numpy()
                ),
                "churn_real": (
                    y_desarrollo.to_numpy()
                ),
                "probabilidad": (
                    probabilidades
                )
            }).to_pickle(
                ruta_checkpoint
            )

        if not np.isfinite(
            probabilidades
        ).all():
            raise ValueError(
                f"{ventana}d {modelo_nombre}: "
                "probabilidades no finitas."
            )

        if not (
            (probabilidades >= 0)
            & (probabilidades <= 1)
        ).all():
            raise ValueError(
                f"{ventana}d {modelo_nombre}: "
                "probabilidades fuera de [0, 1]."
            )

        predicciones_oof_temporales[
            columna_probabilidad
        ] = probabilidades

        resultado = (
            evaluar_configuracion_temporal(
                ventana=ventana,
                modelo=modelo_nombre,
                probabilidades=probabilidades,
                tiempo_segundos=(
                    tiempo_segundos
                ),
                origen=origen
            )
        )

        resultados_comparacion_temporal.append(
            resultado
        )

        print(
            "Average precision:",
            f"{resultado['average_precision']:.4f}"
        )

        print(
            "AUC-ROC:",
            f"{resultado['auc_roc']:.4f}"
        )

        print(
            "Umbral F2:",
            f"{resultado['umbral_f2']:.2f}"
        )

        print(
            "F2:",
            f"{resultado['f2']:.4f}"
        )

        print(
            "Recall:",
            f"{resultado['recall']:.4f}"
        )

        print(
            "Origen:",
            origen
        )

        gc.collect()


# ---------------------------------------------------------
# Tabla comparativa de las nueve configuraciones
# ---------------------------------------------------------

comparacion_temporal = pd.DataFrame(
    resultados_comparacion_temporal
)

comparacion_temporal = (
    comparacion_temporal
    .sort_values(
        [
            "average_precision",
            "auc_roc"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Aplicación de la regla de equivalencia congelada
# ---------------------------------------------------------

configuracion_referencia = (
    comparacion_temporal.iloc[0]
)

comparacion_temporal[
    "diferencia_average_precision"
] = (
    configuracion_referencia[
        "average_precision"
    ]
    - comparacion_temporal[
        "average_precision"
    ]
)

comparacion_temporal[
    "diferencia_auc_roc"
] = (
    configuracion_referencia[
        "auc_roc"
    ]
    - comparacion_temporal[
        "auc_roc"
    ]
)

comparacion_temporal[
    "diferencia_f2"
] = (
    configuracion_referencia["f2"]
    - comparacion_temporal["f2"]
)

comparacion_temporal[
    "diferencia_recall"
] = (
    configuracion_referencia["recall"]
    - comparacion_temporal["recall"]
)

comparacion_temporal[
    "equivalente_a_referencia"
] = (
    (
        comparacion_temporal[
            "diferencia_average_precision"
        ]
        <= TOLERANCIAS_EQUIVALENCIA[
            "average_precision"
        ]
    )
    & (
        comparacion_temporal[
            "diferencia_auc_roc"
        ]
        <= TOLERANCIAS_EQUIVALENCIA[
            "auc_roc"
        ]
    )
    & (
        comparacion_temporal[
            "diferencia_f2"
        ]
        <= TOLERANCIAS_EQUIVALENCIA[
            "f2"
        ]
    )
    & (
        comparacion_temporal[
            "diferencia_recall"
        ]
        <= TOLERANCIAS_EQUIVALENCIA[
            "recall"
        ]
    )
)


# ---------------------------------------------------------
# Preferencia por ventana corta y modelo sencillo
# ---------------------------------------------------------

orden_modelos = {
    modelo: posicion
    for posicion, modelo in enumerate(
        ORDEN_PREFERENCIA_MODELOS,
        start=1
    )
}

comparacion_temporal[
    "orden_preferencia_modelo"
] = (
    comparacion_temporal["modelo"]
    .map(orden_modelos)
)

configuraciones_equivalentes = (
    comparacion_temporal.loc[
        comparacion_temporal[
            "equivalente_a_referencia"
        ]
    ]
    .sort_values(
        [
            "ventana_dias",
            "orden_preferencia_modelo",
            "average_precision"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

if configuraciones_equivalentes.empty:
    raise ValueError(
        "Ninguna configuración cumple la regla "
        "de equivalencia congelada."
    )

configuracion_temporal_propuesta = (
    configuraciones_equivalentes.iloc[0]
)


# ---------------------------------------------------------
# Presentación de resultados
# ---------------------------------------------------------

columnas_presentacion_temporal = [
    "ventana_dias",
    "modelo",
    "average_precision",
    "auc_roc",
    "brier",
    "log_loss",
    "umbral_f2",
    "precision",
    "recall",
    "f1",
    "f2",
    "balanced_accuracy",
    "tasa_alertas",
    "equivalente_a_referencia",
    "origen_probabilidades"
]

display(
    comparacion_temporal[
        columnas_presentacion_temporal
    ].style.format({
        "average_precision": "{:.4f}",
        "auc_roc": "{:.4f}",
        "brier": "{:.4f}",
        "log_loss": "{:.4f}",
        "umbral_f2": "{:.2f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "f2": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "tasa_alertas": "{:.2%}"
    })
)

print(
    "\nConfiguraciones equivalentes:",
    len(configuraciones_equivalentes)
)

print(
    "\nConfiguración de referencia por "
    "average precision:"
)

print(
    "Ventana:",
    int(
        configuracion_referencia[
            "ventana_dias"
        ]
    )
)

print(
    "Modelo:",
    configuracion_referencia[
        "modelo"
    ]
)

print(
    "Average precision:",
    f"{configuracion_referencia['average_precision']:.4f}"
)

print(
    "\nConfiguración propuesta según "
    "la regla congelada:"
)

print(
    "Ventana propuesta:",
    int(
        configuracion_temporal_propuesta[
            "ventana_dias"
        ]
    )
)

print(
    "Modelo propuesto:",
    configuracion_temporal_propuesta[
        "modelo"
    ]
)

print(
    "Umbral propuesto:",
    f"{configuracion_temporal_propuesta['umbral_f2']:.2f}"
)

print(
    "Average precision propuesto:",
    f"{configuracion_temporal_propuesta['average_precision']:.4f}"
)

print(
    "F2 propuesto:",
    f"{configuracion_temporal_propuesta['f2']:.4f}"
)

print(
    "Recall propuesto:",
    f"{configuracion_temporal_propuesta['recall']:.4f}"
)


# ---------------------------------------------------------
# Almacenamiento de resultados
# ---------------------------------------------------------

RUTA_COMPARACION_TEMPORAL = (
    RUTA_RESULTADOS
    / "modelado_comparacion_temporal_oof.csv"
)

RUTA_PREDICCIONES_TEMPORALES = (
    RUTA_RESULTADOS
    / "modelado_predicciones_oof_temporales.pkl"
)

RUTA_PROPUESTA_TEMPORAL = (
    RUTA_RESULTADOS
    / "modelado_propuesta_configuracion_temporal.json"
)

comparacion_temporal.to_csv(
    RUTA_COMPARACION_TEMPORAL,
    index=False
)

predicciones_oof_temporales.to_pickle(
    RUTA_PREDICCIONES_TEMPORALES
)

propuesta_temporal_json = {
    "status": (
        "proposed_by_frozen_rule"
    ),
    "reference_configuration": {
        "window_days": int(
            configuracion_referencia[
                "ventana_dias"
            ]
        ),
        "model": str(
            configuracion_referencia[
                "modelo"
            ]
        ),
        "average_precision": float(
            configuracion_referencia[
                "average_precision"
            ]
        )
    },
    "proposed_configuration": {
        "window_days": int(
            configuracion_temporal_propuesta[
                "ventana_dias"
            ]
        ),
        "model": str(
            configuracion_temporal_propuesta[
                "modelo"
            ]
        ),
        "threshold": float(
            configuracion_temporal_propuesta[
                "umbral_f2"
            ]
        ),
        "average_precision": float(
            configuracion_temporal_propuesta[
                "average_precision"
            ]
        ),
        "auc_roc": float(
            configuracion_temporal_propuesta[
                "auc_roc"
            ]
        ),
        "f2": float(
            configuracion_temporal_propuesta[
                "f2"
            ]
        ),
        "recall": float(
            configuracion_temporal_propuesta[
                "recall"
            ]
        ),
        "precision": float(
            configuracion_temporal_propuesta[
                "precision"
            ]
        )
    },
    "equivalent_configurations": int(
        len(configuraciones_equivalentes)
    ),
    "selection_data": (
        "development_only"
    ),
    "internal_holdout_used": False,
    "external_test_used": False
}

with open(
    RUTA_PROPUESTA_TEMPORAL,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        propuesta_temporal_json,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Validación final
# ---------------------------------------------------------

columnas_probabilidades_temporales = [
    columna
    for columna
    in predicciones_oof_temporales.columns
    if columna.startswith(
        "probabilidad__"
    )
]

comparacion_temporal_valida = (
    comparacion_temporal.shape[0] == 9
    and comparacion_temporal[
        "ventana_dias"
    ].nunique() == 3
    and comparacion_temporal[
        "modelo"
    ].nunique() == 3
    and len(
        columnas_probabilidades_temporales
    ) == 9
    and predicciones_oof_temporales.shape
        == (44_203, 11)
    and predicciones_oof_temporales[
        "customer_id"
    ].nunique() == 44_203
    and predicciones_oof_temporales[
        columnas_probabilidades_temporales
    ].notna().all().all()
    and RUTA_COMPARACION_TEMPORAL.exists()
    and RUTA_PREDICCIONES_TEMPORALES.exists()
    and RUTA_PROPUESTA_TEMPORAL.exists()
)

print(
    "\n¿La comparación temporal es válida?",
    comparacion_temporal_valida
)

## Congelación de la configuración definitiva

**Decisión.** La regla definida antes de la comparación selecciona una ventana de 60 días, una regresión logística con regularización L2 y un umbral de clasificación de `0,20`.

**Justificación.** La configuración de referencia, formada por la ventana de 90 días y XGBoost, obtiene el mayor `average_precision`. Sin embargo, la alternativa de 60 días con regresión logística L2 se mantiene dentro de las tolerancias predefinidas para `average_precision`, `AUC-ROC`, F2 y `recall`. La ventana de 30 días no cumple simultáneamente estos criterios de equivalencia.

La ventana seleccionada requiere 30 días menos de observación que la configuración de referencia. Además, la regresión logística ofrece una estructura más sencilla e interpretable que XGBoost, sin una pérdida de rendimiento considerada material según el protocolo establecido.

**Uso de los datos.** La decisión se basa exclusivamente en las predicciones *out-of-fold* de los 44.203 clientes del conjunto de desarrollo. El holdout interno no interviene en la comparación temporal y el conjunto externo de prueba permanece bloqueado.

A partir de este punto quedan fijados la ventana de 60 días, las 176 variables predictoras, el algoritmo, sus hiperparámetros y el umbral de `0,20`. Estos elementos no se modificarán después de consultar el conjunto externo de prueba.

In [ ]:
# ---------------------------------------------------------
# Recuperación de la referencia y de la propuesta
# ---------------------------------------------------------

fila_referencia_final = (
    comparacion_temporal.loc[
        (comparacion_temporal["ventana_dias"] == 90)
        & (comparacion_temporal["modelo"] == "xgboost")
    ]
    .iloc[0]
)

fila_configuracion_final = (
    comparacion_temporal.loc[
        (comparacion_temporal["ventana_dias"] == 60)
        & (comparacion_temporal["modelo"] == "logistic_l2")
    ]
    .iloc[0]
)


# ---------------------------------------------------------
# Constantes definitivas del proyecto
# ---------------------------------------------------------

VENTANA_FINAL_DIAS = int(
    fila_configuracion_final["ventana_dias"]
)

MODELO_FINAL = str(
    fila_configuracion_final["modelo"]
)

UMBRAL_FINAL = float(
    fila_configuracion_final["umbral_f2"]
)

CRITERIO_FINAL = (
    "shortest_equivalent_window_then_simplest_model"
)

VARIABLES_FINALES = (
    columnas_predictoras.copy()
)

RUTA_DATASET_FINAL = (
    RUTA_DATASET_MODELADO_60D
)


# ---------------------------------------------------------
# Diferencias respecto a la mejor configuración absoluta
# ---------------------------------------------------------

diferencias_referencia_final = {
    "average_precision": float(
        fila_referencia_final["average_precision"]
        - fila_configuracion_final["average_precision"]
    ),
    "auc_roc": float(
        fila_referencia_final["auc_roc"]
        - fila_configuracion_final["auc_roc"]
    ),
    "f2": float(
        fila_referencia_final["f2"]
        - fila_configuracion_final["f2"]
    ),
    "recall": float(
        fila_referencia_final["recall"]
        - fila_configuracion_final["recall"]
    )
}


# ---------------------------------------------------------
# Comprobación de las tolerancias congeladas
# ---------------------------------------------------------

cumplimiento_tolerancias_final = {
    metrica: (
        diferencias_referencia_final[metrica]
        <= TOLERANCIAS_EQUIVALENCIA[metrica]
    )
    for metrica in [
        "average_precision",
        "auc_roc",
        "f2",
        "recall"
    ]
}

display(
    pd.DataFrame({
        "metrica": cumplimiento_tolerancias_final.keys(),
        "diferencia_frente_referencia": [
            diferencias_referencia_final[metrica]
            for metrica
            in cumplimiento_tolerancias_final
        ],
        "tolerancia_maxima": [
            TOLERANCIAS_EQUIVALENCIA[metrica]
            for metrica
            in cumplimiento_tolerancias_final
        ],
        "cumple": (
            cumplimiento_tolerancias_final.values()
        )
    })
)


# ---------------------------------------------------------
# Recuperación de los parámetros exactos del modelo
# ---------------------------------------------------------

modelo_final_base = clone(
    modelos_candidatos[MODELO_FINAL]
)

parametros_modelo_final = {
    clave: (
        valor
        if isinstance(
            valor,
            (
                str,
                int,
                float,
                bool,
                type(None)
            )
        )
        else str(valor)
    )
    for clave, valor
    in modelo_final_base.get_params().items()
}


# ---------------------------------------------------------
# Manifiesto definitivo previo al test externo
# ---------------------------------------------------------

manifiesto_modelo_final = {
    "status": "frozen_before_external_test",
    "selection_rule": CRITERIO_FINAL,
    "window_days": VENTANA_FINAL_DIAS,
    "model": MODELO_FINAL,
    "threshold": UMBRAL_FINAL,
    "features": len(VARIABLES_FINALES),
    "feature_schema": (
        "columnas_predictoras_176_90d.csv"
    ),
    "development_clients": int(
        len(X_desarrollo)
    ),
    "original_train_clients": int(
        len(datos_train_original)
    ),
    "external_test_clients": int(
        len(datos_test_externo)
    ),
    "cv_folds": int(NUMERO_PLIEGUES),
    "cv_random_state": int(SEMILLA_CV),
    "reference_configuration": {
        "window_days": int(
            fila_referencia_final["ventana_dias"]
        ),
        "model": str(
            fila_referencia_final["modelo"]
        ),
        "average_precision": float(
            fila_referencia_final["average_precision"]
        ),
        "auc_roc": float(
            fila_referencia_final["auc_roc"]
        ),
        "f2": float(
            fila_referencia_final["f2"]
        ),
        "recall": float(
            fila_referencia_final["recall"]
        )
    },
    "selected_configuration_oof": {
        "average_precision": float(
            fila_configuracion_final[
                "average_precision"
            ]
        ),
        "auc_roc": float(
            fila_configuracion_final["auc_roc"]
        ),
        "brier": float(
            fila_configuracion_final["brier"]
        ),
        "log_loss": float(
            fila_configuracion_final["log_loss"]
        ),
        "precision": float(
            fila_configuracion_final["precision"]
        ),
        "recall": float(
            fila_configuracion_final["recall"]
        ),
        "f1": float(
            fila_configuracion_final["f1"]
        ),
        "f2": float(
            fila_configuracion_final["f2"]
        ),
        "alert_rate": float(
            fila_configuracion_final[
                "tasa_alertas"
            ]
        )
    },
    "differences_from_reference": (
        diferencias_referencia_final
    ),
    "equivalence_tolerances": (
        TOLERANCIAS_EQUIVALENCIA
    ),
    "all_tolerances_satisfied": bool(
        all(
            cumplimiento_tolerancias_final.values()
        )
    ),
    "internal_holdout_used_for_temporal_selection": False,
    "external_test_used": False,
    "external_test_status": "locked",
    "model_parameters": parametros_modelo_final
}


# ---------------------------------------------------------
# Almacenamiento de la decisión definitiva
# ---------------------------------------------------------

RUTA_MANIFIESTO_MODELO_FINAL = (
    RUTA_RESULTADOS
    / "modelado_configuracion_final_60d.json"
)

RUTA_CONFIGURACION_FINAL_CSV = (
    RUTA_RESULTADOS
    / "modelado_configuracion_final_60d.csv"
)

RUTA_VARIABLES_FINALES = (
    RUTA_RESULTADOS
    / "modelado_variables_finales_60d.csv"
)

with open(
    RUTA_MANIFIESTO_MODELO_FINAL,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        manifiesto_modelo_final,
        archivo,
        ensure_ascii=False,
        indent=4
    )

pd.DataFrame([
    {
        "ventana_dias": VENTANA_FINAL_DIAS,
        "modelo": MODELO_FINAL,
        "umbral": UMBRAL_FINAL,
        "criterio": CRITERIO_FINAL,
        "average_precision_oof": float(
            fila_configuracion_final[
                "average_precision"
            ]
        ),
        "auc_roc_oof": float(
            fila_configuracion_final["auc_roc"]
        ),
        "precision_oof": float(
            fila_configuracion_final["precision"]
        ),
        "recall_oof": float(
            fila_configuracion_final["recall"]
        ),
        "f2_oof": float(
            fila_configuracion_final["f2"]
        )
    }
]).to_csv(
    RUTA_CONFIGURACION_FINAL_CSV,
    index=False
)

pd.DataFrame({
    "orden": np.arange(
        1,
        len(VARIABLES_FINALES) + 1
    ),
    "variable": VARIABLES_FINALES
}).to_csv(
    RUTA_VARIABLES_FINALES,
    index=False
)


# ---------------------------------------------------------
# Validación de la congelación definitiva
# ---------------------------------------------------------

configuracion_final_congelada = (
    comparacion_temporal_valida
    and VENTANA_FINAL_DIAS == 60
    and MODELO_FINAL == "logistic_l2"
    and np.isclose(
        UMBRAL_FINAL,
        0.20
    )
    and bool(
        fila_configuracion_final[
            "equivalente_a_referencia"
        ]
    )
    and all(
        cumplimiento_tolerancias_final.values()
    )
    and len(VARIABLES_FINALES) == 176
    and RUTA_DATASET_FINAL.exists()
    and RUTA_MANIFIESTO_MODELO_FINAL.exists()
    and RUTA_CONFIGURACION_FINAL_CSV.exists()
    and RUTA_VARIABLES_FINALES.exists()
)


# ---------------------------------------------------------
# Presentación de la decisión
# ---------------------------------------------------------

print(
    "Configuración de referencia:",
    "90 días + xgboost"
)

print(
    "Configuración definitiva:",
    f"{VENTANA_FINAL_DIAS} días + {MODELO_FINAL}"
)

print(
    "Umbral definitivo:",
    f"{UMBRAL_FINAL:.2f}"
)

print(
    "Diferencia de average precision:",
    f"{diferencias_referencia_final['average_precision']:.6f}"
)

print(
    "Diferencia de AUC-ROC:",
    f"{diferencias_referencia_final['auc_roc']:.6f}"
)

print(
    "Diferencia de F2:",
    f"{diferencias_referencia_final['f2']:.6f}"
)

print(
    "Diferencia de recall:",
    f"{diferencias_referencia_final['recall']:.6f}"
)

print(
    "Estado del test externo:",
    manifiesto_modelo_final[
        "external_test_status"
    ]
)

print(
    "¿La configuración definitiva se ha congelado "
    "correctamente?",
    configuracion_final_congelada
)

## Evaluación final sobre el conjunto externo de prueba

**Objetivo.** Se estima la capacidad de generalización de la configuración definitiva sobre clientes que no han intervenido en la selección de la ventana temporal, el algoritmo, los hiperparámetros ni el umbral de clasificación.

**Procedimiento.** Una vez congelada la configuración, el modelo se entrena de nuevo con los 52.004 clientes de la partición original de entrenamiento. Este conjunto reúne los clientes de desarrollo y del holdout interno, que pueden utilizarse conjuntamente para el ajuste final porque las decisiones metodológicas ya han sido fijadas.

La evaluación se realiza una única vez sobre los 13.001 clientes del conjunto externo de prueba, aplicando la siguiente configuración:

- Ventana de observación de 60 días.
- 176 variables predictoras.
- Regresión logística con regularización L2.
- Umbral de clasificación fijo de `0,20`.

**Criterio de interpretación.** Se calculan métricas probabilísticas y métricas dependientes del umbral. Los resultados se comparan de forma descriptiva con las estimaciones *out-of-fold* obtenidas en el conjunto de desarrollo para valorar la estabilidad del rendimiento.

Los resultados del conjunto externo de prueba constituyen la evaluación final de la configuración seleccionada. No se realiza ningún reajuste posterior del modelo, las variables, la ventana temporal o el umbral a partir de estas métricas.

In [ ]:
import platform
from time import perf_counter

import joblib
import sklearn

from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score
)


# ---------------------------------------------------------
# Carga del dataset correspondiente a la ventana final
# ---------------------------------------------------------

dataset_final_60d = pd.read_pickle(
    RUTA_DATASET_FINAL
)

particion_normalizada = (
    dataset_final_60d["particion"]
    .astype(str)
    .str.strip()
    .str.lower()
)

valores_particion = sorted(
    particion_normalizada.unique()
)

print(
    "Valores de partición encontrados:",
    valores_particion
)

if set(valores_particion) != {
    "train",
    "test"
}:
    raise ValueError(
        "La columna particion no contiene exactamente "
        "los valores train y test."
    )


# ---------------------------------------------------------
# Separación original de entrenamiento y test externo
# ---------------------------------------------------------

mascara_train_final = (
    particion_normalizada == "train"
)

mascara_test_externo = (
    particion_normalizada == "test"
)

datos_train_final = (
    dataset_final_60d.loc[
        mascara_train_final
    ]
    .copy()
)

datos_test_final = (
    dataset_final_60d.loc[
        mascara_test_externo
    ]
    .copy()
)

X_train_final = datos_train_final[
    VARIABLES_FINALES
].copy()

y_train_final = (
    datos_train_final["churn"]
    .astype(np.int8)
    .copy()
)

X_test_final = datos_test_final[
    VARIABLES_FINALES
].copy()

y_test_final = (
    datos_test_final["churn"]
    .astype(np.int8)
    .copy()
)


# ---------------------------------------------------------
# Validaciones previas a la apertura del test
# ---------------------------------------------------------

solapamiento_clientes_final = set(
    datos_train_final["customer_id"]
).intersection(
    set(
        datos_test_final["customer_id"]
    )
)

validaciones_previas_test = {
    "train_52004_clientes": (
        len(datos_train_final) == 52_004
    ),
    "test_13001_clientes": (
        len(datos_test_final) == 13_001
    ),
    "train_176_variables": (
        X_train_final.shape == (52_004, 176)
    ),
    "test_176_variables": (
        X_test_final.shape == (13_001, 176)
    ),
    "sin_solapamiento_clientes": (
        len(solapamiento_clientes_final) == 0
    ),
    "sin_nulos_train": (
        X_train_final.isna().sum().sum() == 0
    ),
    "sin_nulos_test": (
        X_test_final.isna().sum().sum() == 0
    ),
    "valores_finitos_train": (
        np.isfinite(
            X_train_final.to_numpy()
        ).all()
    ),
    "valores_finitos_test": (
        np.isfinite(
            X_test_final.to_numpy()
        ).all()
    ),
    "target_binario_train": (
        set(y_train_final.unique())
        == {0, 1}
    ),
    "target_binario_test": (
        set(y_test_final.unique())
        == {0, 1}
    ),
    "configuracion_60d": (
        VENTANA_FINAL_DIAS == 60
    ),
    "modelo_logistic_l2": (
        MODELO_FINAL == "logistic_l2"
    ),
    "umbral_020": (
        np.isclose(
            UMBRAL_FINAL,
            0.20
        )
    )
}

display(
    pd.DataFrame({
        "comprobacion": (
            validaciones_previas_test.keys()
        ),
        "resultado": (
            validaciones_previas_test.values()
        )
    })
)

if not all(
    validaciones_previas_test.values()
):
    raise ValueError(
        "No se cumplen todas las condiciones "
        "previas para evaluar el test externo."
    )


# ---------------------------------------------------------
# Entrenamiento definitivo con todo el train original
# ---------------------------------------------------------

modelo_final_entrenado = clone(
    modelos_candidatos[
        MODELO_FINAL
    ]
)

inicio_entrenamiento_final = perf_counter()

modelo_final_entrenado.fit(
    X_train_final,
    y_train_final
)

tiempo_entrenamiento_final = (
    perf_counter()
    - inicio_entrenamiento_final
)


# ---------------------------------------------------------
# Única generación de predicciones sobre el test externo
# ---------------------------------------------------------

inicio_prediccion_test = perf_counter()

probabilidades_test_externo = (
    modelo_final_entrenado
    .predict_proba(
        X_test_final
    )[:, 1]
)

predicciones_test_externo = (
    probabilidades_test_externo
    >= UMBRAL_FINAL
).astype(np.int8)

tiempo_prediccion_test = (
    perf_counter()
    - inicio_prediccion_test
)

if not np.isfinite(
    probabilidades_test_externo
).all():
    raise ValueError(
        "Las probabilidades del test contienen "
        "valores no finitos."
    )

if not (
    (
        probabilidades_test_externo >= 0
    )
    & (
        probabilidades_test_externo <= 1
    )
).all():
    raise ValueError(
        "Existen probabilidades fuera del intervalo "
        "[0, 1]."
    )


# ---------------------------------------------------------
# Métricas definitivas del test externo
# ---------------------------------------------------------

tn_test, fp_test, fn_test, tp_test = (
    confusion_matrix(
        y_test_final,
        predicciones_test_externo,
        labels=[0, 1]
    ).ravel()
)

especificidad_test = (
    tn_test
    / (
        tn_test
        + fp_test
    )
)

metricas_test_externo = {
    "conjunto": "external_test",
    "clientes": int(
        len(y_test_final)
    ),
    "positivos_reales": int(
        y_test_final.sum()
    ),
    "prevalencia_churn": float(
        y_test_final.mean()
    ),
    "ventana_dias": int(
        VENTANA_FINAL_DIAS
    ),
    "modelo": MODELO_FINAL,
    "variables": int(
        len(VARIABLES_FINALES)
    ),
    "umbral": float(
        UMBRAL_FINAL
    ),
    "average_precision": float(
        average_precision_score(
            y_test_final,
            probabilidades_test_externo
        )
    ),
    "auc_roc": float(
        roc_auc_score(
            y_test_final,
            probabilidades_test_externo
        )
    ),
    "brier": float(
        np.mean(
            (
                probabilidades_test_externo
                - y_test_final.to_numpy()
            ) ** 2
        )
    ),
    "log_loss": float(
        log_loss(
            y_test_final,
            probabilidades_test_externo,
            labels=[0, 1]
        )
    ),
    "precision": float(
        precision_score(
            y_test_final,
            predicciones_test_externo,
            zero_division=0
        )
    ),
    "recall": float(
        recall_score(
            y_test_final,
            predicciones_test_externo,
            zero_division=0
        )
    ),
    "especificidad": float(
        especificidad_test
    ),
    "f1": float(
        f1_score(
            y_test_final,
            predicciones_test_externo,
            zero_division=0
        )
    ),
    "f2": float(
        fbeta_score(
            y_test_final,
            predicciones_test_externo,
            beta=2,
            zero_division=0
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            y_test_final,
            predicciones_test_externo
        )
    ),
    "tasa_alertas": float(
        predicciones_test_externo.mean()
    ),
    "verdaderos_negativos": int(
        tn_test
    ),
    "falsos_positivos": int(
        fp_test
    ),
    "falsos_negativos": int(
        fn_test
    ),
    "verdaderos_positivos": int(
        tp_test
    ),
    "tiempo_entrenamiento_segundos": float(
        tiempo_entrenamiento_final
    ),
    "tiempo_prediccion_segundos": float(
        tiempo_prediccion_test
    )
}


# ---------------------------------------------------------
# Comparación con las métricas OOF de desarrollo
# ---------------------------------------------------------

metricas_oof_configuracion_final = {
    "conjunto": "development_oof",
    "clientes": int(
        len(y_desarrollo)
    ),
    "average_precision": float(
        fila_configuracion_final[
            "average_precision"
        ]
    ),
    "auc_roc": float(
        fila_configuracion_final[
            "auc_roc"
        ]
    ),
    "brier": float(
        fila_configuracion_final[
            "brier"
        ]
    ),
    "log_loss": float(
        fila_configuracion_final[
            "log_loss"
        ]
    ),
    "precision": float(
        fila_configuracion_final[
            "precision"
        ]
    ),
    "recall": float(
        fila_configuracion_final[
            "recall"
        ]
    ),
    "f1": float(
        fila_configuracion_final[
            "f1"
        ]
    ),
    "f2": float(
        fila_configuracion_final[
            "f2"
        ]
    ),
    "balanced_accuracy": float(
        fila_configuracion_final[
            "balanced_accuracy"
        ]
    ),
    "tasa_alertas": float(
        fila_configuracion_final[
            "tasa_alertas"
        ]
    )
}

comparacion_oof_test_final = pd.DataFrame([
    metricas_oof_configuracion_final,
    {
        clave: metricas_test_externo.get(
            clave,
            np.nan
        )
        for clave in (
            metricas_oof_configuracion_final.keys()
        )
    }
])

columnas_metricas_comparables = [
    "average_precision",
    "auc_roc",
    "brier",
    "log_loss",
    "precision",
    "recall",
    "f1",
    "f2",
    "balanced_accuracy",
    "tasa_alertas"
]

display(
    comparacion_oof_test_final[
        [
            "conjunto",
            "clientes",
            *columnas_metricas_comparables
        ]
    ].style.format({
        "average_precision": "{:.4f}",
        "auc_roc": "{:.4f}",
        "brier": "{:.4f}",
        "log_loss": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "f2": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "tasa_alertas": "{:.2%}"
    })
)


# ---------------------------------------------------------
# Tabla individual de predicciones
# ---------------------------------------------------------

predicciones_test_final = pd.DataFrame({
    "customer_id": (
        datos_test_final[
            "customer_id"
        ].to_numpy()
    ),
    "churn_real": (
        y_test_final.to_numpy()
    ),
    "probabilidad_churn": (
        probabilidades_test_externo
    ),
    "umbral": float(
        UMBRAL_FINAL
    ),
    "prediccion_churn": (
        predicciones_test_externo
    )
})

predicciones_test_final[
    "clasificacion"
] = np.select(
    [
        (
            predicciones_test_final[
                "churn_real"
            ] == 1
        )
        & (
            predicciones_test_final[
                "prediccion_churn"
            ] == 1
        ),
        (
            predicciones_test_final[
                "churn_real"
            ] == 0
        )
        & (
            predicciones_test_final[
                "prediccion_churn"
            ] == 0
        ),
        (
            predicciones_test_final[
                "churn_real"
            ] == 0
        )
        & (
            predicciones_test_final[
                "prediccion_churn"
            ] == 1
        )
    ],
    [
        "verdadero_positivo",
        "verdadero_negativo",
        "falso_positivo"
    ],
    default="falso_negativo"
)


# ---------------------------------------------------------
# Almacenamiento del modelo y de las evidencias finales
# ---------------------------------------------------------

RUTA_MODELOS = (
    RUTA_PROYECTO
    / "modelos"
)

RUTA_MODELOS.mkdir(
    parents=True,
    exist_ok=True
)

RUTA_MODELO_FINAL_ENTRENADO = (
    RUTA_MODELOS
    / "modelo_final_logistic_l2_60d.joblib"
)

RUTA_METRICAS_TEST_CSV = (
    RUTA_RESULTADOS
    / "modelado_metricas_test_externo_60d.csv"
)

RUTA_METRICAS_TEST_JSON = (
    RUTA_RESULTADOS
    / "modelado_metricas_test_externo_60d.json"
)

RUTA_PREDICCIONES_TEST = (
    RUTA_RESULTADOS
    / "modelado_predicciones_test_externo_60d.csv"
)

RUTA_COMPARACION_OOF_TEST = (
    RUTA_RESULTADOS
    / "modelado_comparacion_oof_test_60d.csv"
)

paquete_modelo_final = {
    "modelo": modelo_final_entrenado,
    "ventana_dias": VENTANA_FINAL_DIAS,
    "variables": VARIABLES_FINALES,
    "umbral": UMBRAL_FINAL,
    "modelo_nombre": MODELO_FINAL,
    "entrenamiento_clientes": len(
        X_train_final
    ),
    "sklearn_version": (
        sklearn.__version__
    ),
    "python_version": (
        platform.python_version()
    )
}

joblib.dump(
    paquete_modelo_final,
    RUTA_MODELO_FINAL_ENTRENADO
)

pd.DataFrame([
    metricas_test_externo
]).to_csv(
    RUTA_METRICAS_TEST_CSV,
    index=False
)

with open(
    RUTA_METRICAS_TEST_JSON,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        metricas_test_externo,
        archivo,
        ensure_ascii=False,
        indent=4
    )

predicciones_test_final.to_csv(
    RUTA_PREDICCIONES_TEST,
    index=False
)

comparacion_oof_test_final.to_csv(
    RUTA_COMPARACION_OOF_TEST,
    index=False
)


# ---------------------------------------------------------
# Validación final de la evaluación externa
# ---------------------------------------------------------

evaluacion_externa_valida = (
    len(datos_train_final) == 52_004
    and len(datos_test_final) == 13_001
    and predicciones_test_final.shape[0]
        == 13_001
    and predicciones_test_final[
        "customer_id"
    ].nunique() == 13_001
    and np.isfinite(
        probabilidades_test_externo
    ).all()
    and (
        tn_test
        + fp_test
        + fn_test
        + tp_test
    ) == 13_001
    and RUTA_MODELO_FINAL_ENTRENADO.exists()
    and RUTA_METRICAS_TEST_CSV.exists()
    and RUTA_METRICAS_TEST_JSON.exists()
    and RUTA_PREDICCIONES_TEST.exists()
    and RUTA_COMPARACION_OOF_TEST.exists()
)


# ---------------------------------------------------------
# Presentación de los resultados definitivos
# ---------------------------------------------------------

print(
    "\nClientes usados para entrenamiento final:",
    len(datos_train_final)
)

print(
    "Clientes en test externo:",
    len(datos_test_final)
)

print(
    "Average precision test:",
    f"{metricas_test_externo['average_precision']:.4f}"
)

print(
    "AUC-ROC test:",
    f"{metricas_test_externo['auc_roc']:.4f}"
)

print(
    "Precision test:",
    f"{metricas_test_externo['precision']:.4f}"
)

print(
    "Recall test:",
    f"{metricas_test_externo['recall']:.4f}"
)

print(
    "F1 test:",
    f"{metricas_test_externo['f1']:.4f}"
)

print(
    "F2 test:",
    f"{metricas_test_externo['f2']:.4f}"
)

print(
    "Tasa de alertas test:",
    f"{metricas_test_externo['tasa_alertas']:.2%}"
)

print(
    "Matriz de confusión:",
    {
        "TN": tn_test,
        "FP": fp_test,
        "FN": fn_test,
        "TP": tp_test
    }
)

print(
    "¿La evaluación externa se ha completado "
    "correctamente?",
    evaluacion_externa_valida
)

## Conclusión de la validación externa

**Resultado.** La configuración congelada —ventana de 60 días, regresión logística con regularización L2 y umbral de `0,20`— mantiene en el conjunto externo de prueba un rendimiento muy similar al estimado mediante predicciones *out-of-fold* sobre el conjunto de desarrollo.

En los 13.001 clientes de prueba se obtiene un `average_precision` de `0,9987`, un `AUC-ROC` de `0,9996`, una `precision` de `0,9705`, un `recall` de `0,9933` y un F2 de `0,9886`.

La matriz de confusión registra 9.933 verdaderos negativos, 90 falsos positivos, 20 falsos negativos y 2.958 verdaderos positivos. Por tanto, el modelo identifica correctamente 2.958 de los 2.978 casos reales de `churn` y clasifica al `23,44 %` de los clientes en la clase positiva.

**Estabilidad.** Las diferencias entre las estimaciones *out-of-fold* de desarrollo y los resultados del conjunto externo de prueba son reducidas. No se observa un deterioro relevante de la capacidad discriminativa ni de las métricas asociadas al umbral seleccionado. Esta concordancia respalda la estabilidad del modelo entre las particiones de clientes disponibles.

**Decisión final.** La configuración permanece sin modificaciones después de consultar el conjunto externo de prueba. Estos resultados constituyen la evaluación final del proyecto.

**Limitación temporal.** La variable objetivo `churn` procede del dataset original y no se dispone de una descripción completa de su construcción temporal. La evaluación confirma la capacidad de generalización entre las particiones proporcionadas, pero no demuestra por sí sola que los 60 días de actividad sean estrictamente anteriores al periodo empleado originalmente para definir el abandono.

In [ ]:
# ---------------------------------------------------------
# Diferencias entre desarrollo OOF y test externo
# ---------------------------------------------------------

metricas_comparacion_final = [
    "average_precision",
    "auc_roc",
    "brier",
    "log_loss",
    "precision",
    "recall",
    "f1",
    "f2",
    "balanced_accuracy",
    "tasa_alertas"
]

filas_diferencias_finales = []

for metrica in metricas_comparacion_final:

    valor_oof = float(
        metricas_oof_configuracion_final[
            metrica
        ]
    )

    valor_test = float(
        metricas_test_externo[
            metrica
        ]
    )

    filas_diferencias_finales.append({
        "metrica": metrica,
        "development_oof": valor_oof,
        "external_test": valor_test,
        "diferencia_test_menos_oof": (
            valor_test - valor_oof
        ),
        "diferencia_absoluta": abs(
            valor_test - valor_oof
        )
    })

diferencias_oof_test_final = pd.DataFrame(
    filas_diferencias_finales
)

display(
    diferencias_oof_test_final.style.format({
        "development_oof": "{:.6f}",
        "external_test": "{:.6f}",
        "diferencia_test_menos_oof": "{:+.6f}",
        "diferencia_absoluta": "{:.6f}"
    })
)


# ---------------------------------------------------------
# Resumen definitivo de la evaluación externa
# ---------------------------------------------------------

resumen_validacion_externa = {
    "status": (
        "external_test_evaluated_once"
    ),
    "configuration": {
        "window_days": int(
            VENTANA_FINAL_DIAS
        ),
        "model": MODELO_FINAL,
        "threshold": float(
            UMBRAL_FINAL
        ),
        "features": int(
            len(VARIABLES_FINALES)
        )
    },
    "training_clients": int(
        len(datos_train_final)
    ),
    "external_test_clients": int(
        len(datos_test_final)
    ),
    "external_test_metrics": {
        clave: (
            valor.item()
            if isinstance(
                valor,
                np.generic
            )
            else valor
        )
        for clave, valor
        in metricas_test_externo.items()
    },
    "confusion_matrix": {
        "true_negatives": int(
            tn_test
        ),
        "false_positives": int(
            fp_test
        ),
        "false_negatives": int(
            fn_test
        ),
        "true_positives": int(
            tp_test
        )
    },
    "comparison_with_development_oof": (
        diferencias_oof_test_final
        .to_dict(
            orient="records"
        )
    ),
    "model_changed_after_external_test": False,
    "threshold_changed_after_external_test": False,
    "feature_set_changed_after_external_test": False,
    "further_tuning_allowed": False,
    "temporal_label_definition_fully_documented": False,
    "interpretation": (
        "Stable predictive performance on the supplied "
        "external customer partition. The result does not "
        "by itself prove strict temporal anticipation because "
        "the original construction horizon of churn is not "
        "fully documented."
    )
}


# ---------------------------------------------------------
# Actualización del manifiesto definitivo
# ---------------------------------------------------------

with open(
    RUTA_MANIFIESTO_MODELO_FINAL,
    "r",
    encoding="utf-8"
) as archivo:
    manifiesto_actualizado = json.load(
        archivo
    )

manifiesto_actualizado[
    "status"
] = "external_test_evaluated_once"

manifiesto_actualizado[
    "external_test_used"
] = True

manifiesto_actualizado[
    "external_test_status"
] = (
    "evaluated_once_no_further_tuning_allowed"
)

manifiesto_actualizado[
    "external_test_metrics"
] = resumen_validacion_externa[
    "external_test_metrics"
]

manifiesto_actualizado[
    "external_test_confusion_matrix"
] = resumen_validacion_externa[
    "confusion_matrix"
]

manifiesto_actualizado[
    "configuration_changed_after_test"
] = False

with open(
    RUTA_MANIFIESTO_MODELO_FINAL,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        manifiesto_actualizado,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Almacenamiento de las evidencias
# ---------------------------------------------------------

RUTA_DIFERENCIAS_OOF_TEST = (
    RUTA_RESULTADOS
    / "modelado_diferencias_oof_test_60d.csv"
)

RUTA_CONCLUSION_VALIDACION_EXTERNA = (
    RUTA_RESULTADOS
    / "modelado_conclusion_validacion_externa_60d.json"
)

diferencias_oof_test_final.to_csv(
    RUTA_DIFERENCIAS_OOF_TEST,
    index=False
)

with open(
    RUTA_CONCLUSION_VALIDACION_EXTERNA,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        resumen_validacion_externa,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Validación del cierre de la evaluación
# ---------------------------------------------------------

cierre_validacion_externa_valido = (
    evaluacion_externa_valida
    and metricas_test_externo[
        "clientes"
    ] == 13_001
    and (
        int(tn_test)
        + int(fp_test)
        + int(fn_test)
        + int(tp_test)
    ) == 13_001
    and resumen_validacion_externa[
        "model_changed_after_external_test"
    ] is False
    and resumen_validacion_externa[
        "further_tuning_allowed"
    ] is False
    and manifiesto_actualizado[
        "external_test_used"
    ] is True
    and manifiesto_actualizado[
        "configuration_changed_after_test"
    ] is False
    and RUTA_DIFERENCIAS_OOF_TEST.exists()
    and RUTA_CONCLUSION_VALIDACION_EXTERNA.exists()
)


# ---------------------------------------------------------
# Presentación del cierre
# ---------------------------------------------------------

print(
    "Configuración evaluada:",
    f"{VENTANA_FINAL_DIAS} días + "
    f"{MODELO_FINAL} + umbral "
    f"{UMBRAL_FINAL:.2f}"
)

print(
    "Casos reales de churn:",
    int(
        y_test_final.sum()
    )
)

print(
    "Churn detectado:",
    int(
        tp_test
    )
)

print(
    "Churn no detectado:",
    int(
        fn_test
    )
)

print(
    "Falsas alertas:",
    int(
        fp_test
    )
)

print(
    "¿Se modificó la configuración tras el test?",
    resumen_validacion_externa[
        "model_changed_after_external_test"
    ]
)

print(
    "¿Se permite continuar ajustando el modelo?",
    resumen_validacion_externa[
        "further_tuning_allowed"
    ]
)

print(
    "¿Se ha cerrado correctamente la "
    "validación externa?",
    cierre_validacion_externa_valido
)

## Interpretación global del modelo final

**Objetivo.** Se identifican las variables con mayor influencia en las predicciones del modelo y la dirección de su asociación con la probabilidad estimada de `churn`.

**Criterio de interpretación.** El pipeline del modelo aplica un escalado previo a las variables predictoras. Por tanto, cada coeficiente de la regresión logística representa el cambio en el logaritmo de las *odds* asociado al incremento de una desviación estándar en la variable correspondiente, manteniendo constantes las demás. Esta estandarización facilita la comparación de coeficientes asociados a variables con escalas originales diferentes.

Un coeficiente positivo indica que un valor mayor de la variable se asocia con una probabilidad estimada de `churn` más alta. Un coeficiente negativo indica una asociación con una probabilidad menor. El valor absoluto del coeficiente se utiliza como una medida de relevancia global dentro del modelo.

**Datos utilizados.** La interpretación se obtiene directamente del modelo final entrenado con los 52.004 clientes de la partición original de entrenamiento. El modelo no se vuelve a ajustar y las etiquetas del conjunto externo de prueba no intervienen en el cálculo de los coeficientes.

**Limitaciones.** Los coeficientes representan asociaciones predictivas condicionadas al resto de las variables y no deben interpretarse como efectos causales. Además, la regularización y la presencia de variables correlacionadas pueden reducir o distribuir la contribución entre predictores relacionados, por lo que su magnitud debe interpretarse con prudencia.

In [ ]:
import json

import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# Recuperación del modelo final si fuera necesario
# ---------------------------------------------------------

if "modelo_final_entrenado" not in globals():

    paquete_modelo_recuperado = joblib.load(
        RUTA_MODELO_FINAL_ENTRENADO
    )

    modelo_final_entrenado = (
        paquete_modelo_recuperado[
            "modelo"
        ]
    )


# ---------------------------------------------------------
# Identificación de los componentes del pipeline
# ---------------------------------------------------------

if not hasattr(
    modelo_final_entrenado,
    "named_steps"
):
    raise ValueError(
        "El modelo final no presenta la estructura "
        "esperada de pipeline."
    )

escalador_final = next(
    (
        paso
        for paso
        in modelo_final_entrenado
        .named_steps
        .values()
        if isinstance(
            paso,
            StandardScaler
        )
    ),
    None
)

regresion_final = next(
    (
        paso
        for paso
        in modelo_final_entrenado
        .named_steps
        .values()
        if isinstance(
            paso,
            LogisticRegression
        )
    ),
    None
)

if escalador_final is None:
    raise ValueError(
        "No se ha encontrado StandardScaler "
        "en el pipeline final."
    )

if regresion_final is None:
    raise ValueError(
        "No se ha encontrado LogisticRegression "
        "en el pipeline final."
    )


# ---------------------------------------------------------
# Extracción de coeficientes estandarizados
# ---------------------------------------------------------

coeficientes_finales = (
    regresion_final.coef_
    .ravel()
)

if len(coeficientes_finales) != len(
    VARIABLES_FINALES
):
    raise ValueError(
        "El número de coeficientes no coincide "
        "con el número de variables."
    )

tabla_coeficientes_finales = pd.DataFrame({
    "variable": VARIABLES_FINALES,
    "coeficiente_estandarizado": (
        coeficientes_finales
    )
})

tabla_coeficientes_finales[
    "coeficiente_absoluto"
] = (
    tabla_coeficientes_finales[
        "coeficiente_estandarizado"
    ].abs()
)

tabla_coeficientes_finales[
    "importancia_relativa"
] = (
    tabla_coeficientes_finales[
        "coeficiente_absoluto"
    ]
    / tabla_coeficientes_finales[
        "coeficiente_absoluto"
    ].sum()
)

tabla_coeficientes_finales[
    "odds_ratio_por_desviacion"
] = np.exp(
    tabla_coeficientes_finales[
        "coeficiente_estandarizado"
    ]
)

tabla_coeficientes_finales[
    "direccion"
] = np.select(
    [
        tabla_coeficientes_finales[
            "coeficiente_estandarizado"
        ] > 0,
        tabla_coeficientes_finales[
            "coeficiente_estandarizado"
        ] < 0
    ],
    [
        "mayor_valor_mayor_riesgo",
        "mayor_valor_menor_riesgo"
    ],
    default="sin_efecto_lineal"
)


# ---------------------------------------------------------
# Separación entre alcance e indicador
# ---------------------------------------------------------

partes_variable = (
    tabla_coeficientes_finales[
        "variable"
    ]
    .str.rsplit(
        "__",
        n=1,
        expand=True
    )
)

tabla_coeficientes_finales[
    "alcance"
] = partes_variable[0]

tabla_coeficientes_finales[
    "indicador"
] = partes_variable[1]


def clasificar_familia_variable(
    alcance
):
    if alcance == "agg_global":
        return "agregado_global"

    if alcance == "agg_apps":
        return "agregado_aplicaciones"

    if alcance == "agg_voice":
        return "agregado_voz"

    if alcance == "agg_packages":
        return "agregado_paquetes"

    if alcance.startswith(
        "usage_app_"
    ):
        return "perfil_aplicacion"

    if alcance.startswith(
        "usage_voice_"
    ):
        return "perfil_voz"

    if alcance.startswith(
        "usage_pack_"
    ):
        return "perfil_paquete"

    return "otra_familia"


tabla_coeficientes_finales[
    "familia"
] = (
    tabla_coeficientes_finales[
        "alcance"
    ]
    .map(
        clasificar_familia_variable
    )
)

tabla_coeficientes_finales = (
    tabla_coeficientes_finales
    .sort_values(
        "coeficiente_absoluto",
        ascending=False
    )
    .reset_index(drop=True)
)

tabla_coeficientes_finales[
    "posicion_importancia"
] = np.arange(
    1,
    len(tabla_coeficientes_finales) + 1
)


# ---------------------------------------------------------
# Importancia agregada por indicador y familia
# ---------------------------------------------------------

importancia_por_indicador = (
    tabla_coeficientes_finales
    .groupby(
        "indicador",
        as_index=False
    )
    .agg(
        variables=(
            "variable",
            "count"
        ),
        suma_coeficiente_absoluto=(
            "coeficiente_absoluto",
            "sum"
        ),
        importancia_relativa=(
            "importancia_relativa",
            "sum"
        )
    )
    .sort_values(
        "importancia_relativa",
        ascending=False
    )
    .reset_index(drop=True)
)

importancia_por_familia = (
    tabla_coeficientes_finales
    .groupby(
        "familia",
        as_index=False
    )
    .agg(
        variables=(
            "variable",
            "count"
        ),
        suma_coeficiente_absoluto=(
            "coeficiente_absoluto",
            "sum"
        ),
        importancia_relativa=(
            "importancia_relativa",
            "sum"
        )
    )
    .sort_values(
        "importancia_relativa",
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Variables con asociaciones positivas y negativas
# ---------------------------------------------------------

variables_mayor_riesgo = (
    tabla_coeficientes_finales.loc[
        tabla_coeficientes_finales[
            "coeficiente_estandarizado"
        ] > 0
    ]
    .sort_values(
        "coeficiente_estandarizado",
        ascending=False
    )
    .head(10)
)

variables_menor_riesgo = (
    tabla_coeficientes_finales.loc[
        tabla_coeficientes_finales[
            "coeficiente_estandarizado"
        ] < 0
    ]
    .sort_values(
        "coeficiente_estandarizado",
        ascending=True
    )
    .head(10)
)


# ---------------------------------------------------------
# Presentación de los principales resultados
# ---------------------------------------------------------

print(
    "Intercepto del modelo:",
    f"{regresion_final.intercept_[0]:.6f}"
)

print(
    "Variables interpretadas:",
    len(tabla_coeficientes_finales)
)

print(
    "\nVariables con mayor coeficiente absoluto:"
)

display(
    tabla_coeficientes_finales[
        [
            "posicion_importancia",
            "variable",
            "coeficiente_estandarizado",
            "coeficiente_absoluto",
            "odds_ratio_por_desviacion",
            "direccion",
            "familia",
            "indicador"
        ]
    ]
    .head(20)
    .style.format({
        "coeficiente_estandarizado": "{:+.4f}",
        "coeficiente_absoluto": "{:.4f}",
        "odds_ratio_por_desviacion": "{:.4f}"
    })
)

print(
    "\nIndicadores con mayor importancia acumulada:"
)

display(
    importancia_por_indicador
    .head(11)
    .style.format({
        "suma_coeficiente_absoluto": "{:.4f}",
        "importancia_relativa": "{:.2%}"
    })
)

print(
    "\nImportancia acumulada por familia:"
)

display(
    importancia_por_familia
    .style.format({
        "suma_coeficiente_absoluto": "{:.4f}",
        "importancia_relativa": "{:.2%}"
    })
)

print(
    "\nVariables asociadas con mayor riesgo:"
)

display(
    variables_mayor_riesgo[
        [
            "variable",
            "coeficiente_estandarizado",
            "odds_ratio_por_desviacion"
        ]
    ]
    .style.format({
        "coeficiente_estandarizado": "{:+.4f}",
        "odds_ratio_por_desviacion": "{:.4f}"
    })
)

print(
    "\nVariables asociadas con menor riesgo:"
)

display(
    variables_menor_riesgo[
        [
            "variable",
            "coeficiente_estandarizado",
            "odds_ratio_por_desviacion"
        ]
    ]
    .style.format({
        "coeficiente_estandarizado": "{:+.4f}",
        "odds_ratio_por_desviacion": "{:.4f}"
    })
)


# ---------------------------------------------------------
# Almacenamiento de evidencias interpretativas
# ---------------------------------------------------------

RUTA_COEFICIENTES_MODELO_FINAL = (
    RUTA_RESULTADOS
    / "modelado_coeficientes_logistic_l2_60d.csv"
)

RUTA_IMPORTANCIA_INDICADORES = (
    RUTA_RESULTADOS
    / "modelado_importancia_por_indicador_60d.csv"
)

RUTA_IMPORTANCIA_FAMILIAS = (
    RUTA_RESULTADOS
    / "modelado_importancia_por_familia_60d.csv"
)

RUTA_RESUMEN_INTERPRETABILIDAD = (
    RUTA_RESULTADOS
    / "modelado_resumen_interpretabilidad_60d.json"
)

tabla_coeficientes_finales.to_csv(
    RUTA_COEFICIENTES_MODELO_FINAL,
    index=False
)

importancia_por_indicador.to_csv(
    RUTA_IMPORTANCIA_INDICADORES,
    index=False
)

importancia_por_familia.to_csv(
    RUTA_IMPORTANCIA_FAMILIAS,
    index=False
)

resumen_interpretabilidad = {
    "model": MODELO_FINAL,
    "window_days": int(
        VENTANA_FINAL_DIAS
    ),
    "features": int(
        len(VARIABLES_FINALES)
    ),
    "intercept": float(
        regresion_final.intercept_[0]
    ),
    "coefficient_interpretation": (
        "Standardized coefficient: change in log-odds "
        "for an increase of one standard deviation."
    ),
    "top_20_absolute_coefficients": (
        tabla_coeficientes_finales
        .head(20)
        .to_dict(
            orient="records"
        )
    ),
    "importance_by_indicator": (
        importancia_por_indicador
        .to_dict(
            orient="records"
        )
    ),
    "importance_by_family": (
        importancia_por_familia
        .to_dict(
            orient="records"
        )
    ),
    "causal_interpretation_allowed": False
}

with open(
    RUTA_RESUMEN_INTERPRETABILIDAD,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        resumen_interpretabilidad,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Validación del bloque
# ---------------------------------------------------------

interpretabilidad_valida = (
    len(tabla_coeficientes_finales) == 176
    and tabla_coeficientes_finales[
        "variable"
    ].nunique() == 176
    and np.isfinite(
        tabla_coeficientes_finales[
            [
                "coeficiente_estandarizado",
                "coeficiente_absoluto",
                "odds_ratio_por_desviacion"
            ]
        ].to_numpy()
    ).all()
    and np.isclose(
        tabla_coeficientes_finales[
            "importancia_relativa"
        ].sum(),
        1.0
    )
    and importancia_por_indicador[
        "indicador"
    ].nunique() == 11
    and RUTA_COEFICIENTES_MODELO_FINAL.exists()
    and RUTA_IMPORTANCIA_INDICADORES.exists()
    and RUTA_IMPORTANCIA_FAMILIAS.exists()
    and RUTA_RESUMEN_INTERPRETABILIDAD.exists()
)

print(
    "\n¿La interpretación global del modelo "
    "se ha generado correctamente?",
    interpretabilidad_valida
)

## Estabilidad de los coeficientes del modelo

**Objetivo.** Se comprueba si las asociaciones identificadas por la regresión logística se mantienen entre diferentes muestras de entrenamiento, evitando atribuir una relevancia excesiva a coeficientes dependientes de una división concreta de los datos.

**Criterio de análisis.** La configuración definitiva de 60 días se ajusta en las cinco divisiones utilizadas durante la validación cruzada. Para cada variable predictora se analizan la media y la dispersión de sus coeficientes, así como la proporción de pliegues en los que conserva la misma dirección de asociación.

La estabilidad del signo no constituye un criterio adicional de selección ni modifica el modelo final. Se utiliza únicamente como un diagnóstico descriptivo de la robustez de la interpretación.

También se calcula, para cada familia de variables, la media de los coeficientes absolutos. Esta medida facilita la comparación entre familias sin favorecer automáticamente a aquellas que contienen un mayor número de variables predictoras.

**Datos utilizados.** El análisis emplea exclusivamente los clientes del conjunto de desarrollo. El conjunto externo de prueba no interviene en el cálculo ni en la interpretación de la estabilidad.

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression


# ---------------------------------------------------------
# Datos de desarrollo correspondientes a 60 días
# ---------------------------------------------------------

X_interpretacion_cv = (
    matrices_desarrollo_temporales[
        VENTANA_FINAL_DIAS
    ]["X"]
)

y_interpretacion_cv = (
    matrices_desarrollo_temporales[
        VENTANA_FINAL_DIAS
    ]["y"]
)

if X_interpretacion_cv.shape != (
    44_203,
    176
):
    raise ValueError(
        "La matriz de desarrollo de 60 días "
        "no tiene las dimensiones esperadas."
    )


# ---------------------------------------------------------
# Ajuste del mismo modelo en los cinco pliegues
# ---------------------------------------------------------

coeficientes_por_pliegue = []

for numero_pliegue, (
    indices_entrenamiento,
    indices_validacion
) in enumerate(
    validacion_cruzada.split(
        X_interpretacion_cv,
        y_interpretacion_cv
    ),
    start=1
):

    modelo_pliegue = clone(
        modelos_candidatos[
            MODELO_FINAL
        ]
    )

    modelo_pliegue.fit(
        X_interpretacion_cv.iloc[
            indices_entrenamiento
        ],
        y_interpretacion_cv.iloc[
            indices_entrenamiento
        ]
    )

    regresion_pliegue = next(
        (
            paso
            for paso
            in modelo_pliegue
            .named_steps
            .values()
            if isinstance(
                paso,
                LogisticRegression
            )
        ),
        None
    )

    if regresion_pliegue is None:
        raise ValueError(
            f"No se encontró la regresión en "
            f"el pliegue {numero_pliegue}."
        )

    coeficientes_pliegue = (
        regresion_pliegue.coef_
        .ravel()
    )

    for variable, coeficiente in zip(
        VARIABLES_FINALES,
        coeficientes_pliegue
    ):
        coeficientes_por_pliegue.append({
            "pliegue": numero_pliegue,
            "variable": variable,
            "coeficiente": float(
                coeficiente
            ),
            "coeficiente_absoluto": float(
                abs(coeficiente)
            ),
            "signo": int(
                np.sign(coeficiente)
            )
        })

    print(
        f"Pliegue {numero_pliegue}/5 completado | "
        f"entrenamiento: "
        f"{len(indices_entrenamiento):,} | "
        f"validación: "
        f"{len(indices_validacion):,}"
    )


coeficientes_cv = pd.DataFrame(
    coeficientes_por_pliegue
)


# ---------------------------------------------------------
# Resumen de estabilidad por variable
# ---------------------------------------------------------

estabilidad_coeficientes = (
    coeficientes_cv
    .groupby(
        "variable",
        as_index=False
    )
    .agg(
        coeficiente_medio=(
            "coeficiente",
            "mean"
        ),
        desviacion_coeficiente=(
            "coeficiente",
            "std"
        ),
        coeficiente_minimo=(
            "coeficiente",
            "min"
        ),
        coeficiente_maximo=(
            "coeficiente",
            "max"
        ),
        coeficiente_absoluto_medio=(
            "coeficiente_absoluto",
            "mean"
        ),
        pliegues_signo_positivo=(
            "signo",
            lambda valores: int(
                (valores > 0).sum()
            )
        ),
        pliegues_signo_negativo=(
            "signo",
            lambda valores: int(
                (valores < 0).sum()
            )
        )
    )
)

estabilidad_coeficientes[
    "consistencia_signo"
] = (
    estabilidad_coeficientes[
        [
            "pliegues_signo_positivo",
            "pliegues_signo_negativo"
        ]
    ]
    .max(axis=1)
    / NUMERO_PLIEGUES
)

estabilidad_coeficientes[
    "signo_dominante"
] = np.select(
    [
        estabilidad_coeficientes[
            "pliegues_signo_positivo"
        ]
        > estabilidad_coeficientes[
            "pliegues_signo_negativo"
        ],
        estabilidad_coeficientes[
            "pliegues_signo_negativo"
        ]
        > estabilidad_coeficientes[
            "pliegues_signo_positivo"
        ]
    ],
    [
        "positivo",
        "negativo"
    ],
    default="indeterminado"
)


# ---------------------------------------------------------
# Integración con los coeficientes del modelo final
# ---------------------------------------------------------

estabilidad_coeficientes = (
    estabilidad_coeficientes
    .merge(
        tabla_coeficientes_finales[
            [
                "variable",
                "coeficiente_estandarizado",
                "coeficiente_absoluto",
                "familia",
                "indicador",
                "posicion_importancia"
            ]
        ],
        on="variable",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        "coeficiente_absoluto",
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Importancia normalizada por familia
# ---------------------------------------------------------

importancia_familias_normalizada = (
    tabla_coeficientes_finales
    .groupby(
        "familia",
        as_index=False
    )
    .agg(
        numero_variables=(
            "variable",
            "count"
        ),
        suma_coeficiente_absoluto=(
            "coeficiente_absoluto",
            "sum"
        ),
        media_coeficiente_absoluto=(
            "coeficiente_absoluto",
            "mean"
        ),
        mediana_coeficiente_absoluto=(
            "coeficiente_absoluto",
            "median"
        ),
        coeficiente_maximo=(
            "coeficiente_absoluto",
            "max"
        )
    )
)

importancia_familias_normalizada[
    "importancia_relativa_total"
] = (
    importancia_familias_normalizada[
        "suma_coeficiente_absoluto"
    ]
    / importancia_familias_normalizada[
        "suma_coeficiente_absoluto"
    ].sum()
)

importancia_familias_normalizada = (
    importancia_familias_normalizada
    .sort_values(
        "media_coeficiente_absoluto",
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Resumen de estabilidad de las variables principales
# ---------------------------------------------------------

estabilidad_top_20 = (
    estabilidad_coeficientes.loc[
        estabilidad_coeficientes[
            "posicion_importancia"
        ] <= 20,
        [
            "posicion_importancia",
            "variable",
            "coeficiente_estandarizado",
            "coeficiente_medio",
            "desviacion_coeficiente",
            "coeficiente_minimo",
            "coeficiente_maximo",
            "consistencia_signo",
            "signo_dominante",
            "familia",
            "indicador"
        ]
    ]
    .sort_values(
        "posicion_importancia"
    )
)

print(
    "\nEstabilidad de las veinte variables "
    "principales:"
)

display(
    estabilidad_top_20.style.format({
        "coeficiente_estandarizado": "{:+.4f}",
        "coeficiente_medio": "{:+.4f}",
        "desviacion_coeficiente": "{:.4f}",
        "coeficiente_minimo": "{:+.4f}",
        "coeficiente_maximo": "{:+.4f}",
        "consistencia_signo": "{:.0%}"
    })
)

print(
    "\nImportancia normalizada por familia:"
)

display(
    importancia_familias_normalizada
    .style.format({
        "suma_coeficiente_absoluto": "{:.4f}",
        "media_coeficiente_absoluto": "{:.4f}",
        "mediana_coeficiente_absoluto": "{:.4f}",
        "coeficiente_maximo": "{:.4f}",
        "importancia_relativa_total": "{:.2%}"
    })
)


# ---------------------------------------------------------
# Indicadores generales de estabilidad
# ---------------------------------------------------------

variables_signo_100 = int(
    (
        estabilidad_coeficientes[
            "consistencia_signo"
        ] == 1.0
    ).sum()
)

variables_signo_80 = int(
    (
        estabilidad_coeficientes[
            "consistencia_signo"
        ] >= 0.8
    ).sum()
)

top20_signo_100 = int(
    (
        estabilidad_top_20[
            "consistencia_signo"
        ] == 1.0
    ).sum()
)

print(
    "\nVariables con el mismo signo en "
    "los cinco pliegues:",
    variables_signo_100
)

print(
    "Variables con consistencia de signo "
    "igual o superior al 80 %:",
    variables_signo_80
)

print(
    "Variables del top 20 con el mismo signo "
    "en los cinco pliegues:",
    top20_signo_100
)


# ---------------------------------------------------------
# Almacenamiento de evidencias
# ---------------------------------------------------------

RUTA_COEFICIENTES_POR_PLIEGUE = (
    RUTA_RESULTADOS
    / "modelado_coeficientes_por_pliegue_60d.csv"
)

RUTA_ESTABILIDAD_COEFICIENTES = (
    RUTA_RESULTADOS
    / "modelado_estabilidad_coeficientes_60d.csv"
)

RUTA_FAMILIAS_NORMALIZADAS = (
    RUTA_RESULTADOS
    / "modelado_importancia_familias_normalizada_60d.csv"
)

RUTA_RESUMEN_ESTABILIDAD = (
    RUTA_RESULTADOS
    / "modelado_resumen_estabilidad_coeficientes_60d.json"
)

coeficientes_cv.to_csv(
    RUTA_COEFICIENTES_POR_PLIEGUE,
    index=False
)

estabilidad_coeficientes.to_csv(
    RUTA_ESTABILIDAD_COEFICIENTES,
    index=False
)

importancia_familias_normalizada.to_csv(
    RUTA_FAMILIAS_NORMALIZADAS,
    index=False
)

resumen_estabilidad = {
    "window_days": int(
        VENTANA_FINAL_DIAS
    ),
    "model": MODELO_FINAL,
    "folds": int(
        NUMERO_PLIEGUES
    ),
    "development_clients": int(
        len(X_interpretacion_cv)
    ),
    "variables": int(
        len(VARIABLES_FINALES)
    ),
    "variables_same_sign_all_folds": (
        variables_signo_100
    ),
    "variables_sign_consistency_ge_80": (
        variables_signo_80
    ),
    "top_20_same_sign_all_folds": (
        top20_signo_100
    ),
    "external_test_used": False
}

with open(
    RUTA_RESUMEN_ESTABILIDAD,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        resumen_estabilidad,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Validación final
# ---------------------------------------------------------

estabilidad_coeficientes_valida = (
    coeficientes_cv.shape
        == (176 * NUMERO_PLIEGUES, 5)
    and estabilidad_coeficientes.shape[0]
        == 176
    and estabilidad_coeficientes[
        "variable"
    ].nunique() == 176
    and estabilidad_top_20.shape[0] == 20
    and importancia_familias_normalizada[
        "familia"
    ].nunique() == 7
    and np.isfinite(
        estabilidad_coeficientes[
            [
                "coeficiente_medio",
                "desviacion_coeficiente",
                "consistencia_signo"
            ]
        ].to_numpy()
    ).all()
    and RUTA_COEFICIENTES_POR_PLIEGUE.exists()
    and RUTA_ESTABILIDAD_COEFICIENTES.exists()
    and RUTA_FAMILIAS_NORMALIZADAS.exists()
    and RUTA_RESUMEN_ESTABILIDAD.exists()
)

print(
    "\n¿El análisis de estabilidad se ha "
    "generado correctamente?",
    estabilidad_coeficientes_valida
)

## Calibración y concentración del riesgo

**Objetivo.** Se evalúa si las probabilidades generadas por el modelo final son coherentes con las frecuencias observadas de `churn` y se analiza hasta qué punto el riesgo se concentra entre los clientes con puntuaciones más elevadas.

**Criterio de análisis.** La calibración se estudia agrupando a los clientes según su probabilidad estimada y comparando, en cada grupo, la probabilidad media predicha con la proporción observada de `churn`. Además, se calculan el error de calibración esperado y la mejora del `Brier score` respecto a un modelo de referencia que asigna a todos los clientes la prevalencia observada en entrenamiento.

La capacidad del modelo para priorizar clientes se analiza mediante deciles de riesgo y curvas de captura acumulada. Para diferentes porcentajes de clientes con mayor probabilidad estimada, se calculan la proporción acumulada de casos de `churn` identificados, la `precision` y el *lift* respecto a una selección aleatoria.

**Datos utilizados.** El análisis se realiza sobre el conjunto externo de prueba una vez completada la evaluación final. Su finalidad es exclusivamente descriptiva y sus resultados no se utilizan para modificar la ventana temporal, las variables predictoras, el modelo, sus hiperparámetros ni el umbral de clasificación.

In [ ]:
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# Base de análisis
# ---------------------------------------------------------

analisis_riesgo_test = (
    predicciones_test_final[
        [
            "customer_id",
            "churn_real",
            "probabilidad_churn",
            "prediccion_churn"
        ]
    ]
    .copy()
)

numero_clientes_test = len(
    analisis_riesgo_test
)

churn_total_test = int(
    analisis_riesgo_test[
        "churn_real"
    ].sum()
)

prevalencia_test = float(
    analisis_riesgo_test[
        "churn_real"
    ].mean()
)

if numero_clientes_test != 13_001:
    raise ValueError(
        "El número de clientes del test no coincide "
        "con el esperado."
    )


# ---------------------------------------------------------
# 1. Tabla de calibración por grupos de probabilidad
# ---------------------------------------------------------

analisis_riesgo_test[
    "grupo_calibracion"
] = pd.qcut(
    analisis_riesgo_test[
        "probabilidad_churn"
    ].rank(
        method="first"
    ),
    q=10,
    labels=False
) + 1

tabla_calibracion = (
    analisis_riesgo_test
    .groupby(
        "grupo_calibracion",
        as_index=False
    )
    .agg(
        clientes=(
            "customer_id",
            "count"
        ),
        probabilidad_media=(
            "probabilidad_churn",
            "mean"
        ),
        churn_observado=(
            "churn_real",
            "mean"
        ),
        casos_churn=(
            "churn_real",
            "sum"
        )
    )
)

tabla_calibracion[
    "diferencia_absoluta"
] = (
    tabla_calibracion[
        "probabilidad_media"
    ]
    - tabla_calibracion[
        "churn_observado"
    ]
).abs()

tabla_calibracion[
    "peso"
] = (
    tabla_calibracion[
        "clientes"
    ]
    / numero_clientes_test
)

error_calibracion_esperado = float(
    (
        tabla_calibracion[
            "diferencia_absoluta"
        ]
        * tabla_calibracion[
            "peso"
        ]
    ).sum()
)

error_calibracion_maximo = float(
    tabla_calibracion[
        "diferencia_absoluta"
    ].max()
)


# ---------------------------------------------------------
# 2. Brier score y comparación con el modelo nulo
# ---------------------------------------------------------

brier_modelo = float(
    np.mean(
        (
            analisis_riesgo_test[
                "probabilidad_churn"
            ].to_numpy()
            - analisis_riesgo_test[
                "churn_real"
            ].to_numpy()
        ) ** 2
    )
)

brier_modelo_nulo = float(
    np.mean(
        (
            prevalencia_test
            - analisis_riesgo_test[
                "churn_real"
            ].to_numpy()
        ) ** 2
    )
)

brier_skill_score = float(
    1
    - (
        brier_modelo
        / brier_modelo_nulo
    )
)


# ---------------------------------------------------------
# 3. Deciles ordenados de mayor a menor riesgo
# ---------------------------------------------------------

ranking_riesgo_test = (
    analisis_riesgo_test
    .sort_values(
        "probabilidad_churn",
        ascending=False
    )
    .reset_index(drop=True)
)

ranking_riesgo_test[
    "posicion_riesgo"
] = np.arange(
    1,
    numero_clientes_test + 1
)

ranking_riesgo_test[
    "decil_riesgo"
] = (
    np.floor(
        np.arange(
            numero_clientes_test
        )
        * 10
        / numero_clientes_test
    )
    .astype(int)
    + 1
)

tabla_deciles_riesgo = (
    ranking_riesgo_test
    .groupby(
        "decil_riesgo",
        as_index=False
    )
    .agg(
        clientes=(
            "customer_id",
            "count"
        ),
        probabilidad_media=(
            "probabilidad_churn",
            "mean"
        ),
        probabilidad_minima=(
            "probabilidad_churn",
            "min"
        ),
        probabilidad_maxima=(
            "probabilidad_churn",
            "max"
        ),
        casos_churn=(
            "churn_real",
            "sum"
        ),
        tasa_churn=(
            "churn_real",
            "mean"
        )
    )
)

tabla_deciles_riesgo[
    "porcentaje_churn_total"
] = (
    tabla_deciles_riesgo[
        "casos_churn"
    ]
    / churn_total_test
)

tabla_deciles_riesgo[
    "lift"
] = (
    tabla_deciles_riesgo[
        "tasa_churn"
    ]
    / prevalencia_test
)

tabla_deciles_riesgo[
    "churn_acumulado"
] = (
    tabla_deciles_riesgo[
        "casos_churn"
    ].cumsum()
)

tabla_deciles_riesgo[
    "captura_acumulada"
] = (
    tabla_deciles_riesgo[
        "churn_acumulado"
    ]
    / churn_total_test
)


# ---------------------------------------------------------
# 4. Captura para diferentes tamaños de campaña
# ---------------------------------------------------------

porcentajes_campana = [
    0.05,
    0.10,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50
]

resultados_captura = []

for porcentaje in porcentajes_campana:

    numero_seleccionado = int(
        np.ceil(
            numero_clientes_test
            * porcentaje
        )
    )

    clientes_seleccionados = (
        ranking_riesgo_test
        .head(
            numero_seleccionado
        )
    )

    churn_detectado = int(
        clientes_seleccionados[
            "churn_real"
        ].sum()
    )

    precision_segmento = float(
        clientes_seleccionados[
            "churn_real"
        ].mean()
    )

    resultados_captura.append({
        "porcentaje_clientes": (
            porcentaje
        ),
        "clientes_seleccionados": (
            numero_seleccionado
        ),
        "churn_detectado": (
            churn_detectado
        ),
        "captura_churn": (
            churn_detectado
            / churn_total_test
        ),
        "precision_segmento": (
            precision_segmento
        ),
        "lift": (
            precision_segmento
            / prevalencia_test
        )
    })

tabla_captura_riesgo = pd.DataFrame(
    resultados_captura
)


# ---------------------------------------------------------
# 5. Curva de captura acumulada completa
# ---------------------------------------------------------

ranking_riesgo_test[
    "clientes_acumulados"
] = (
    ranking_riesgo_test[
        "posicion_riesgo"
    ]
    / numero_clientes_test
)

ranking_riesgo_test[
    "churn_acumulado"
] = (
    ranking_riesgo_test[
        "churn_real"
    ].cumsum()
)

ranking_riesgo_test[
    "captura_churn_acumulada"
] = (
    ranking_riesgo_test[
        "churn_acumulado"
    ]
    / churn_total_test
)


# ---------------------------------------------------------
# Presentación de tablas
# ---------------------------------------------------------

print(
    "Prevalencia de churn en test:",
    f"{prevalencia_test:.2%}"
)

print(
    "Brier score del modelo:",
    f"{brier_modelo:.6f}"
)

print(
    "Brier score del modelo nulo:",
    f"{brier_modelo_nulo:.6f}"
)

print(
    "Brier skill score:",
    f"{brier_skill_score:.4f}"
)

print(
    "Error de calibración esperado:",
    f"{error_calibracion_esperado:.4f}"
)

print(
    "Error máximo de calibración:",
    f"{error_calibracion_maximo:.4f}"
)

print(
    "\nCalibración por grupos:"
)

display(
    tabla_calibracion.style.format({
        "probabilidad_media": "{:.4f}",
        "churn_observado": "{:.4f}",
        "diferencia_absoluta": "{:.4f}",
        "peso": "{:.2%}"
    })
)

print(
    "\nResultados por decil de riesgo:"
)

display(
    tabla_deciles_riesgo.style.format({
        "probabilidad_media": "{:.4f}",
        "probabilidad_minima": "{:.4f}",
        "probabilidad_maxima": "{:.4f}",
        "tasa_churn": "{:.2%}",
        "porcentaje_churn_total": "{:.2%}",
        "lift": "{:.2f}",
        "captura_acumulada": "{:.2%}"
    })
)

print(
    "\nCaptura según el tamaño de la campaña:"
)

display(
    tabla_captura_riesgo.style.format({
        "porcentaje_clientes": "{:.0%}",
        "captura_churn": "{:.2%}",
        "precision_segmento": "{:.2%}",
        "lift": "{:.2f}"
    })
)


# ---------------------------------------------------------
# 6. Gráfico de calibración
# ---------------------------------------------------------

figura_calibracion, eje_calibracion = (
    plt.subplots(
        figsize=(7, 6)
    )
)

eje_calibracion.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Calibración perfecta"
)

eje_calibracion.plot(
    tabla_calibracion[
        "probabilidad_media"
    ],
    tabla_calibracion[
        "churn_observado"
    ],
    marker="o",
    label="Modelo final"
)

eje_calibracion.set_title(
    "Calibración del modelo en el test externo"
)

eje_calibracion.set_xlabel(
    "Probabilidad media estimada"
)

eje_calibracion.set_ylabel(
    "Proporción observada de churn"
)

eje_calibracion.set_xlim(
    0,
    1
)

eje_calibracion.set_ylim(
    0,
    1
)

eje_calibracion.grid(
    alpha=0.3
)

eje_calibracion.legend()

figura_calibracion.tight_layout()


# ---------------------------------------------------------
# 7. Gráfico de captura acumulada
# ---------------------------------------------------------

figura_captura, eje_captura = (
    plt.subplots(
        figsize=(7, 6)
    )
)

eje_captura.plot(
    ranking_riesgo_test[
        "clientes_acumulados"
    ],
    ranking_riesgo_test[
        "captura_churn_acumulada"
    ],
    label="Modelo final"
)

eje_captura.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Selección aleatoria"
)

eje_captura.set_title(
    "Captura acumulada de clientes con churn"
)

eje_captura.set_xlabel(
    "Proporción de clientes priorizados"
)

eje_captura.set_ylabel(
    "Proporción acumulada de churn capturado"
)

eje_captura.set_xlim(
    0,
    1
)

eje_captura.set_ylim(
    0,
    1
)

eje_captura.grid(
    alpha=0.3
)

eje_captura.legend()

figura_captura.tight_layout()


# ---------------------------------------------------------
# 8. Almacenamiento de evidencias
# ---------------------------------------------------------

RUTA_EVIDENCIAS = (
    RUTA_PROYECTO
    / "evidencias"
)

RUTA_EVIDENCIAS.mkdir(
    parents=True,
    exist_ok=True
)

RUTA_TABLA_CALIBRACION = (
    RUTA_RESULTADOS
    / "modelado_calibracion_test_60d.csv"
)

RUTA_DECILES_RIESGO = (
    RUTA_RESULTADOS
    / "modelado_deciles_riesgo_test_60d.csv"
)

RUTA_CAPTURA_RIESGO = (
    RUTA_RESULTADOS
    / "modelado_captura_riesgo_test_60d.csv"
)

RUTA_RESUMEN_CALIBRACION = (
    RUTA_RESULTADOS
    / "modelado_resumen_calibracion_riesgo_60d.json"
)

RUTA_GRAFICO_CALIBRACION = (
    RUTA_EVIDENCIAS
    / "figura_calibracion_test_60d.png"
)

RUTA_GRAFICO_CAPTURA = (
    RUTA_EVIDENCIAS
    / "figura_captura_acumulada_test_60d.png"
)

tabla_calibracion.to_csv(
    RUTA_TABLA_CALIBRACION,
    index=False
)

tabla_deciles_riesgo.to_csv(
    RUTA_DECILES_RIESGO,
    index=False
)

tabla_captura_riesgo.to_csv(
    RUTA_CAPTURA_RIESGO,
    index=False
)

figura_calibracion.savefig(
    RUTA_GRAFICO_CALIBRACION,
    dpi=300,
    bbox_inches="tight"
)

figura_captura.savefig(
    RUTA_GRAFICO_CAPTURA,
    dpi=300,
    bbox_inches="tight"
)

resumen_calibracion_riesgo = {
    "window_days": int(
        VENTANA_FINAL_DIAS
    ),
    "model": MODELO_FINAL,
    "threshold": float(
        UMBRAL_FINAL
    ),
    "external_test_clients": int(
        numero_clientes_test
    ),
    "external_test_churn": int(
        churn_total_test
    ),
    "prevalence": prevalencia_test,
    "brier_model": brier_modelo,
    "brier_null_model": (
        brier_modelo_nulo
    ),
    "brier_skill_score": (
        brier_skill_score
    ),
    "expected_calibration_error": (
        error_calibracion_esperado
    ),
    "maximum_calibration_error": (
        error_calibracion_maximo
    ),
    "descriptive_analysis_only": True,
    "configuration_modified": False
}

with open(
    RUTA_RESUMEN_CALIBRACION,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        resumen_calibracion_riesgo,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# 9. Validación final
# ---------------------------------------------------------

analisis_calibracion_valido = (
    tabla_calibracion[
        "clientes"
    ].sum() == 13_001
    and tabla_deciles_riesgo[
        "clientes"
    ].sum() == 13_001
    and tabla_deciles_riesgo[
        "casos_churn"
    ].sum() == 2_978
    and tabla_captura_riesgo.shape[0]
        == len(porcentajes_campana)
    and 0 <= error_calibracion_esperado <= 1
    and 0 <= error_calibracion_maximo <= 1
    and np.isfinite(
        brier_skill_score
    )
    and RUTA_TABLA_CALIBRACION.exists()
    and RUTA_DECILES_RIESGO.exists()
    and RUTA_CAPTURA_RIESGO.exists()
    and RUTA_RESUMEN_CALIBRACION.exists()
    and RUTA_GRAFICO_CALIBRACION.exists()
    and RUTA_GRAFICO_CAPTURA.exists()
)

print(
    "\n¿El análisis de calibración y concentración "
    "del riesgo se ha generado correctamente?",
    analisis_calibracion_valido
)

## Conclusión sobre la calibración y la utilidad operativa

**Calibración.** Las probabilidades estimadas muestran una correspondencia estrecha con las frecuencias observadas de `churn`. El `Brier score` del modelo es de `0,0050`, frente a `0,1766` para el modelo de referencia basado únicamente en la prevalencia. El error de calibración esperado es de `0,0011` y la mayor diferencia observada entre los grupos de probabilidad es de `0,0080`.

**Concentración del riesgo.** La ordenación de los clientes según su probabilidad estimada concentra una parte elevada de los casos de `churn` en una fracción reducida de la cartera. El 20 % de los clientes con mayor riesgo reúne el `87,24 %` de los casos observados, mientras que el 25 % concentra el `99,63 %`.

**Aplicación operativa.** Estos resultados permiten estimar la cobertura alcanzable por una campaña en función de la capacidad disponible. Los porcentajes de priorización representan escenarios descriptivos y no sustituyen el umbral de clasificación definitivo de `0,20`, que permanece congelado.

**Limitaciones.** Los resultados muestran una capacidad elevada de discriminación y concentración del riesgo en el conjunto externo de prueba, pero no implican relaciones causales ni resuelven la incertidumbre sobre la construcción temporal original de la variable objetivo. Además, las medidas de calibración agrupadas dependen del número y de la definición de los intervalos utilizados.

In [ ]:
# ---------------------------------------------------------
# Escenarios operativos seleccionados
# ---------------------------------------------------------

porcentajes_resumen_operativo = [
    0.20,
    0.25,
    0.30
]

escenarios_operativos = (
    tabla_captura_riesgo.loc[
        tabla_captura_riesgo[
            "porcentaje_clientes"
        ].isin(
            porcentajes_resumen_operativo
        )
    ]
    .copy()
    .sort_values(
        "porcentaje_clientes"
    )
    .reset_index(drop=True)
)

if len(escenarios_operativos) != 3:
    raise ValueError(
        "No se han recuperado correctamente "
        "los tres escenarios operativos."
    )

display(
    escenarios_operativos.style.format({
        "porcentaje_clientes": "{:.0%}",
        "captura_churn": "{:.2%}",
        "precision_segmento": "{:.2%}",
        "lift": "{:.2f}"
    })
)


# ---------------------------------------------------------
# Resumen operativo definitivo
# ---------------------------------------------------------

resumen_operativo_final = {
    "status": (
        "descriptive_operational_analysis_completed"
    ),
    "configuration": {
        "window_days": int(
            VENTANA_FINAL_DIAS
        ),
        "model": str(
            MODELO_FINAL
        ),
        "threshold": float(
            UMBRAL_FINAL
        ),
        "features": int(
            len(VARIABLES_FINALES)
        )
    },
    "external_test": {
        "clients": int(
            numero_clientes_test
        ),
        "churn_cases": int(
            churn_total_test
        ),
        "prevalence": float(
            prevalencia_test
        ),
        "alert_rate_at_fixed_threshold": float(
            metricas_test_externo[
                "tasa_alertas"
            ]
        ),
        "recall_at_fixed_threshold": float(
            metricas_test_externo[
                "recall"
            ]
        ),
        "precision_at_fixed_threshold": float(
            metricas_test_externo[
                "precision"
            ]
        )
    },
    "calibration": {
        "brier_model": float(
            brier_modelo
        ),
        "brier_null_model": float(
            brier_modelo_nulo
        ),
        "brier_skill_score": float(
            brier_skill_score
        ),
        "expected_calibration_error": float(
            error_calibracion_esperado
        ),
        "maximum_calibration_error": float(
            error_calibracion_maximo
        )
    },
    "campaign_scenarios": [
        {
            "customer_percentage": float(
                fila["porcentaje_clientes"]
            ),
            "selected_clients": int(
                fila["clientes_seleccionados"]
            ),
            "detected_churn": int(
                fila["churn_detectado"]
            ),
            "churn_capture": float(
                fila["captura_churn"]
            ),
            "segment_precision": float(
                fila["precision_segmento"]
            ),
            "lift": float(
                fila["lift"]
            )
        }
        for _, fila
        in escenarios_operativos.iterrows()
    ],
    "fixed_threshold_modified": False,
    "model_configuration_modified": False,
    "analysis_purpose": (
        "descriptive_campaign_capacity_scenarios"
    ),
    "causal_interpretation_allowed": False,
    "temporal_label_definition_fully_documented": False
}


# ---------------------------------------------------------
# Actualización del manifiesto final
# ---------------------------------------------------------

with open(
    RUTA_MANIFIESTO_MODELO_FINAL,
    "r",
    encoding="utf-8"
) as archivo:
    manifiesto_modelo_actualizado = (
        json.load(archivo)
    )

manifiesto_modelo_actualizado[
    "calibration"
] = resumen_operativo_final[
    "calibration"
]

manifiesto_modelo_actualizado[
    "operational_campaign_scenarios"
] = resumen_operativo_final[
    "campaign_scenarios"
]

manifiesto_modelo_actualizado[
    "threshold_changed_after_operational_analysis"
] = False

with open(
    RUTA_MANIFIESTO_MODELO_FINAL,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        manifiesto_modelo_actualizado,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Almacenamiento de evidencias
# ---------------------------------------------------------

RUTA_RESUMEN_OPERATIVO_FINAL = (
    RUTA_RESULTADOS
    / "modelado_resumen_operativo_final_60d.json"
)

RUTA_ESCENARIOS_OPERATIVOS = (
    RUTA_RESULTADOS
    / "modelado_escenarios_operativos_60d.csv"
)

with open(
    RUTA_RESUMEN_OPERATIVO_FINAL,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        resumen_operativo_final,
        archivo,
        ensure_ascii=False,
        indent=4
    )

escenarios_operativos.to_csv(
    RUTA_ESCENARIOS_OPERATIVOS,
    index=False
)


# ---------------------------------------------------------
# Validación del bloque
# ---------------------------------------------------------

captura_20 = float(
    escenarios_operativos.loc[
        np.isclose(
            escenarios_operativos[
                "porcentaje_clientes"
            ],
            0.20
        ),
        "captura_churn"
    ].iloc[0]
)

captura_25 = float(
    escenarios_operativos.loc[
        np.isclose(
            escenarios_operativos[
                "porcentaje_clientes"
            ],
            0.25
        ),
        "captura_churn"
    ].iloc[0]
)

captura_30 = float(
    escenarios_operativos.loc[
        np.isclose(
            escenarios_operativos[
                "porcentaje_clientes"
            ],
            0.30
        ),
        "captura_churn"
    ].iloc[0]
)

resumen_operativo_valido = (
    escenarios_operativos.shape[0] == 3
    and captura_20 < captura_25
    and captura_25 < captura_30
    and captura_30 <= 1.0
    and brier_modelo < brier_modelo_nulo
    and brier_skill_score > 0
    and resumen_operativo_final[
        "fixed_threshold_modified"
    ] is False
    and resumen_operativo_final[
        "model_configuration_modified"
    ] is False
    and manifiesto_modelo_actualizado[
        "threshold_changed_after_operational_analysis"
    ] is False
    and RUTA_RESUMEN_OPERATIVO_FINAL.exists()
    and RUTA_ESCENARIOS_OPERATIVOS.exists()
)


# ---------------------------------------------------------
# Presentación del cierre operativo
# ---------------------------------------------------------

print(
    "Captura de churn priorizando el 20 %:",
    f"{captura_20:.2%}"
)

print(
    "Captura de churn priorizando el 25 %:",
    f"{captura_25:.2%}"
)

print(
    "Captura de churn priorizando el 30 %:",
    f"{captura_30:.2%}"
)

print(
    "Brier skill score:",
    f"{brier_skill_score:.4f}"
)

print(
    "¿Se ha modificado el umbral definitivo?",
    resumen_operativo_final[
        "fixed_threshold_modified"
    ]
)

print(
    "¿Se ha cerrado correctamente el "
    "análisis operativo?",
    resumen_operativo_valido
)

## Síntesis visual y cierre del modelado

**Objetivo.** Se consolidan las principales evidencias obtenidas durante la selección, validación e interpretación del modelo final.

Se generan las siguientes representaciones:

- Comparación del rendimiento de las ventanas temporales.
- Curvas ROC y de `precision-recall` sobre el conjunto externo de prueba.
- Matriz de confusión de la configuración definitiva.
- Coeficientes de mayor magnitud de la regresión logística.

Estas figuras tienen una finalidad exclusivamente descriptiva y no se utilizan para modificar la configuración congelada.

Además, se almacena un resumen final en formato JSON y un inventario de los archivos generados. Estos elementos facilitan la trazabilidad entre el notebook, las evidencias obtenidas y la memoria del proyecto.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_curve
)


# ---------------------------------------------------------
# Rutas de salida
# ---------------------------------------------------------

RUTA_EVIDENCIAS.mkdir(
    parents=True,
    exist_ok=True
)

RUTA_FIGURA_COMPARACION_TEMPORAL = (
    RUTA_EVIDENCIAS
    / "figura_comparacion_ventanas_modelos.png"
)

RUTA_FIGURA_ROC_TEST = (
    RUTA_EVIDENCIAS
    / "figura_curva_roc_test_60d.png"
)

RUTA_FIGURA_PR_TEST = (
    RUTA_EVIDENCIAS
    / "figura_curva_precision_recall_test_60d.png"
)

RUTA_FIGURA_MATRIZ_CONFUSION = (
    RUTA_EVIDENCIAS
    / "figura_matriz_confusion_test_60d.png"
)

RUTA_FIGURA_COEFICIENTES = (
    RUTA_EVIDENCIAS
    / "figura_coeficientes_principales_60d.png"
)


# ---------------------------------------------------------
# 1. Comparación de ventanas y modelos
# ---------------------------------------------------------

figura_temporal, eje_temporal = plt.subplots(
    figsize=(8, 6)
)

for nombre_modelo in MODELOS_COMPARACION_TEMPORAL:

    datos_modelo = (
        comparacion_temporal.loc[
            comparacion_temporal[
                "modelo"
            ] == nombre_modelo
        ]
        .sort_values(
            "ventana_dias"
        )
    )

    eje_temporal.plot(
        datos_modelo[
            "ventana_dias"
        ],
        datos_modelo[
            "average_precision"
        ],
        marker="o",
        label=nombre_modelo
    )

eje_temporal.set_title(
    "Average precision por ventana y modelo"
)

eje_temporal.set_xlabel(
    "Ventana de observación (días)"
)

eje_temporal.set_ylabel(
    "Average precision OOF"
)

eje_temporal.set_xticks(
    VENTANAS_COMPARACION
)

eje_temporal.grid(
    alpha=0.3
)

eje_temporal.legend()

figura_temporal.tight_layout()

figura_temporal.savefig(
    RUTA_FIGURA_COMPARACION_TEMPORAL,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# 2. Curva ROC del test externo
# ---------------------------------------------------------

fpr_test, tpr_test, _ = roc_curve(
    y_test_final,
    probabilidades_test_externo
)

figura_roc, eje_roc = plt.subplots(
    figsize=(7, 6)
)

eje_roc.plot(
    fpr_test,
    tpr_test,
    label=(
        "Modelo final "
        f"(AUC = "
        f"{metricas_test_externo['auc_roc']:.4f})"
    )
)

eje_roc.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Clasificación aleatoria"
)

eje_roc.set_title(
    "Curva ROC en el test externo"
)

eje_roc.set_xlabel(
    "Tasa de falsos positivos"
)

eje_roc.set_ylabel(
    "Tasa de verdaderos positivos"
)

eje_roc.set_xlim(
    0,
    1
)

eje_roc.set_ylim(
    0,
    1
)

eje_roc.grid(
    alpha=0.3
)

eje_roc.legend()

figura_roc.tight_layout()

figura_roc.savefig(
    RUTA_FIGURA_ROC_TEST,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# 3. Curva precision-recall del test externo
# ---------------------------------------------------------

precision_curva, recall_curva, _ = (
    precision_recall_curve(
        y_test_final,
        probabilidades_test_externo
    )
)

figura_pr, eje_pr = plt.subplots(
    figsize=(7, 6)
)

eje_pr.plot(
    recall_curva,
    precision_curva,
    label=(
        "Modelo final "
        f"(AP = "
        f"{metricas_test_externo['average_precision']:.4f})"
    )
)

eje_pr.axhline(
    prevalencia_test,
    linestyle="--",
    label=(
        "Prevalencia "
        f"({prevalencia_test:.2%})"
    )
)

eje_pr.set_title(
    "Curva precision-recall en el test externo"
)

eje_pr.set_xlabel(
    "Recall"
)

eje_pr.set_ylabel(
    "Precision"
)

eje_pr.set_xlim(
    0,
    1
)

eje_pr.set_ylim(
    0,
    1
)

eje_pr.grid(
    alpha=0.3
)

eje_pr.legend()

figura_pr.tight_layout()

figura_pr.savefig(
    RUTA_FIGURA_PR_TEST,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# 4. Matriz de confusión
# ---------------------------------------------------------

matriz_confusion_final = np.array([
    [
        int(tn_test),
        int(fp_test)
    ],
    [
        int(fn_test),
        int(tp_test)
    ]
])

figura_confusion, eje_confusion = (
    plt.subplots(
        figsize=(6, 5)
    )
)

visualizador_confusion = (
    ConfusionMatrixDisplay(
        confusion_matrix=(
            matriz_confusion_final
        ),
        display_labels=[
            "No churn",
            "Churn"
        ]
    )
)

visualizador_confusion.plot(
    ax=eje_confusion,
    values_format="d",
    colorbar=False
)

eje_confusion.set_title(
    "Matriz de confusión del test externo"
)

figura_confusion.tight_layout()

figura_confusion.savefig(
    RUTA_FIGURA_MATRIZ_CONFUSION,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# 5. Principales coeficientes del modelo
# ---------------------------------------------------------

coeficientes_grafico = (
    tabla_coeficientes_finales
    .head(20)
    .sort_values(
        "coeficiente_estandarizado"
    )
)

figura_coeficientes, eje_coeficientes = (
    plt.subplots(
        figsize=(10, 8)
    )
)

eje_coeficientes.barh(
    coeficientes_grafico[
        "variable"
    ],
    coeficientes_grafico[
        "coeficiente_estandarizado"
    ]
)

eje_coeficientes.axvline(
    0,
    linewidth=1
)

eje_coeficientes.set_title(
    "Principales coeficientes estandarizados"
)

eje_coeficientes.set_xlabel(
    "Coeficiente de la regresión logística"
)

eje_coeficientes.set_ylabel(
    "Variable"
)

eje_coeficientes.grid(
    axis="x",
    alpha=0.3
)

figura_coeficientes.tight_layout()

figura_coeficientes.savefig(
    RUTA_FIGURA_COEFICIENTES,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# 6. Resumen final del modelado
# ---------------------------------------------------------

resumen_final_modelado = {
    "status": "modeling_closed",
    "dataset": {
        "clients_total": 65_005,
        "original_train_clients": 52_004,
        "external_test_clients": 13_001,
        "predictor_variables": 176,
        "activity_profiles": 12
    },
    "selected_configuration": {
        "window_days": int(
            VENTANA_FINAL_DIAS
        ),
        "model": str(
            MODELO_FINAL
        ),
        "threshold": float(
            UMBRAL_FINAL
        ),
        "selection_rule": (
            CRITERIO_FINAL
        )
    },
    "external_test_metrics": {
        "average_precision": float(
            metricas_test_externo[
                "average_precision"
            ]
        ),
        "auc_roc": float(
            metricas_test_externo[
                "auc_roc"
            ]
        ),
        "precision": float(
            metricas_test_externo[
                "precision"
            ]
        ),
        "recall": float(
            metricas_test_externo[
                "recall"
            ]
        ),
        "f1": float(
            metricas_test_externo[
                "f1"
            ]
        ),
        "f2": float(
            metricas_test_externo[
                "f2"
            ]
        ),
        "alert_rate": float(
            metricas_test_externo[
                "tasa_alertas"
            ]
        ),
        "brier": float(
            metricas_test_externo[
                "brier"
            ]
        ),
        "log_loss": float(
            metricas_test_externo[
                "log_loss"
            ]
        )
    },
    "confusion_matrix": {
        "true_negatives": int(
            tn_test
        ),
        "false_positives": int(
            fp_test
        ),
        "false_negatives": int(
            fn_test
        ),
        "true_positives": int(
            tp_test
        )
    },
    "calibration": {
        "brier_skill_score": float(
            brier_skill_score
        ),
        "expected_calibration_error": float(
            error_calibracion_esperado
        ),
        "maximum_calibration_error": float(
            error_calibracion_maximo
        )
    },
    "operational_scenarios": {
        "capture_top_20_percent": float(
            captura_20
        ),
        "capture_top_25_percent": float(
            captura_25
        ),
        "capture_top_30_percent": float(
            captura_30
        )
    },
    "interpretability": {
        "variables_same_sign_all_folds": int(
            variables_signo_100
        ),
        "variables_sign_consistency_ge_80": int(
            variables_signo_80
        ),
        "top_20_same_sign_all_folds": int(
            top20_signo_100
        )
    },
    "methodological_limits": {
        "causal_interpretation_allowed": False,
        "temporal_label_definition_fully_documented": False,
        "strict_temporal_anticipation_proven": False
    },
    "configuration_changed_after_external_test": False,
    "further_model_tuning_allowed": False
}

RUTA_RESUMEN_FINAL_MODELADO = (
    RUTA_RESULTADOS
    / "modelado_resumen_final_60d.json"
)

with open(
    RUTA_RESUMEN_FINAL_MODELADO,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        resumen_final_modelado,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# 7. Inventario de archivos generados
# ---------------------------------------------------------

directorios_inventario = [
    RUTA_RESULTADOS,
    RUTA_EVIDENCIAS,
    RUTA_MODELOS
]

filas_inventario = []

for directorio in directorios_inventario:

    for ruta_archivo in sorted(
        directorio.rglob("*")
    ):

        if not ruta_archivo.is_file():
            continue

        estadisticas = ruta_archivo.stat()

        filas_inventario.append({
            "directorio": (
                directorio.name
            ),
            "archivo": (
                ruta_archivo.name
            ),
            "ruta_relativa": str(
                ruta_archivo.relative_to(
                    RUTA_PROYECTO
                )
            ),
            "extension": (
                ruta_archivo.suffix.lower()
            ),
            "tamano_mb": (
                estadisticas.st_size
                / (1024 ** 2)
            )
        })

inventario_evidencias = (
    pd.DataFrame(
        filas_inventario
    )
    .sort_values(
        [
            "directorio",
            "archivo"
        ]
    )
    .reset_index(drop=True)
)

RUTA_INVENTARIO_EVIDENCIAS = (
    RUTA_RESULTADOS
    / "inventario_evidencias_modelado.csv"
)

inventario_evidencias.to_csv(
    RUTA_INVENTARIO_EVIDENCIAS,
    index=False
)

print(
    "Archivos inventariados:",
    len(inventario_evidencias)
)

display(
    inventario_evidencias.tail(20).style.format({
        "tamano_mb": "{:.3f}"
    })
)


# ---------------------------------------------------------
# 8. Actualización final del manifiesto
# ---------------------------------------------------------

with open(
    RUTA_MANIFIESTO_MODELO_FINAL,
    "r",
    encoding="utf-8"
) as archivo:
    manifiesto_cierre = json.load(
        archivo
    )

manifiesto_cierre[
    "status"
] = "modeling_closed"

manifiesto_cierre[
    "further_model_tuning_allowed"
] = False

manifiesto_cierre[
    "final_figures"
] = [
    str(
        RUTA_FIGURA_COMPARACION_TEMPORAL
        .relative_to(
            RUTA_PROYECTO
        )
    ),
    str(
        RUTA_FIGURA_ROC_TEST
        .relative_to(
            RUTA_PROYECTO
        )
    ),
    str(
        RUTA_FIGURA_PR_TEST
        .relative_to(
            RUTA_PROYECTO
        )
    ),
    str(
        RUTA_FIGURA_MATRIZ_CONFUSION
        .relative_to(
            RUTA_PROYECTO
        )
    ),
    str(
        RUTA_FIGURA_COEFICIENTES
        .relative_to(
            RUTA_PROYECTO
        )
    )
]

with open(
    RUTA_MANIFIESTO_MODELO_FINAL,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        manifiesto_cierre,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# 9. Validación del cierre del notebook
# ---------------------------------------------------------

rutas_figuras_finales = [
    RUTA_FIGURA_COMPARACION_TEMPORAL,
    RUTA_FIGURA_ROC_TEST,
    RUTA_FIGURA_PR_TEST,
    RUTA_FIGURA_MATRIZ_CONFUSION,
    RUTA_FIGURA_COEFICIENTES
]

cierre_modelado_valido = (
    configuracion_final_congelada
    and evaluacion_externa_valida
    and cierre_validacion_externa_valido
    and interpretabilidad_valida
    and estabilidad_coeficientes_valida
    and analisis_calibracion_valido
    and resumen_operativo_valido
    and all(
        ruta.exists()
        for ruta in rutas_figuras_finales
    )
    and RUTA_RESUMEN_FINAL_MODELADO.exists()
    and RUTA_INVENTARIO_EVIDENCIAS.exists()
    and RUTA_MODELO_FINAL_ENTRENADO.exists()
    and manifiesto_cierre[
        "status"
    ] == "modeling_closed"
    and manifiesto_cierre[
        "further_model_tuning_allowed"
    ] is False
)

print(
    "\nConfiguración final:",
    f"{VENTANA_FINAL_DIAS} días + "
    f"{MODELO_FINAL} + "
    f"umbral {UMBRAL_FINAL:.2f}"
)

print(
    "Figuras finales guardadas:",
    len(rutas_figuras_finales)
)

print(
    "Archivos incluidos en el inventario:",
    len(inventario_evidencias)
)

print(
    "Estado final del manifiesto:",
    manifiesto_cierre[
        "status"
    ]
)

print(
    "¿Se permiten más ajustes del modelo?",
    manifiesto_cierre[
        "further_model_tuning_allowed"
    ]
)

print(
    "¿El notebook de modelado se ha cerrado "
    "correctamente?",
    cierre_modelado_valido
)